# Fast hybrid embedding retrieval and negative sampling

This standalone Kaggle notebook implements the fast embedding workflow as a
contrastive shared-encoder two-tower model. It caches frozen multilingual MiniLM
catalog embeddings and exact top-500 retrieval once, mixes representative random
negatives with conservative lexical/embedding semi-hard negatives, and keeps
checkpoint selection term-grouped and separate from the outer holdout.

The default `MODE="pilot"` uses a fixed stratified 80/20 term holdout. After
selecting a recipe by Macro-F1 (Recall@100 breaks ties), use `MODE="confirm"` to
run five grouped folds for only that pilot recipe. Use `MODE="final"` with the
confirmed blend JSON to train on all positive terms, score every submission pair
by encoding each unique query/item once, and write a validated submission.

All long stages are resumable. Save the packaged artifact archive or the output
directory as a Kaggle dataset between sessions. The original
`grouped_5fold_oof_kaggle.ipynb` is not used or modified.

The negative mix follows the representative-random plus conservative semi-hard
pattern supported by [Facebook's retrieval study](https://ar5iv.labs.arxiv.org/html/2006.11632)
and [JD's e-commerce model](https://ar5iv.labs.arxiv.org/html/2006.02282).


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

# --------------------------- EDIT THIS CELL ---------------------------
MODE = "pilot"  # "pilot", "confirm", or "final"
SEED = 42
BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = "cuda"
SKIP_EXISTING = True
PACKAGE_ARTIFACTS = True

# The pilot evaluates both contrastive variants. Confirm/final read the winning
# pilot/confirmed JSON from an attached cache or an earlier run.
PILOT_SELECTED_BLEND = None  # e.g. Path("/kaggle/input/.../selected_blend.json")
CONFIRMED_SELECTED_BLEND = None  # required for MODE="final"
COMPUTE_CONTRASTIVE_RECALL = True

RUN_GBDT = True
GBDT_MODELS = ["lgbm"]  # add "catboost" for the full current baseline
CATBOOST_DEVICES = "0"
CATBOOST_ITERATIONS = 2000
LGBM_ESTIMATORS = 4000

TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 64
ENCODE_BATCH_SIZE = 512
WEIGHT_STEP = 0.10

# Final GBDT components normally come from the confirmed full-data baseline.
# Every entry uses: selected-component-name -> (score CSV, score column).
# The notebook adds the selected frozen/contrastive semantic component itself.
FINAL_COMPONENTS = {
    # "lgbm_prob": (Path("/kaggle/input/my-scores/pu_lgbm_scores.csv"), "prob"),
    # "catboost_prob": (Path("/kaggle/input/my-scores/catboost_scores.csv"), "prob"),
    # "lexical_score": (Path("/kaggle/input/my-scores/lexical_scores.csv"), "score"),
}
EXPECTED_SUBMISSION_ROWS = 3_359_679
# ---------------------------------------------------------------------

WORK_DIR = Path("/kaggle/working")
SRC_DIR = WORK_DIR / "src"
ROOT_OUT = WORK_DIR / "fast_hybrid_embedding"
CACHE_DIR = ROOT_OUT / "frozen_cache"
RUN_DIR = ROOT_OUT / MODE
SRC_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

data_candidates = [
    Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle"),
    Path("/kaggle/input/trendyol-e-ticaret-yarismasi-2026-kaggle"),
]
DATA_DIR = next(
    (path for path in data_candidates if (path / "training_pairs.csv").exists()), None
)
if DATA_DIR is None:
    matches = list(Path("/kaggle/input").glob("**/training_pairs.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Could not uniquely locate competition data; matches={matches[:10]}"
        )
    DATA_DIR = matches[0].parent

print(f"MODE={MODE} DATA_DIR={DATA_DIR} RUN_DIR={RUN_DIR}")

## Runtime dependencies

In [ ]:
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "torch": "torch",
    "transformers": "transformers",
    "lightgbm": "lightgbm",
    "catboost": "catboost",
    "rapidfuzz": "rapidfuzz",
    "text_unidecode": "text-unidecode",
}
missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    print("installing", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies are available")

## Embedded source modules

In [ ]:
%%writefile src/lexical_baseline.py
"""Build Turkish-aware lexical relevance features, scores, and submissions."""

from __future__ import annotations

import argparse
import csv
import math
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz
from text_unidecode import unidecode
from tqdm import tqdm

TOKEN_RE = re.compile(r"[a-z0-9]+")

STOPWORDS = {
    "a",
    "ve",
    "ile",
    "icin",
    "bir",
    "adet",
    "ad",
    "li",
    "set",
    "seti",
    "takim",
    "takimi",
    "urun",
    "model",
    "uyumlu",
    "orjinal",
    "orijinal",
}

COLORS = {
    "beyaz",
    "siyah",
    "kirmizi",
    "mavi",
    "yesil",
    "sari",
    "pembe",
    "mor",
    "gri",
    "lacivert",
    "kahverengi",
    "bej",
    "krem",
    "turuncu",
    "bordo",
    "gold",
    "altin",
    "gumus",
    "haki",
    "taba",
    "lila",
    "ekru",
    "vizon",
    "füme",
}

GENDER_ALLOWED = {
    "kadin": {"kadin", "unisex", "unknown"},
    "bayan": {"kadin", "unisex", "unknown"},
    "erkek": {"erkek", "unisex", "unknown"},
    "unisex": {"kadin", "erkek", "unisex", "unknown"},
}

AGE_ALLOWED = {
    "bebek": {"bebek", "unknown"},
    "cocuk": {"cocuk", "bebek", "unknown"},
    "kiz": {"cocuk", "bebek", "unknown"},
}


def normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = unidecode(str(value).lower())
    text = text.replace("&", " ")
    return re.sub(r"\s+", " ", text).strip()


def stem_token(token: str) -> str:
    if len(token) <= 4:
        return token

    for suffix in (
        "larin",
        "lerin",
        "lari",
        "leri",
        "lara",
        "lere",
        "larda",
        "lerde",
        "ndan",
        "nden",
        "dan",
        "den",
        "nin",
        "nun",
        "ler",
        "lar",
        "lik",
        "luk",
        "cuk",
        "cik",
        "ci",
        "cu",
        "si",
        "su",
        "u",
        "i",
    ):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token


@lru_cache(maxsize=1_000_000)
def token_set(text: str) -> frozenset[str]:
    out: set[str] = set()
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        out.add(token)
        stemmed = stem_token(token)
        if stemmed not in STOPWORDS:
            out.add(stemmed)
    return frozenset(out)


def query_tokens(text: str) -> tuple[str, ...]:
    seen: set[str] = set()
    out: list[str] = []
    for token in TOKEN_RE.findall(text):
        if token in STOPWORDS:
            continue
        stemmed = stem_token(token)
        for candidate in (token, stemmed):
            if candidate and candidate not in STOPWORDS and candidate not in seen:
                seen.add(candidate)
                out.append(candidate)
                break
    return tuple(out)


def coverage(query: tuple[str, ...], field: frozenset[str]) -> float:
    if not query:
        return 0.0
    return sum(1 for token in query if token in field) / len(query)


def color_tokens(query: tuple[str, ...]) -> set[str]:
    return {token for token in query if token in COLORS}


def load_terms(path: Path) -> dict[str, dict[str, object]]:
    terms = pd.read_csv(path)
    records: dict[str, dict[str, object]] = {}
    for row in terms.itertuples(index=False):
        norm_query = normalize_text(row.query)
        q_tokens = query_tokens(norm_query)
        records[row.term_id] = {
            "query": norm_query,
            "tokens": q_tokens,
            "colors": color_tokens(q_tokens),
        }
    return records


def load_items(path: Path) -> dict[str, tuple[str, str, str, str, str, str]]:
    items: dict[str, tuple[str, str, str, str, str, str]] = {}
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in tqdm(reader, desc="items"):
            item_id = row["item_id"]
            items[item_id] = (
                normalize_text(row.get("title")),
                normalize_text(row.get("category")),
                normalize_text(row.get("brand")),
                normalize_text(row.get("gender")) or "unknown",
                normalize_text(row.get("age_group")) or "unknown",
                normalize_text(row.get("attributes")),
            )
    return items


def pair_features(
    term: dict[str, object], item: tuple[str, str, str, str, str, str]
) -> dict[str, float]:
    query = term["query"]  # type: ignore[index]
    q_tokens = term["tokens"]  # type: ignore[index]
    q_colors = term["colors"]  # type: ignore[index]
    title, category, brand, gender, age_group, attrs = item

    title_tokens = token_set(title)
    category_tokens = token_set(category)
    brand_tokens = token_set(brand)
    attr_tokens = token_set(attrs)
    full_tokens = title_tokens | category_tokens | brand_tokens | attr_tokens

    title_cov = coverage(q_tokens, title_tokens)
    category_cov = coverage(q_tokens, category_tokens)
    brand_cov = coverage(q_tokens, brand_tokens)
    attr_cov = coverage(q_tokens, attr_tokens)
    full_cov = coverage(q_tokens, full_tokens)

    title_ratio = fuzz.token_set_ratio(query, title) / 100.0
    category_ratio = fuzz.token_set_ratio(query, category) / 100.0
    brand_ratio = max(
        fuzz.ratio(query, brand) / 100.0,
        fuzz.partial_ratio(query, brand) / 100.0 if brand else 0.0,
    )

    query_in_title = float(bool(query and query in title))
    query_in_category = float(bool(query and query in category))
    query_in_brand = float(bool(query and query in brand))
    full_complete = float(full_cov >= 0.999)
    full_high = float(0.75 <= full_cov < 0.999)
    title_complete = float(title_cov >= 0.999)
    category_complete = float(category_cov >= 0.999)
    brand_complete = float(brand_cov >= 0.999 and bool(brand))
    short_complete = float(len(q_tokens) <= 2 and full_cov >= 0.999)

    gender_mismatch = 0.0
    for gender_word, allowed in GENDER_ALLOWED.items():
        if (
            gender_word in q_tokens
            and gender not in allowed
            and gender_word not in full_tokens
        ):
            gender_mismatch = 1.0
            break

    age_mismatch = 0.0
    for age_word, allowed in AGE_ALLOWED.items():
        if (
            age_word in q_tokens
            and age_group not in allowed
            and age_word not in full_tokens
        ):
            age_mismatch = 1.0
            break

    color_mismatch = float(bool(q_colors and full_tokens.isdisjoint(q_colors)))
    weak_match = float(
        full_cov < 0.34 and max(title_ratio, category_ratio, brand_ratio) < 0.55
    )

    features = {
        "n_query_tokens": float(len(q_tokens)),
        "title_cov": title_cov,
        "category_cov": category_cov,
        "brand_cov": brand_cov,
        "attr_cov": attr_cov,
        "full_cov": full_cov,
        "title_ratio": title_ratio,
        "category_ratio": category_ratio,
        "brand_ratio": brand_ratio,
        "query_in_title": query_in_title,
        "query_in_category": query_in_category,
        "query_in_brand": query_in_brand,
        "full_complete": full_complete,
        "full_high": full_high,
        "title_complete": title_complete,
        "category_complete": category_complete,
        "brand_complete": brand_complete,
        "short_complete": short_complete,
        "gender_mismatch": gender_mismatch,
        "age_mismatch": age_mismatch,
        "color_mismatch": color_mismatch,
        "weak_match": weak_match,
        "title_len": float(len(title_tokens)),
        "category_len": float(len(category_tokens)),
        "brand_len": float(len(brand_tokens)),
        "attr_len": float(len(attr_tokens)),
    }

    features["lexical_score"] = lexical_score_from_features(features)
    return features


def lexical_score_from_features(features: dict[str, float]) -> float:
    title_cov = features["title_cov"]
    category_cov = features["category_cov"]
    brand_cov = features["brand_cov"]
    attr_cov = features["attr_cov"]
    full_cov = features["full_cov"]
    title_ratio = features["title_ratio"]
    category_ratio = features["category_ratio"]
    brand_ratio = features["brand_ratio"]

    score = 0.0
    score += 2.1 * full_cov
    score += 1.4 * title_cov
    score += 1.0 * category_cov
    score += 1.2 * brand_cov
    score += 0.55 * attr_cov
    score += 0.75 * title_ratio
    score += 0.45 * category_ratio
    score += 0.55 * brand_ratio

    if features["query_in_title"]:
        score += 1.0
    if features["query_in_category"]:
        score += 0.8
    if features["query_in_brand"]:
        score += 1.1
    if features["full_complete"]:
        score += 1.2
    elif features["full_high"]:
        score += 0.55
    if features["title_complete"]:
        score += 0.9
    if features["category_complete"]:
        score += 0.7
    if features["brand_complete"]:
        score += 0.9
    if features["short_complete"]:
        score += 0.55

    if features["gender_mismatch"]:
        score -= 0.85
    if features["age_mismatch"]:
        score -= 0.75
    if features["color_mismatch"]:
        score -= 0.7
    if features["weak_match"]:
        score -= 0.9

    return score


def score_pair(
    term: dict[str, object], item: tuple[str, str, str, str, str, str]
) -> float:
    return pair_features(term, item)["lexical_score"]


def score_pairs(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")

    in_path = data_dir / args.pairs
    total = args.limit
    if total is None:
        with in_path.open(newline="", encoding="utf-8") as handle:
            total = sum(1 for _ in handle) - 1

    with in_path.open(newline="", encoding="utf-8") as in_handle, out_path.open(
        "w", newline="", encoding="utf-8"
    ) as out_handle:
        reader = csv.DictReader(in_handle)
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "score"])
        for i, row in enumerate(tqdm(reader, total=total, desc="pairs")):
            if args.limit is not None and i >= args.limit:
                break
            item = items[row["item_id"]]
            term = terms[row["term_id"]]
            writer.writerow(
                [row["id"], row["term_id"], f"{score_pair(term, item):.6f}"]
            )


def summarize_scores(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    print(
        scores["score"].describe(
            percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        )
    )
    for threshold in args.thresholds:
        pred = scores["score"] >= threshold
        print(
            f"threshold={threshold:.3f} positives={int(pred.sum())} rate={pred.mean():.5f}"
        )

    by_term = scores.groupby("term_id")["score"].agg(["count", "max", "mean"])
    print(by_term.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    pred = scores["score"] >= args.threshold

    if args.min_per_term > 0 or args.max_per_term > 0:
        scores = scores.assign(pred=pred.astype("int8"))
        parts = []
        for _, group in tqdm(scores.groupby("term_id", sort=False), desc="terms"):
            group = group.copy()
            order = group["score"].sort_values(ascending=False).index
            if (
                args.min_per_term > 0
                and group.loc[order[: args.min_per_term], "score"].max()
                >= args.min_score
            ):
                group.loc[order[: args.min_per_term], "pred"] = 1
            if args.max_per_term > 0:
                positives = group.index[group["pred"] == 1]
                if len(positives) > args.max_per_term:
                    keep = (
                        group.loc[positives]
                        .sort_values("score", ascending=False)
                        .head(args.max_per_term)
                        .index
                    )
                    group.loc[positives, "pred"] = 0
                    group.loc[keep, "pred"] = 1
            parts.append(group[["id", "pred"]])
        scored = pd.concat(parts, ignore_index=True)
        pred_values = scored["pred"].astype("int8")
        ids = scored["id"]
    else:
        pred_values = pred.astype("int8")
        ids = scores["id"]

    output = pd.DataFrame({"id": ids, "prediction": pred_values})
    output.to_csv(args.output, index=False)
    counts = Counter(output["prediction"])
    print(f"wrote {args.output}")
    print(
        f"rows={len(output)} positives={counts[1]} rate={counts[1] / len(output):.5f}"
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    score = sub.add_parser("score")
    score.add_argument("--data-dir", default="data")
    score.add_argument("--pairs", default="submission_pairs.csv")
    score.add_argument("--output", default="outputs/lexical_scores.csv")
    score.add_argument("--limit", type=int)
    score.set_defaults(func=score_pairs)

    summary = sub.add_parser("summary")
    summary.add_argument("--scores", default="outputs/lexical_scores.csv")
    summary.add_argument(
        "--thresholds", type=float, nargs="+", default=[3.0, 3.5, 4.0, 4.5, 5.0]
    )
    summary.set_defaults(func=summarize_scores)

    submit = sub.add_parser("submit")
    submit.add_argument("--scores", default="outputs/lexical_scores.csv")
    submit.add_argument("--output", default="outputs/submission_lexical.csv")
    submit.add_argument("--threshold", type=float, required=True)
    submit.add_argument("--min-per-term", type=int, default=0)
    submit.add_argument("--max-per-term", type=int, default=0)
    submit.add_argument("--min-score", type=float, default=0.0)
    submit.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_term_negatives.py
"""Mine deterministic TF-IDF negatives from training queries and catalog items."""

from __future__ import annotations

import argparse
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

from lexical_baseline import normalize_text


def item_search_text(items: pd.DataFrame) -> pd.Series:
    title = items["title"].map(normalize_text)
    category = items["category"].map(normalize_text)
    brand = items["brand"].map(normalize_text)
    attrs = items["attributes"].map(normalize_text)
    return (
        title + " " + title + " " + brand + " " + brand + " " + category + " " + attrs
    ).str.strip()


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def topk_sparse_row(row: sparse.spmatrix, k: int) -> tuple[np.ndarray, np.ndarray]:
    row = row.tocsr()
    indices = row.indices
    data = row.data
    if len(data) == 0:
        return indices, data

    if len(data) > k:
        selected = np.argpartition(data, -k)[-k:]
        selected = selected[np.argsort(-data[selected], kind="mergesort")]
    else:
        selected = np.argsort(-data, kind="mergesort")
    return indices[selected], data[selected]


def sample_easy_items(
    item_count: int,
    excluded: set[int],
    n_items: int,
    rng: np.random.Generator,
) -> list[int]:
    if n_items <= 0:
        return []

    selected: list[int] = []
    attempts = 0
    max_attempts = max(100, n_items * 50)
    while len(selected) < n_items and attempts < max_attempts:
        item_pos = int(rng.integers(0, item_count))
        attempts += 1
        if item_pos in excluded:
            continue
        excluded.add(item_pos)
        selected.append(item_pos)
    return selected


def load_positive_pairs(path: Path) -> pd.DataFrame:
    pairs = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in pairs.columns:
        pairs = pairs.loc[pairs["label"].astype(str).ne("0")]
    pairs = pairs[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if pairs.empty:
        raise ValueError(f"No positive training pairs found in {path}")
    return pairs


def mine_negatives(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "title", "category", "brand", "attributes"],
    )
    positives = load_positive_pairs(data_dir / "training_pairs.csv")

    train_term_ids = positives["term_id"].drop_duplicates().to_numpy()
    if args.limit_terms:
        train_term_ids = train_term_ids[: args.limit_terms]
        positives = positives.loc[
            positives["term_id"].isin(train_term_ids)
        ].reset_index(drop=True)
    if args.limit_items:
        items = items.head(args.limit_items).reset_index(drop=True)

    terms = terms.loc[
        terms["term_id"].isin(train_term_ids), ["term_id", "query"]
    ].copy()
    missing_terms = set(train_term_ids) - set(terms["term_id"])
    if missing_terms:
        examples = sorted(missing_terms)[:5]
        raise KeyError(
            f"{len(missing_terms)} training term_ids are missing from terms.csv, "
            f"examples={examples}"
        )

    terms["query_norm"] = terms["query"].map(normalize_text)
    terms = terms.set_index("term_id").loc[train_term_ids].reset_index()
    item_ids = items["item_id"].astype(str).to_numpy()
    item_to_pos = {item_id: i for i, item_id in enumerate(item_ids)}

    positive_by_term: dict[str, set[int]] = {}
    for term_id, group in positives.groupby("term_id", sort=False):
        positions = {
            item_to_pos[item_id]
            for item_id in group["item_id"].astype(str)
            if item_id in item_to_pos
        }
        positive_by_term[str(term_id)] = positions

    item_text = item_search_text(items)
    corpus = pd.concat([terms["query_norm"], item_text], ignore_index=True)
    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, args.word_ngram_max),
        min_df=args.min_df,
        max_features=args.max_features,
        dtype=np.float32,
        norm="l2",
        sublinear_tf=True,
    )
    vectorizer.fit(corpus)
    term_matrix = vectorizer.transform(terms["query_norm"])
    item_matrix = vectorizer.transform(item_text)
    item_matrix_t = item_matrix.T.tocsr()
    print(
        f"terms={len(terms):,} items={len(items):,} positives={len(positives):,} "
        f"vocab={len(vectorizer.vocabulary_):,}"
    )

    rng = np.random.default_rng(args.seed)
    counts = {"hard": 0, "mid": 0, "easy": 0}
    total = 0
    positive_keys = set(pair_key(positives))

    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["term_id", "item_id", "negative_band", "tfidf_score"])

        for start in tqdm(
            range(0, len(terms), args.chunk_size), desc="mine train-term negatives"
        ):
            stop = min(start + args.chunk_size, len(terms))
            scores = term_matrix[start:stop] @ item_matrix_t
            scores = scores.tocsr()

            for row_offset, term_id in enumerate(
                terms["term_id"].iloc[start:stop].astype(str)
            ):
                pos_items = set(positive_by_term.get(term_id, set()))
                excluded = set(pos_items)
                row_indices, row_scores = topk_sparse_row(
                    scores.getrow(row_offset), args.retrieve_topk
                )

                candidates: list[tuple[int, float]] = []
                for item_pos, score in zip(row_indices, row_scores):
                    item_pos = int(item_pos)
                    if item_pos in excluded:
                        continue
                    excluded.add(item_pos)
                    candidates.append((item_pos, float(score)))

                hard = candidates[: args.hard_per_term]
                mid_pool = candidates[args.hard_per_term :]
                if len(mid_pool) > args.mid_per_term:
                    chosen = rng.choice(
                        len(mid_pool), size=args.mid_per_term, replace=False
                    )
                    mid = [mid_pool[int(i)] for i in chosen]
                else:
                    mid = mid_pool

                rows: list[tuple[str, str, str, float]] = []
                for item_pos, score in hard:
                    rows.append((term_id, item_ids[item_pos], "hard", score))
                for item_pos, score in mid:
                    rows.append((term_id, item_ids[item_pos], "mid", score))

                easy_positions = sample_easy_items(
                    len(item_ids), excluded, args.easy_per_term, rng
                )
                for item_pos in easy_positions:
                    rows.append((term_id, item_ids[item_pos], "easy", 0.0))

                for out_term_id, item_id, band, score in rows:
                    if f"{out_term_id}\t{item_id}" in positive_keys:
                        continue
                    writer.writerow([out_term_id, item_id, band, f"{score:.8f}"])
                    counts[band] += 1
                    total += 1

    print(f"wrote {output_path}")
    print(
        f"negatives={total:,} hard={counts['hard']:,} "
        f"mid={counts['mid']:,} easy={counts['easy']:,}"
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output", default="outputs/train_term_negatives.csv")
    parser.add_argument("--hard-per-term", type=int, default=25)
    parser.add_argument("--mid-per-term", type=int, default=15)
    parser.add_argument("--easy-per-term", type=int, default=10)
    parser.add_argument("--retrieve-topk", type=int, default=300)
    parser.add_argument("--chunk-size", type=int, default=128)
    parser.add_argument("--max-features", type=int, default=250_000)
    parser.add_argument("--min-df", type=int, default=2)
    parser.add_argument("--word-ngram-max", type=int, default=2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--limit-terms",
        type=int,
        help="Debug only: mine negatives for the first N train terms.",
    )
    parser.add_argument(
        "--limit-items", type=int, help="Debug only: restrict the item catalog."
    )
    args = parser.parse_args()
    mine_negatives(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pu_lgbm.py
"""Train and score a lexical positive-unlabeled LightGBM model."""

from __future__ import annotations

import argparse
import csv
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from lexical_baseline import load_items, load_terms, pair_features

FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "lexical_score",
]


def rows_to_matrix(
    rows: pd.DataFrame, terms: dict, items: dict, desc: str
) -> np.ndarray:
    matrix = np.empty((len(rows), len(FEATURE_NAMES)), dtype=np.float32)
    for i, row in enumerate(
        tqdm(rows.itertuples(index=False), total=len(rows), desc=desc)
    ):
        features = pair_features(terms[row.term_id], items[row.item_id])
        matrix[i] = [features[name] for name in FEATURE_NAMES]
    return matrix


def sample_frame(
    frame: pd.DataFrame, mask: pd.Series, n: int, seed: int
) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def load_train_term_negatives(path: Path, positives: pd.DataFrame) -> pd.DataFrame:
    negatives = pd.read_csv(
        path, usecols=["term_id", "item_id"], dtype=str, keep_default_na=False
    )
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    positive_keys = set(pair_key(positives))
    keep = ~pair_key(negatives).isin(positive_keys)
    dropped = int((~keep).sum())
    negatives = negatives.loc[keep].reset_index(drop=True)
    if negatives.empty:
        raise ValueError(f"No usable negatives found in {path}")
    if dropped:
        print(f"dropped {dropped:,} negatives that overlap known positives")
    print(f"using train-term negatives from {path} rows={len(negatives):,}")
    return negatives


def build_unlabeled_negatives(args: argparse.Namespace) -> pd.DataFrame:
    negative_path = Path(args.negatives) if args.negatives else None
    if negative_path and negative_path.exists():
        positives = pd.read_csv(
            Path(args.data_dir) / "training_pairs.csv",
            usecols=["term_id", "item_id"],
            dtype=str,
        )
        return load_train_term_negatives(negative_path, positives)

    if not args.allow_submission_negatives:
        raise FileNotFoundError(
            f"Train-term negatives were not found at {negative_path}. "
            "Create train-term negatives first, or pass "
            "`--allow-submission-negatives` to use the older submission-pair fallback."
        )

    data_dir = Path(args.data_dir)
    pairs = pd.read_csv(data_dir / "submission_pairs.csv")
    scores = pd.read_csv(args.lexical_scores, usecols=["score"])
    pairs["score"] = scores["score"].astype("float32").to_numpy()

    parts = [
        sample_frame(pairs, pairs["score"] < args.low_max, args.n_low, args.seed),
        sample_frame(
            pairs,
            (pairs["score"] >= args.low_max) & (pairs["score"] < args.mid_max),
            args.n_mid,
            args.seed + 1,
        ),
        sample_frame(
            pairs,
            (pairs["score"] >= args.mid_max) & (pairs["score"] < args.hard_max),
            args.n_hard,
            args.seed + 2,
        ),
    ]
    negatives = pd.concat(parts, ignore_index=True)
    negatives = negatives.drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


def train(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    model_path = Path(args.model)
    model_path.parent.mkdir(parents=True, exist_ok=True)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")

    positives = pd.read_csv(
        data_dir / "training_pairs.csv", usecols=["term_id", "item_id"]
    )
    negatives = build_unlabeled_negatives(args)

    if args.n_pos and len(positives) > args.n_pos:
        positives = positives.sample(n=args.n_pos, random_state=args.seed)

    x_pos = rows_to_matrix(positives, terms, items, "pos")
    x_neg = rows_to_matrix(negatives, terms, items, "neg")
    x = np.vstack([x_pos, x_neg])
    y = np.concatenate(
        [np.ones(len(x_pos), dtype=np.int8), np.zeros(len(x_neg), dtype=np.int8)]
    )

    x_train, x_valid, y_train, y_valid = train_test_split(
        x, y, test_size=args.valid_size, random_state=args.seed, stratify=y
    )

    clf = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        num_leaves=args.num_leaves,
        min_child_samples=args.min_child_samples,
        subsample=args.subsample,
        colsample_bytree=args.colsample_bytree,
        reg_lambda=args.reg_lambda,
        class_weight="balanced",
        random_state=args.seed,
        n_jobs=args.n_jobs,
    )
    clf.fit(
        x_train,
        y_train,
        eval_set=[(x_valid, y_valid)],
        eval_metric="binary_logloss",
        callbacks=[lgb.log_evaluation(period=50)],
    )

    valid_prob = clf.predict_proba(x_valid)[:, 1]
    best = (0.0, 0.5)
    for threshold in np.linspace(0.05, 0.95, 91):
        pred = (valid_prob >= threshold).astype(np.int8)
        score = f1_score(y_valid, pred, average="macro")
        if score > best[0]:
            best = (score, float(threshold))
    print(f"synthetic valid macro_f1={best[0]:.5f} threshold={best[1]:.3f}")

    importances = pd.Series(clf.feature_importances_, index=FEATURE_NAMES).sort_values(
        ascending=False
    )
    print(importances.head(20).to_string())
    joblib.dump(
        {"model": clf, "features": FEATURE_NAMES, "threshold": best[1]}, model_path
    )
    print(f"wrote {model_path}")


def predict(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    model_bundle = joblib.load(args.model)
    clf = model_bundle["model"]
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")

    with (data_dir / "submission_pairs.csv").open(
        newline="", encoding="utf-8"
    ) as in_handle, out_path.open("w", newline="", encoding="utf-8") as out_handle:
        reader = csv.DictReader(in_handle)
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        ids: list[str] = []
        term_ids: list[str] = []
        rows: list[tuple[str, str]] = []
        for row in tqdm(reader, desc="read/predict"):
            ids.append(row["id"])
            term_ids.append(row["term_id"])
            rows.append((row["term_id"], row["item_id"]))
            if len(rows) >= args.chunk_size:
                write_predictions(writer, clf, ids, term_ids, rows, terms, items)
                ids, term_ids, rows = [], [], []
        if rows:
            write_predictions(writer, clf, ids, term_ids, rows, terms, items)
    print(f"wrote {out_path}")


def write_predictions(writer, clf, ids, term_ids, rows, terms, items) -> None:
    matrix = np.empty((len(rows), len(FEATURE_NAMES)), dtype=np.float32)
    for i, (term_id, item_id) in enumerate(rows):
        features = pair_features(terms[term_id], items[item_id])
        matrix[i] = [features[name] for name in FEATURE_NAMES]
    probs = clf.predict_proba(matrix)[:, 1]
    writer.writerows(
        (row_id, term_id, f"{prob:.8f}")
        for row_id, term_id, prob in zip(ids, term_ids, probs)
    )


def submit_file(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    if args.rate is not None:
        threshold = scores["prob"].quantile(1.0 - args.rate)
    else:
        threshold = args.threshold
    pred = (scores["prob"] >= threshold).astype("int8")
    output = pd.DataFrame({"id": scores["id"], "prediction": pred})
    output.to_csv(args.output, index=False)
    print(f"threshold={threshold:.8f}")
    print(f"wrote {args.output}")
    print(f"rows={len(output)} positives={int(pred.sum())} rate={pred.mean():.5f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    train_cmd = sub.add_parser("train")
    train_cmd.add_argument("--data-dir", default="data")
    train_cmd.add_argument("--lexical-scores", default="outputs/lexical_scores.csv")
    train_cmd.add_argument("--negatives", default="outputs/train_term_negatives.csv")
    train_cmd.add_argument("--allow-submission-negatives", action="store_true")
    train_cmd.add_argument("--model", default="outputs/pu_lgbm.joblib")
    train_cmd.add_argument("--n-pos", type=int, default=0)
    train_cmd.add_argument("--n-low", type=int, default=350_000)
    train_cmd.add_argument("--n-mid", type=int, default=350_000)
    train_cmd.add_argument("--n-hard", type=int, default=150_000)
    train_cmd.add_argument("--low-max", type=float, default=2.0)
    train_cmd.add_argument("--mid-max", type=float, default=4.5)
    train_cmd.add_argument("--hard-max", type=float, default=6.5)
    train_cmd.add_argument("--valid-size", type=float, default=0.15)
    train_cmd.add_argument("--n-estimators", type=int, default=700)
    train_cmd.add_argument("--learning-rate", type=float, default=0.035)
    train_cmd.add_argument("--num-leaves", type=int, default=64)
    train_cmd.add_argument("--min-child-samples", type=int, default=80)
    train_cmd.add_argument("--subsample", type=float, default=0.9)
    train_cmd.add_argument("--colsample-bytree", type=float, default=0.9)
    train_cmd.add_argument("--reg-lambda", type=float, default=2.0)
    train_cmd.add_argument("--n-jobs", type=int, default=-1)
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.set_defaults(func=train)

    pred_cmd = sub.add_parser("predict")
    pred_cmd.add_argument("--data-dir", default="data")
    pred_cmd.add_argument("--model", default="outputs/pu_lgbm.joblib")
    pred_cmd.add_argument("--output", default="outputs/pu_lgbm_scores.csv")
    pred_cmd.add_argument("--chunk-size", type=int, default=100_000)
    pred_cmd.set_defaults(func=predict)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument("--scores", default="outputs/pu_lgbm_scores.csv")
    submit_cmd.add_argument("--output", default="outputs/submission_pu_lgbm.csv")
    submit_cmd.add_argument("--threshold", type=float, default=0.5)
    submit_cmd.add_argument("--rate", type=float)
    submit_cmd.set_defaults(func=submit_file)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pu_catboost.py
"""Train and score a category-aware positive-unlabeled CatBoost model."""

from __future__ import annotations

import argparse
import csv
import gc
import math
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

from lexical_baseline import load_items, load_terms, normalize_text, pair_features

BASE_FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "lexical_score",
]

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5
CATEGORY_FEATURE_NAMES: list[str] = []
for level in CATEGORY_LEVELS:
    CATEGORY_FEATURE_NAMES.extend(
        [
            f"{level}_top1_match",
            f"{level}_top3_match",
            f"{level}_top5_match",
            f"{level}_candidate_prob",
            f"{level}_top1_prob",
            f"{level}_rank",
        ]
    )

ITEM_CAT_FEATURES = ["item_cat_l1", "item_cat_l2", "item_gender", "item_age_group"]
FEATURE_NAMES = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES + ITEM_CAT_FEATURES
CAT_FEATURES = ITEM_CAT_FEATURES


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(
    frame: pd.DataFrame, category_col: str = "category"
) -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_term_category_lookup(
    topk: pd.DataFrame,
    levels: tuple[str, ...] = CATEGORY_LEVELS,
    k: int = CATEGORY_TOPK,
) -> dict[str, dict[str, object]]:
    lookup: dict[str, dict[str, object]] = {}
    for row in topk.itertuples(index=False):
        row_dict = row._asdict()
        entry: dict[str, object] = {}
        for level in levels:
            scores: dict[str, float] = {}
            for rank in range(1, k + 1):
                label = row_dict.get(f"{level}_top{rank}")
                score = row_dict.get(f"{level}_prob{rank}")
                if isinstance(label, str) and label:
                    scores[label] = float(score)
            entry[level] = scores
            entry[f"{level}_top1_prob"] = next(iter(scores.values()), 0.0)
        lookup[row_dict["term_id"]] = entry
    return lookup


def build_item_meta_lookup(items_path: Path) -> dict[str, dict[str, str]]:
    items = pd.read_csv(
        items_path, usecols=["item_id", "category", "gender", "age_group"]
    )
    items = add_category_levels(items)
    lookup: dict[str, dict[str, str]] = {}
    for row in tqdm(
        items.itertuples(index=False), total=len(items), desc="item metadata"
    ):
        lookup[row.item_id] = {
            "cat_l1": row.cat_l1,
            "cat_l2": row.cat_l2,
            "item_cat_l1": row.cat_l1 or "unknown",
            "item_cat_l2": row.cat_l2 or "unknown",
            "item_gender": normalize_text(row.gender) or "unknown",
            "item_age_group": normalize_text(row.age_group) or "unknown",
        }
    return lookup


def category_pair_features(
    term_id: str,
    item_id: str,
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
) -> dict[str, float | str]:
    term_entry = term_category_lookup.get(term_id, {})
    item_entry = item_meta_lookup.get(item_id, {})
    out: dict[str, float | str] = {
        "item_cat_l1": item_entry.get("item_cat_l1", "unknown"),
        "item_cat_l2": item_entry.get("item_cat_l2", "unknown"),
        "item_gender": item_entry.get("item_gender", "unknown"),
        "item_age_group": item_entry.get("item_age_group", "unknown"),
    }

    for level in CATEGORY_LEVELS:
        scores = term_entry.get(level, {})
        if not isinstance(scores, dict):
            scores = {}
        ordered_labels = list(scores.keys())
        candidate = item_entry.get(level, "")
        rank = ordered_labels.index(candidate) + 1 if candidate in scores else 0

        out[f"{level}_top1_match"] = float(rank == 1)
        out[f"{level}_top3_match"] = float(1 <= rank <= 3)
        out[f"{level}_top5_match"] = float(1 <= rank <= 5)
        out[f"{level}_candidate_prob"] = float(scores.get(candidate, 0.0))
        out[f"{level}_top1_prob"] = float(term_entry.get(f"{level}_top1_prob", 0.0))
        out[f"{level}_rank"] = float(rank)

    return out


def rows_to_frame(
    rows: pd.DataFrame,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str]],
    term_category_lookup: dict[str, dict[str, object]],
    item_meta_lookup: dict[str, dict[str, str]],
    desc: str,
) -> pd.DataFrame:
    numeric_names = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric = np.empty((len(rows), len(numeric_names)), dtype=np.float32)
    cat_values = {name: np.empty(len(rows), dtype=object) for name in ITEM_CAT_FEATURES}

    iterator = tqdm(rows.itertuples(index=False), total=len(rows), desc=desc)
    for i, row in enumerate(iterator):
        features: dict[str, float | str] = pair_features(
            terms[row.term_id], items[row.item_id]
        )
        features.update(
            category_pair_features(
                row.term_id, row.item_id, term_category_lookup, item_meta_lookup
            )
        )
        numeric[i] = [float(features.get(name, 0.0)) for name in numeric_names]
        for name in ITEM_CAT_FEATURES:
            cat_values[name][i] = features.get(name, "unknown") or "unknown"

    frame = pd.DataFrame(numeric, columns=numeric_names)
    for name in ITEM_CAT_FEATURES:
        frame[name] = cat_values[name]
    return frame[FEATURE_NAMES]


def sample_frame(
    frame: pd.DataFrame, mask: pd.Series, n: int, seed: int
) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def pair_key(frame: pd.DataFrame) -> pd.Series:
    return frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)


def load_train_term_negatives(path: Path, positives: pd.DataFrame) -> pd.DataFrame:
    negatives = pd.read_csv(
        path, usecols=["term_id", "item_id"], dtype=str, keep_default_na=False
    )
    negatives = negatives.drop_duplicates(["term_id", "item_id"]).reset_index(drop=True)
    positive_keys = set(pair_key(positives))
    keep = ~pair_key(negatives).isin(positive_keys)
    dropped = int((~keep).sum())
    negatives = negatives.loc[keep].reset_index(drop=True)
    if negatives.empty:
        raise ValueError(f"No usable negatives found in {path}")
    if dropped:
        print(f"dropped {dropped:,} negatives that overlap known positives")
    print(f"using train-term negatives from {path} rows={len(negatives):,}")
    return negatives


def build_unlabeled_negatives(args: argparse.Namespace) -> pd.DataFrame:
    negative_path = Path(args.negatives) if args.negatives else None
    if negative_path and negative_path.exists():
        positives = pd.read_csv(
            Path(args.data_dir) / "training_pairs.csv",
            usecols=["term_id", "item_id"],
            dtype=str,
        )
        return load_train_term_negatives(negative_path, positives)

    if not args.allow_submission_negatives:
        raise FileNotFoundError(
            f"Train-term negatives were not found at {negative_path}. "
            "Create train-term negatives first, or pass "
            "`--allow-submission-negatives` to use the older submission-pair fallback."
        )

    data_dir = Path(args.data_dir)
    pairs = pd.read_csv(
        data_dir / "submission_pairs.csv", usecols=["id", "term_id", "item_id"]
    )
    scores = pd.read_csv(args.lexical_scores, usecols=["score"])
    if len(pairs) != len(scores):
        raise ValueError("submission_pairs and lexical_scores row counts differ")

    pairs["score"] = scores["score"].astype("float32").to_numpy()
    parts = [
        sample_frame(pairs, pairs["score"] < args.low_max, args.n_low, args.seed),
        sample_frame(
            pairs,
            (pairs["score"] >= args.low_max) & (pairs["score"] < args.mid_max),
            args.n_mid,
            args.seed + 1,
        ),
        sample_frame(
            pairs,
            (pairs["score"] >= args.mid_max) & (pairs["score"] < args.hard_max),
            args.n_hard,
            args.seed + 2,
        ),
    ]
    negatives = pd.concat(parts, ignore_index=True).drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


def load_feature_context(args: argparse.Namespace):
    data_dir = Path(args.data_dir)
    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")
    item_meta_lookup = build_item_meta_lookup(data_dir / "items.csv")

    topk_path = Path(args.term_category_topk)
    if not topk_path.exists():
        raise FileNotFoundError(
            f"term category top-k file not found: {topk_path}. "
            "Run the category-aware notebook first or pass --term-category-topk."
        )
    term_category_topk = pd.read_csv(topk_path)
    term_category_lookup = build_term_category_lookup(term_category_topk)
    return terms, items, term_category_lookup, item_meta_lookup


def train(args: argparse.Namespace) -> None:
    start_time = time.time()
    model_path = Path(args.model)
    model_path.parent.mkdir(parents=True, exist_ok=True)

    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)
    positives = pd.read_csv(
        Path(args.data_dir) / "training_pairs.csv", usecols=["term_id", "item_id"]
    )
    negatives = build_unlabeled_negatives(args)

    if args.n_pos and len(positives) > args.n_pos:
        positives = positives.sample(n=args.n_pos, random_state=args.seed)

    x_pos = rows_to_frame(
        positives,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        "positive features",
    )
    x_neg = rows_to_frame(
        negatives,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        "negative features",
    )
    x = pd.concat([x_pos, x_neg], ignore_index=True)
    y = np.concatenate(
        [np.ones(len(x_pos), dtype=np.int8), np.zeros(len(x_neg), dtype=np.int8)]
    )
    print(f"matrix={x.shape} positives={int(y.sum()):,} positive_rate={y.mean():.4f}")

    del x_pos, x_neg
    gc.collect()

    x_train, x_valid, y_train, y_valid = train_test_split(
        x,
        y,
        test_size=args.valid_size,
        random_state=args.seed,
        stratify=y,
    )
    del x
    gc.collect()

    model_params = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": args.iterations,
        "learning_rate": args.learning_rate,
        "depth": args.depth,
        "l2_leaf_reg": args.l2_leaf_reg,
        "random_strength": args.random_strength,
        "bootstrap_type": "Bernoulli",
        "subsample": args.subsample,
        "auto_class_weights": "Balanced",
        "random_seed": args.seed,
        "thread_count": args.thread_count,
        "allow_writing_files": False,
        "verbose": args.verbose,
        "task_type": args.task_type,
        "devices": args.devices,
    }
    if args.task_type == "CPU":
        model_params["rsm"] = args.rsm
    elif args.rsm != 1.0:
        print("CatBoost GPU ignores rsm for Logloss; training without rsm.")

    model = CatBoostClassifier(**model_params)

    train_pool = Pool(x_train, y_train, cat_features=CAT_FEATURES)
    valid_pool = Pool(x_valid, y_valid, cat_features=CAT_FEATURES)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=False)

    valid_prob = model.predict_proba(valid_pool)[:, 1]
    best = (0.0, 0.5)
    for threshold in np.linspace(0.05, 0.95, 91):
        pred = (valid_prob >= threshold).astype(np.int8)
        score = f1_score(y_valid, pred, average="macro")
        if score > best[0]:
            best = (float(score), float(threshold))
    print(f"synthetic valid macro_f1={best[0]:.5f} threshold={best[1]:.3f}")

    importances = pd.Series(
        model.get_feature_importance(valid_pool), index=FEATURE_NAMES
    ).sort_values(ascending=False)
    print(importances.head(35).to_string())

    model.save_model(str(model_path))
    joblib.dump(
        {
            "features": FEATURE_NAMES,
            "cat_features": CAT_FEATURES,
            "threshold": best[1],
            "iterations": args.iterations,
            "term_category_topk": str(args.term_category_topk),
        },
        model_path.with_suffix(".meta.joblib"),
    )
    elapsed = time.time() - start_time
    print(f"wrote {model_path}")
    print(f"elapsed_minutes={elapsed / 60:.1f}")


def predict(args: argparse.Namespace) -> None:
    data_dir = Path(args.data_dir)
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    model = CatBoostClassifier()
    model.load_model(str(args.model))
    terms, items, term_category_lookup, item_meta_lookup = load_feature_context(args)

    written = 0
    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(
            data_dir / "submission_pairs.csv", chunksize=args.chunk_size
        )
        for chunk in tqdm(reader, desc="predict chunks"):
            matrix = rows_to_frame(
                chunk,
                terms,
                items,
                term_category_lookup,
                item_meta_lookup,
                "chunk features",
            )
            pool = Pool(matrix, cat_features=CAT_FEATURES)
            prob = model.predict_proba(pool)[:, 1]
            writer.writerows(
                (row_id, term_id, f"{score:.8f}")
                for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob)
            )
            written += len(chunk)
            del matrix, pool, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")


def make_submission(args: argparse.Namespace) -> None:
    scores = pd.read_csv(args.scores)
    if args.rate is not None:
        threshold = scores["prob"].quantile(1.0 - args.rate)
    else:
        threshold = args.threshold
    pred = (scores["prob"] >= threshold).astype("int8")
    output = pd.DataFrame({"id": scores["id"], "prediction": pred})
    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(out_path, index=False)
    print(f"threshold={threshold:.8f}")
    print(f"wrote {out_path}")
    print(f"rows={len(output):,} positives={int(pred.sum()):,} rate={pred.mean():.5f}")


def main() -> None:
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    common = argparse.ArgumentParser(add_help=False)
    common.add_argument("--data-dir", default="data")
    common.add_argument(
        "--term-category-topk",
        default="outputs/category_aware_notebook/term_category_topk.csv",
    )

    train_cmd = sub.add_parser("train", parents=[common])
    train_cmd.add_argument("--lexical-scores", default="outputs/lexical_scores.csv")
    train_cmd.add_argument("--negatives", default="outputs/train_term_negatives.csv")
    train_cmd.add_argument("--allow-submission-negatives", action="store_true")
    train_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    train_cmd.add_argument("--n-pos", type=int, default=0)
    train_cmd.add_argument("--n-low", type=int, default=350_000)
    train_cmd.add_argument("--n-mid", type=int, default=350_000)
    train_cmd.add_argument("--n-hard", type=int, default=150_000)
    train_cmd.add_argument("--low-max", type=float, default=2.0)
    train_cmd.add_argument("--mid-max", type=float, default=4.5)
    train_cmd.add_argument("--hard-max", type=float, default=6.5)
    train_cmd.add_argument("--valid-size", type=float, default=0.15)
    train_cmd.add_argument("--iterations", type=int, default=5_000)
    train_cmd.add_argument("--learning-rate", type=float, default=0.025)
    train_cmd.add_argument("--depth", type=int, default=6)
    train_cmd.add_argument("--l2-leaf-reg", type=float, default=6.0)
    train_cmd.add_argument("--random-strength", type=float, default=0.5)
    train_cmd.add_argument("--subsample", type=float, default=0.85)
    train_cmd.add_argument("--rsm", type=float, default=0.95)
    train_cmd.add_argument("--thread-count", type=int, default=-1)
    train_cmd.add_argument("--verbose", type=int, default=100)
    train_cmd.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    train_cmd.add_argument("--devices", default="0")
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.set_defaults(func=train)

    pred_cmd = sub.add_parser("predict", parents=[common])
    pred_cmd.add_argument("--model", default="outputs/catboost_category_aware.cbm")
    pred_cmd.add_argument(
        "--output", default="outputs/catboost_category_aware_scores.csv"
    )
    pred_cmd.add_argument("--chunk-size", type=int, default=150_000)
    pred_cmd.set_defaults(func=predict)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument(
        "--scores", default="outputs/catboost_category_aware_scores.csv"
    )
    submit_cmd.add_argument("--output", default="outputs/catboost_category_aware.csv")
    submit_cmd.add_argument("--threshold", type=float, default=0.5)
    submit_cmd.add_argument("--rate", type=float)
    submit_cmd.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/category_signal.py
"""Learn query-to-category priors for category-aware relevance models."""

from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from tqdm import tqdm

from lexical_baseline import normalize_text

CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(
    frame: pd.DataFrame, category_col: str = "category"
) -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_category_examples(
    terms: pd.DataFrame,
    items: pd.DataFrame,
    positive_pairs: pd.DataFrame,
    max_examples: int | None = None,
    seed: int = 42,
) -> pd.DataFrame:
    required_term_ids = set(positive_pairs["term_id"].astype(str))
    required_item_ids = set(positive_pairs["item_id"].astype(str))
    term_subset = terms.loc[
        terms["term_id"].astype(str).isin(required_term_ids), ["term_id", "query"]
    ]
    item_subset = items.loc[
        items["item_id"].astype(str).isin(required_item_ids), ["item_id", "category"]
    ]

    examples = (
        positive_pairs[["term_id", "item_id"]]
        .merge(term_subset, on="term_id", how="left", validate="many_to_one")
        .merge(item_subset, on="item_id", how="left", validate="many_to_one")
        .dropna(subset=["query", "category"])
    )
    examples = add_category_levels(examples)
    examples["query_norm"] = examples["query"].map(normalize_text)

    # A broad query can have hundreds of positive products in the same category.
    # Keep one query/category path so it does not dominate the category classifier.
    examples = examples.drop_duplicates(["query_norm", "cat_l2"])
    if max_examples is not None and len(examples) > max_examples:
        examples = examples.sample(n=max_examples, random_state=seed)

    columns = ["query_norm", "cat_l1", "cat_l2", "cat_l3", "cat_leaf"]
    return examples[columns].reset_index(drop=True)


@dataclass
class ConstantCategoryModel:
    """Return a fixed category distribution when training has one class."""

    label: str

    def __post_init__(self) -> None:
        self.classes_ = np.asarray([self.label], dtype=object)

    def predict_proba(self, matrix: Any) -> np.ndarray:
        return np.ones((matrix.shape[0], 1), dtype=np.float32)


class CategorySignal:
    """Fit TF-IDF category classifiers and return each query's top categories."""

    def __init__(
        self,
        levels: tuple[str, ...] = CATEGORY_LEVELS,
        topk: int = CATEGORY_TOPK,
        max_features: int = 120_000,
        max_iter: int = 80,
        min_class_count: int = 2,
        seed: int = 42,
    ):
        self.levels = levels
        self.topk = topk
        self.max_features = max_features
        self.max_iter = max_iter
        self.min_class_count = min_class_count
        self.seed = seed
        self.vectorizer: TfidfVectorizer | None = None
        self.models: dict[str, Any] = {}

    def fit(self, examples: pd.DataFrame) -> "CategorySignal":
        if examples.empty:
            raise ValueError("CategorySignal requires at least one training example")

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2 if len(examples) >= 2 else 1,
            max_features=self.max_features,
            dtype=np.float32,
            sublinear_tf=True,
        )
        x_all = self.vectorizer.fit_transform(examples["query_norm"])
        print(f"category tfidf shape={x_all.shape}")

        for level in self.levels:
            counts = examples[level].value_counts()
            keep_labels = counts.index[counts >= self.min_class_count]
            if len(keep_labels) == 0:
                keep_labels = counts.index[:1]
            keep = examples[level].isin(keep_labels).to_numpy()
            y = examples.loc[keep, level].astype(str).to_numpy()
            x = x_all[keep]
            classes = np.unique(y)

            start = time.time()
            if len(classes) == 1:
                model: Any = ConstantCategoryModel(str(classes[0]))
            else:
                model = SGDClassifier(
                    loss="log_loss",
                    penalty="l2",
                    alpha=2e-5,
                    max_iter=self.max_iter,
                    tol=1e-3,
                    class_weight="balanced",
                    random_state=self.seed,
                    n_jobs=-1,
                )
                model.fit(x, y)
            self.models[level] = model
            elapsed = time.time() - start
            print(
                f"{level}: classes={len(classes):,} examples={len(y):,} seconds={elapsed:.1f}"
            )
        return self

    def predict_terms(
        self, terms: pd.DataFrame, chunk_size: int = 25_000
    ) -> pd.DataFrame:
        if self.vectorizer is None:
            raise RuntimeError("fit CategorySignal before predicting")
        if terms["term_id"].duplicated().any():
            raise ValueError("terms contains duplicate term_id values")

        term_ids = terms["term_id"].astype(str).to_numpy()
        queries = terms["query"].map(normalize_text).to_numpy()
        chunks: list[pd.DataFrame] = []

        for start in tqdm(range(0, len(terms), chunk_size), desc="category topk"):
            end = min(start + chunk_size, len(terms))
            matrix = self.vectorizer.transform(queries[start:end])
            out: dict[str, object] = {"term_id": term_ids[start:end]}

            for level, model in self.models.items():
                proba = model.predict_proba(matrix)
                k = min(self.topk, proba.shape[1])
                if k == proba.shape[1]:
                    indices = np.argsort(-proba, axis=1)[:, :k]
                else:
                    indices = np.argpartition(proba, -k, axis=1)[:, -k:]
                    local_order = np.argsort(
                        -np.take_along_axis(proba, indices, axis=1), axis=1
                    )
                    indices = np.take_along_axis(indices, local_order, axis=1)
                scores = np.take_along_axis(proba, indices, axis=1).astype(np.float32)
                labels = model.classes_[indices]

                for rank in range(k):
                    out[f"{level}_top{rank + 1}"] = labels[:, rank]
                    out[f"{level}_prob{rank + 1}"] = scores[:, rank]

            chunks.append(pd.DataFrame(out))

        if not chunks:
            return pd.DataFrame(columns=["term_id"])
        return pd.concat(chunks, ignore_index=True)


In [ ]:
%%writefile src/ensemble_scores.py
"""Align, rank-normalize, and blend model scores into competition submissions."""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

SCORE_COLUMNS = ("prob", "score")


def score_col(frame: pd.DataFrame, path: str | Path) -> str:
    for column in SCORE_COLUMNS:
        if column in frame.columns:
            return column
    raise ValueError(f"{path} must contain one of {SCORE_COLUMNS}")


def load_score(path: str | Path, name: str, nrows: int | None = None) -> pd.DataFrame:
    frame = pd.read_csv(
        path, dtype={"id": str, "term_id": str}, keep_default_na=False, nrows=nrows
    )
    required = {"id", "term_id"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")

    column = score_col(frame, path)
    out = frame[["id", "term_id", column]].copy()
    out = out.rename(columns={column: name})
    out[name] = pd.to_numeric(out[name], errors="coerce").astype("float32")

    if out["id"].duplicated().any():
        examples = out.loc[out["id"].duplicated(), "id"].head(5).tolist()
        raise ValueError(f"{path} contains duplicate ids, examples={examples}")
    if out[name].isna().any():
        raise ValueError(f"{path} contains null/non-numeric scores")
    return out


def rank_normalize(frame: pd.DataFrame, column: str, scope: str) -> pd.Series:
    if scope == "global":
        return frame[column].rank(method="average", pct=True).astype("float32")
    if scope == "term":
        return (
            frame.groupby("term_id", sort=False)[column]
            .rank(method="average", pct=True)
            .astype("float32")
        )
    raise ValueError(f"Unknown rank scope: {scope}")


def align_scores(
    catboost: pd.DataFrame, lgbm: pd.DataFrame, transformer: pd.DataFrame
) -> pd.DataFrame:
    merged = catboost.merge(
        lgbm[["id", "term_id", "lgbm"]], on="id", how="inner", suffixes=("", "_lgbm")
    )
    if "term_id_lgbm" in merged.columns:
        mismatch = merged["term_id"].ne(merged["term_id_lgbm"])
        if mismatch.any():
            raise ValueError(
                f"CatBoost/LGBM term_id mismatch rows={int(mismatch.sum())}"
            )
        merged = merged.drop(columns=["term_id_lgbm"])

    merged = merged.merge(
        transformer[["id", "term_id", "transformer"]],
        on="id",
        how="inner",
        suffixes=("", "_transformer"),
    )
    if "term_id_transformer" in merged.columns:
        mismatch = merged["term_id"].ne(merged["term_id_transformer"])
        if mismatch.any():
            raise ValueError(
                f"GBDT/transformer term_id mismatch rows={int(mismatch.sum())}"
            )
        merged = merged.drop(columns=["term_id_transformer"])

    expected = len(catboost)
    if len(merged) != expected:
        raise ValueError(
            f"Score files do not align on id: expected={expected:,} merged={len(merged):,}"
        )
    return merged


def normalized_weighted_sum(
    columns: list[pd.Series], weights: list[float]
) -> pd.Series:
    total = float(sum(weights))
    if total <= 0:
        raise ValueError("Blend weights must sum to a positive value")
    output = np.zeros(len(columns[0]), dtype=np.float32)
    for column, weight in zip(columns, weights):
        output += column.to_numpy(dtype=np.float32) * np.float32(weight / total)
    return pd.Series(output, index=columns[0].index, dtype="float32")


def report_correlations(frame: pd.DataFrame) -> None:
    rank_cols = ["catboost_rank", "lgbm_rank", "transformer_rank", "gbdt_rank", "prob"]
    corr = frame[rank_cols].corr(method="pearson")
    print("rank correlation matrix")
    print(corr.to_string(float_format=lambda value: f"{value:.4f}"))


def blend(args: argparse.Namespace) -> None:
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    nrows = args.limit_rows or None

    catboost = load_score(args.catboost, "catboost", nrows=nrows)
    lgbm = load_score(args.lgbm, "lgbm", nrows=nrows)
    transformer = load_score(args.transformer, "transformer", nrows=nrows)
    merged = align_scores(catboost, lgbm, transformer)

    merged["catboost_rank"] = rank_normalize(merged, "catboost", args.rank_scope)
    merged["lgbm_rank"] = rank_normalize(merged, "lgbm", args.rank_scope)
    merged["transformer_rank"] = rank_normalize(merged, "transformer", args.rank_scope)
    merged["gbdt_score"] = normalized_weighted_sum(
        [merged["catboost_rank"], merged["lgbm_rank"]],
        [args.catboost_weight, args.lgbm_weight],
    )
    merged["gbdt_rank"] = rank_normalize(merged, "gbdt_score", args.rank_scope)
    merged["prob"] = normalized_weighted_sum(
        [merged["gbdt_rank"], merged["transformer_rank"]],
        [args.gbdt_weight, args.transformer_weight],
    )

    columns = ["id", "term_id", "prob"]
    if args.include_components:
        columns.extend(["gbdt_score", "catboost_rank", "lgbm_rank", "transformer_rank"])
    merged[columns].to_csv(output_path, index=False)

    print(f"wrote {output_path} rows={len(merged):,}")
    print(
        "weights "
        f"gbdt=({args.catboost_weight:.3f} catboost, {args.lgbm_weight:.3f} lgbm) "
        f"final=({args.gbdt_weight:.3f} gbdt, {args.transformer_weight:.3f} transformer)"
    )
    if args.report_correlations:
        report_correlations(merged)


def rate_suffix(rate: float) -> str:
    return f"r{int(round(rate * 100)):02d}"


def make_prediction(
    scores: pd.Series,
    rate: float,
    term_ids: pd.Series | None = None,
    min_positive_excess_base: int = 0,
) -> np.ndarray:
    if not 0.0 < rate < 1.0:
        raise ValueError(f"rate must be between 0 and 1, got {rate}")
    n_pos = int(round(len(scores) * rate))
    prediction = np.zeros(len(scores), dtype=np.int8)
    if n_pos <= 0:
        return prediction

    if min_positive_excess_base <= 0:
        selected = scores.nlargest(n_pos).index
        prediction[selected] = 1
        return prediction

    if term_ids is None:
        raise ValueError(
            "term_ids are required when min_positive_excess_base is enabled"
        )
    if len(term_ids) != len(scores):
        raise ValueError("term_ids and scores must have the same number of rows")

    # The competition candidate set appears to contain a base retrieval slate of
    # 100 items per term, unioned with known positives that retrieval missed.  If
    # that inference is correct, a term with N candidates has at least N - 100
    # positives.  Protect the highest-scoring rows needed to satisfy that lower
    # bound, then spend the remaining global positive budget normally.
    frame = pd.DataFrame(
        {
            "term_id": term_ids.astype(str).to_numpy(),
            "score": scores.to_numpy(dtype=np.float64),
        }
    )
    group_size = frame.groupby("term_id", sort=False)["score"].transform("size")
    minimum = (group_size - min_positive_excess_base).clip(lower=0)
    within_term_rank = frame.groupby("term_id", sort=False)["score"].rank(
        method="first",
        ascending=False,
    )
    mandatory = within_term_rank.le(minimum).to_numpy()
    n_mandatory = int(mandatory.sum())
    if n_mandatory > n_pos:
        minimum_rate = n_mandatory / len(scores)
        raise ValueError(
            "Requested positive rate is too small for the per-term lower bounds: "
            f"mandatory={n_mandatory:,} budget={n_pos:,} minimum_rate={minimum_rate:.6f}"
        )

    prediction[mandatory] = 1
    remaining = n_pos - n_mandatory
    if remaining:
        eligible_positions = np.flatnonzero(~mandatory)
        eligible_scores = frame.loc[~mandatory, "score"].to_numpy()
        if remaining >= len(eligible_positions):
            selected_positions = eligible_positions
        else:
            selected_local = np.argpartition(eligible_scores, -remaining)[-remaining:]
            selected_positions = eligible_positions[selected_local]
        prediction[selected_positions] = 1
    return prediction


def submit(args: argparse.Namespace) -> None:
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    scores = pd.read_csv(
        args.scores,
        dtype={"id": str},
        keep_default_na=False,
        nrows=args.limit_rows or None,
    )
    scores = scores.reset_index(drop=True)
    column = args.score_col or score_col(scores, args.scores)
    scores[column] = pd.to_numeric(scores[column], errors="coerce")
    if scores[column].isna().any():
        raise ValueError(f"{args.scores} contains null/non-numeric values in {column}")

    for rate in args.rates:
        prediction = make_prediction(
            scores[column],
            rate,
            term_ids=scores["term_id"] if "term_id" in scores.columns else None,
            min_positive_excess_base=args.min_positive_excess_base,
        )
        output = pd.DataFrame({"id": scores["id"], "prediction": prediction})
        output_path = output_dir / f"{args.prefix}_{rate_suffix(rate)}.csv"
        output.to_csv(output_path, index=False)
        print(
            f"{output_path}: rows={len(output):,} positives={int(prediction.sum()):,} "
            f"rate={prediction.mean():.5f} "
            f"min_positive_excess_base={args.min_positive_excess_base}"
        )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Rank-normalized two-level GBDT + transformer blending."
    )
    sub = parser.add_subparsers(dest="cmd", required=True)

    blend_cmd = sub.add_parser("blend")
    blend_cmd.add_argument(
        "--catboost", default="outputs/catboost_category_aware_2026-06-30_scores.csv"
    )
    blend_cmd.add_argument("--lgbm", default="outputs/pu_lgbm_v1_scores.csv")
    blend_cmd.add_argument(
        "--transformer", default="outputs/transformer_biencoder_scores.csv"
    )
    blend_cmd.add_argument("--output", default="outputs/moe_gbdt_rankblend_scores.csv")
    blend_cmd.add_argument("--catboost-weight", type=float, default=0.70)
    blend_cmd.add_argument("--lgbm-weight", type=float, default=0.30)
    blend_cmd.add_argument("--gbdt-weight", type=float, default=0.50)
    blend_cmd.add_argument("--transformer-weight", type=float, default=0.50)
    blend_cmd.add_argument("--rank-scope", choices=["global", "term"], default="global")
    blend_cmd.add_argument("--include-components", action="store_true")
    blend_cmd.add_argument("--report-correlations", action="store_true")
    blend_cmd.add_argument("--limit-rows", type=int, default=0)
    blend_cmd.set_defaults(func=blend)

    submit_cmd = sub.add_parser("submit")
    submit_cmd.add_argument("--scores", default="outputs/moe_gbdt_rankblend_scores.csv")
    submit_cmd.add_argument("--output-dir", default="outputs")
    submit_cmd.add_argument("--prefix", default="submission_moe_gbdt_rankblend")
    submit_cmd.add_argument(
        "--rates", type=float, nargs="+", default=[0.18, 0.20, 0.22, 0.24]
    )
    submit_cmd.add_argument("--score-col")
    submit_cmd.add_argument(
        "--min-positive-excess-base",
        type=int,
        default=0,
        help=(
            "Opt-in per-term constraint: require at least max(candidate_count - BASE, 0) "
            "positives per term before filling the remaining global rate budget. "
            "Use 100 to test the inferred base-100 candidate construction."
        ),
    )
    submit_cmd.add_argument("--limit-rows", type=int, default=0)
    submit_cmd.set_defaults(func=submit)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/grouped_oof_validation.py
"""Run leakage-aware term-grouped validation for tree and transformer models."""

from __future__ import annotations

import argparse
import csv
import gc
import itertools
import json
import time
from pathlib import Path
from typing import Iterable

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold
from tqdm import tqdm

from category_signal import CategorySignal, build_category_examples
from pu_catboost import (
    BASE_FEATURE_NAMES,
    CAT_FEATURES,
    CATEGORY_FEATURE_NAMES,
    build_item_meta_lookup,
    build_term_category_lookup,
    rows_to_frame,
)
from lexical_baseline import load_items, load_terms, normalize_text
from train_term_negatives import item_search_text, topk_sparse_row

SLATE_COLUMNS = [
    "slate_id",
    "term_id",
    "item_id",
    "fold",
    "label",
    "candidate_count",
    "is_retrieved",
    "retrieval_rank",
    "retrieval_score",
]
OPTIONAL_SEMANTIC_FEATURES = ["semantic_cosine", "semantic_rank_pct"]


def load_positive_pairs(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in frame.columns:
        frame = frame.loc[frame["label"].astype(str).ne("0")]
    frame = frame[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if frame.empty:
        raise ValueError(f"No positive pairs found in {path}")
    return frame


def make_term_folds(positives: pd.DataFrame, n_splits: int, seed: int) -> pd.DataFrame:
    counts = (
        positives.groupby("term_id", sort=True)
        .size()
        .rename("n_positives")
        .reset_index()
    )
    if len(counts) < n_splits:
        raise ValueError(f"Need at least {n_splits} terms, found {len(counts)}")

    # Stratifying quantiles of query breadth distributes both head and tail queries
    # across folds while term_id grouping prevents any query leakage.
    n_bins = max(1, min(10, len(counts) // n_splits))
    bins = pd.qcut(
        counts["n_positives"].rank(method="first"),
        q=n_bins,
        labels=False,
        duplicates="drop",
    ).astype(int)
    counts["fold"] = np.int16(-1)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dummy = np.zeros(len(counts), dtype=np.int8)
    for fold, (_, valid_index) in enumerate(splitter.split(dummy, bins)):
        counts.loc[valid_index, "fold"] = np.int16(fold)

    if counts["fold"].lt(0).any():
        raise AssertionError("Some terms were not assigned to a fold")
    return counts[["term_id", "fold", "n_positives"]]


def sample_zero_score_items(
    item_count: int,
    selected: set[int],
    needed: int,
    rng: np.random.Generator,
) -> list[int]:
    output: list[int] = []
    max_attempts = max(1_000, needed * 100)
    attempts = 0
    while len(output) < needed and attempts < max_attempts:
        position = int(rng.integers(0, item_count))
        attempts += 1
        if position in selected:
            continue
        selected.add(position)
        output.append(position)
    if len(output) != needed:
        for position in range(item_count):
            if position in selected:
                continue
            selected.add(position)
            output.append(position)
            if len(output) == needed:
                break
    if len(output) != needed:
        raise ValueError(f"Could not fill a {needed}-item zero-score retrieval tail")
    return output


def restrict_debug_catalog(
    items: pd.DataFrame,
    positives: pd.DataFrame,
    limit_items: int,
) -> pd.DataFrame:
    if limit_items <= 0 or len(items) <= limit_items:
        return items
    required = set(positives["item_id"].astype(str))
    keep = items["item_id"].astype(str).isin(required)
    head_positions = np.arange(len(items)) < limit_items
    restricted = (
        items.loc[keep | head_positions]
        .drop_duplicates("item_id")
        .reset_index(drop=True)
    )
    print(
        f"debug catalog restricted from {len(items):,} to {len(restricted):,}; "
        f"retained_positive_items={len(required):,}"
    )
    return restricted


def build_slates(args: argparse.Namespace) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    positives = load_positive_pairs(data_dir / "training_pairs.csv")
    terms = pd.read_csv(data_dir / "terms.csv", dtype=str, keep_default_na=False)
    train_terms = terms.loc[
        terms["term_id"].isin(set(positives["term_id"])), ["term_id", "query"]
    ].copy()
    train_terms = train_terms.sort_values("term_id").reset_index(drop=True)
    if args.limit_terms:
        train_terms = train_terms.head(args.limit_terms).copy()
        positives = positives.loc[
            positives["term_id"].isin(set(train_terms["term_id"]))
        ].reset_index(drop=True)

    missing_terms = set(positives["term_id"]) - set(train_terms["term_id"])
    if missing_terms:
        raise KeyError(f"Missing {len(missing_terms)} positive term ids from terms.csv")

    folds = make_term_folds(positives, args.n_splits, args.seed)
    folds_path = output_dir / "term_folds.csv"
    folds.to_csv(folds_path, index=False)
    fold_by_term = folds.set_index("term_id")["fold"].astype(int).to_dict()

    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "title", "category", "brand", "attributes"],
    )
    items = restrict_debug_catalog(items, positives, args.limit_items)
    if len(items) < args.base_candidates:
        raise ValueError(
            f"Catalog has {len(items)} items, fewer than base_candidates={args.base_candidates}"
        )

    item_ids = items["item_id"].astype(str).to_numpy()
    item_to_position = {item_id: position for position, item_id in enumerate(item_ids)}
    missing_items = set(positives["item_id"]) - set(item_to_position)
    if missing_items:
        raise KeyError(f"Missing {len(missing_items)} positive item ids from items.csv")

    train_terms["query_norm"] = train_terms["query"].map(normalize_text)
    item_text = item_search_text(items)
    corpus = pd.concat([train_terms["query_norm"], item_text], ignore_index=True)
    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, args.word_ngram_max),
        min_df=args.min_df,
        max_features=args.max_features,
        dtype=np.float32,
        norm="l2",
        sublinear_tf=True,
    )
    vectorizer.fit(corpus)
    joblib.dump(vectorizer, output_dir / "retrieval_vectorizer.joblib")
    term_matrix = vectorizer.transform(train_terms["query_norm"])
    item_matrix_t = vectorizer.transform(item_text).T.tocsr()
    print(
        f"terms={len(train_terms):,} catalog_items={len(items):,} positives={len(positives):,} "
        f"vocab={len(vectorizer.vocabulary_):,}"
    )

    positives_by_term: dict[str, list[int]] = {}
    for term_id, group in positives.groupby("term_id", sort=False):
        positives_by_term[str(term_id)] = [
            item_to_position[item_id] for item_id in group["item_id"].astype(str)
        ]

    slate_path = output_dir / "validation_slates.csv"
    rng = np.random.default_rng(args.seed)
    row_id = 0
    positive_rows = 0
    retrieved_positive_rows = 0
    rows_by_fold = np.zeros(args.n_splits, dtype=np.int64)

    with slate_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(SLATE_COLUMNS)

        for start in tqdm(
            range(0, len(train_terms), args.chunk_size), desc="build validation slates"
        ):
            stop = min(start + args.chunk_size, len(train_terms))
            score_chunk = (term_matrix[start:stop] @ item_matrix_t).tocsr()

            for row_offset, term_id in enumerate(
                train_terms["term_id"].iloc[start:stop].astype(str)
            ):
                indices, scores = topk_sparse_row(
                    score_chunk.getrow(row_offset), args.base_candidates
                )
                retrieved_positions = [
                    int(value) for value in indices[: args.base_candidates]
                ]
                retrieved_scores = [
                    float(value) for value in scores[: args.base_candidates]
                ]
                selected = set(retrieved_positions)
                missing = args.base_candidates - len(retrieved_positions)
                if missing:
                    fill = sample_zero_score_items(len(items), selected, missing, rng)
                    retrieved_positions.extend(fill)
                    retrieved_scores.extend([0.0] * len(fill))

                positive_positions = positives_by_term[term_id]
                positive_set = set(positive_positions)
                appended_positions = sorted(positive_set - set(retrieved_positions))
                candidate_positions = retrieved_positions + appended_positions
                candidate_count = len(candidate_positions)
                fold = int(fold_by_term[term_id])

                retrieval_metadata = {
                    position: (rank, score)
                    for rank, (position, score) in enumerate(
                        zip(retrieved_positions, retrieved_scores),
                        start=1,
                    )
                }
                for position in candidate_positions:
                    label = int(position in positive_set)
                    rank, retrieval_score = retrieval_metadata.get(position, (0, 0.0))
                    is_retrieved = int(rank > 0)
                    writer.writerow(
                        [
                            f"OOF_{row_id:010d}",
                            term_id,
                            item_ids[position],
                            fold,
                            label,
                            candidate_count,
                            is_retrieved,
                            rank,
                            f"{retrieval_score:.8f}",
                        ]
                    )
                    row_id += 1
                    positive_rows += label
                    retrieved_positive_rows += label * is_retrieved
                    rows_by_fold[fold] += 1

    if positive_rows != len(positives):
        raise AssertionError(
            f"Expected {len(positives):,} positive slate rows, wrote {positive_rows:,}"
        )

    summary = {
        "n_splits": args.n_splits,
        "seed": args.seed,
        "base_candidates": args.base_candidates,
        "retrieval_max_features": args.max_features,
        "retrieval_min_df": args.min_df,
        "retrieval_word_ngram_max": args.word_ngram_max,
        "terms": len(train_terms),
        "catalog_items": len(items),
        "positive_pairs": len(positives),
        "slate_rows": row_id,
        "positive_rate": positive_rows / row_id,
        "retrieved_positive_recall": retrieved_positive_rows / positive_rows,
        "rows_by_fold": rows_by_fold.tolist(),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    (output_dir / "slate_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))
    print(f"wrote {folds_path}")
    print(f"wrote {slate_path}")


def sample_rows(frame: pd.DataFrame, maximum: int, seed: int) -> pd.DataFrame:
    if maximum <= 0 or len(frame) <= maximum:
        return frame.reset_index(drop=True)
    positive = frame.loc[frame["label"].eq(1)]
    negative = frame.loc[frame["label"].eq(0)]
    positive_share = len(positive) / len(frame)
    n_positive = min(len(positive), max(1, int(round(maximum * positive_share))))
    n_negative = min(len(negative), maximum - n_positive)
    parts = [
        (
            positive.sample(n=n_positive, random_state=seed)
            if len(positive) > n_positive
            else positive
        ),
        (
            negative.sample(n=n_negative, random_state=seed + 1)
            if len(negative) > n_negative
            else negative
        ),
    ]
    return (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1.0, random_state=seed + 2)
        .reset_index(drop=True)
    )


def fit_category_signal_for_fold(
    data_dir: Path,
    output_dir: Path,
    train_term_ids: set[str],
    slate_term_ids: set[str],
    args: argparse.Namespace,
) -> pd.DataFrame:
    terms = pd.read_csv(
        data_dir / "terms.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["term_id", "query"],
    )
    items = pd.read_csv(
        data_dir / "items.csv",
        dtype=str,
        keep_default_na=False,
        usecols=["item_id", "category"],
    )
    positives = load_positive_pairs(data_dir / "training_pairs.csv")
    positives = positives.loc[positives["term_id"].isin(train_term_ids)].reset_index(
        drop=True
    )
    examples = build_category_examples(
        terms,
        items,
        positives,
        max_examples=args.category_max_examples or None,
        seed=args.seed,
    )
    signal = CategorySignal(
        topk=args.category_topk,
        max_features=args.category_max_features,
        max_iter=args.category_max_iter,
        min_class_count=args.category_min_class_count,
        seed=args.seed,
    ).fit(examples)
    prediction_terms = terms.loc[
        terms["term_id"].isin(slate_term_ids), ["term_id", "query"]
    ].copy()
    topk = signal.predict_terms(prediction_terms, chunk_size=args.category_chunk_size)
    topk_path = output_dir / "term_category_topk.csv"
    topk.to_csv(topk_path, index=False)
    joblib.dump(signal, output_dir / "category_signal.joblib")
    print(f"wrote {topk_path}")
    return topk


def train_catboost(
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    x_inner_valid: pd.DataFrame | None,
    y_inner_valid: np.ndarray | None,
    x_score: pd.DataFrame,
    output_dir: Path,
    args: argparse.Namespace,
) -> tuple[CatBoostClassifier, np.ndarray]:
    params: dict[str, object] = {
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": args.catboost_iterations,
        "learning_rate": args.catboost_learning_rate,
        "depth": args.catboost_depth,
        "l2_leaf_reg": args.catboost_l2_leaf_reg,
        "random_strength": args.catboost_random_strength,
        "bootstrap_type": "Bernoulli",
        "subsample": args.catboost_subsample,
        "auto_class_weights": "Balanced",
        "random_seed": args.seed,
        "thread_count": args.threads,
        "allow_writing_files": False,
        "verbose": args.verbose,
        "task_type": args.task_type,
        "devices": args.devices,
    }
    if args.task_type == "CPU":
        params["rsm"] = args.catboost_rsm

    train_pool = Pool(x_fit, y_fit, cat_features=CAT_FEATURES)
    score_pool = Pool(x_score, cat_features=CAT_FEATURES)
    model = CatBoostClassifier(**params)
    if x_inner_valid is not None and y_inner_valid is not None:
        inner_valid_pool = Pool(x_inner_valid, y_inner_valid, cat_features=CAT_FEATURES)
        model.fit(
            train_pool,
            eval_set=inner_valid_pool,
            use_best_model=True,
            early_stopping_rounds=args.early_stopping_rounds,
        )
    else:
        model.fit(train_pool)
    probability = model.predict_proba(score_pool)[:, 1].astype(np.float32)
    model.save_model(str(output_dir / "catboost.cbm"))
    return model, probability


def train_lgbm(
    x_fit: pd.DataFrame,
    y_fit: np.ndarray,
    x_inner_valid: pd.DataFrame | None,
    y_inner_valid: np.ndarray | None,
    x_score: pd.DataFrame,
    output_dir: Path,
    args: argparse.Namespace,
) -> tuple[lgb.LGBMClassifier, np.ndarray]:
    numeric_features = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES
    numeric_features.extend(
        feature for feature in OPTIONAL_SEMANTIC_FEATURES if feature in x_fit.columns
    )
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=args.lgbm_estimators,
        learning_rate=args.lgbm_learning_rate,
        num_leaves=args.lgbm_num_leaves,
        min_child_samples=args.lgbm_min_child_samples,
        subsample=args.lgbm_subsample,
        colsample_bytree=args.lgbm_colsample_bytree,
        reg_lambda=args.lgbm_reg_lambda,
        class_weight="balanced",
        random_state=args.seed,
        n_jobs=args.threads,
    )
    callbacks: list[object] = [lgb.log_evaluation(period=args.verbose)]
    fit_kwargs: dict[str, object] = {"callbacks": callbacks}
    if x_inner_valid is not None and y_inner_valid is not None:
        if args.early_stopping_rounds > 0:
            callbacks.append(
                lgb.early_stopping(args.early_stopping_rounds, verbose=True)
            )
        fit_kwargs["eval_set"] = [(x_inner_valid[numeric_features], y_inner_valid)]
        fit_kwargs["eval_metric"] = "binary_logloss"
    model.fit(x_fit[numeric_features], y_fit, **fit_kwargs)
    probability = model.predict_proba(x_score[numeric_features])[:, 1].astype(
        np.float32
    )
    joblib.dump(
        {"model": model, "features": numeric_features}, output_dir / "lgbm.joblib"
    )
    return model, probability


def inner_group_split(
    rows: pd.DataFrame,
    valid_size: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    if valid_size <= 0.0:
        return np.arange(len(rows), dtype=np.int64), np.empty(0, dtype=np.int64)
    if not 0.0 < valid_size < 1.0:
        raise ValueError(f"inner-valid-size must be in [0, 1), got {valid_size}")
    splitter = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=seed)
    fit_index, inner_valid_index = next(
        splitter.split(rows, rows["label"], groups=rows["term_id"])
    )
    if set(rows.iloc[fit_index]["term_id"]) & set(
        rows.iloc[inner_valid_index]["term_id"]
    ):
        raise AssertionError("Term leakage across inner early-stopping split")
    return fit_index, inner_valid_index


def run_fold(args: argparse.Namespace, fold: int | None = None) -> Path:
    started = time.time()
    selected_fold = int(args.fold if fold is None else fold)
    data_dir = Path(args.data_dir)
    root_output_dir = Path(args.output_dir)
    output_dir = root_output_dir / f"fold_{selected_fold}"
    output_dir.mkdir(parents=True, exist_ok=True)

    slates = pd.read_csv(
        args.slates or root_output_dir / "validation_slates.csv",
        dtype={"slate_id": str, "term_id": str, "item_id": str},
        keep_default_na=False,
    )
    required = set(SLATE_COLUMNS)
    missing = required - set(slates.columns)
    if missing:
        raise ValueError(f"Slate file is missing columns: {sorted(missing)}")
    for column in [
        "fold",
        "label",
        "candidate_count",
        "is_retrieved",
        "retrieval_rank",
    ]:
        slates[column] = pd.to_numeric(slates[column], errors="raise").astype(np.int32)
    slates["retrieval_score"] = pd.to_numeric(
        slates["retrieval_score"], errors="raise"
    ).astype(np.float32)

    available_folds = sorted(slates["fold"].unique().tolist())
    if selected_fold not in available_folds:
        raise ValueError(
            f"fold={selected_fold} not present; available={available_folds}"
        )
    train_rows = slates.loc[slates["fold"].ne(selected_fold)].copy()
    valid_rows = slates.loc[slates["fold"].eq(selected_fold)].copy()
    train_rows = sample_rows(
        train_rows, args.max_train_rows, args.seed + selected_fold * 10
    )
    valid_rows = sample_rows(
        valid_rows, args.max_valid_rows, args.seed + selected_fold * 10 + 1
    )
    if train_rows.empty or valid_rows.empty:
        raise ValueError("Fold split produced empty training or validation rows")
    print(
        f"fold={selected_fold} train_rows={len(train_rows):,} valid_rows={len(valid_rows):,} "
        f"train_terms={train_rows['term_id'].nunique():,} "
        f"valid_terms={valid_rows['term_id'].nunique():,}"
    )

    train_term_ids = set(train_rows["term_id"].astype(str))
    slate_term_ids = set(slates["term_id"].astype(str))
    topk = fit_category_signal_for_fold(
        data_dir,
        output_dir,
        train_term_ids,
        slate_term_ids,
        args,
    )
    term_category_lookup = build_term_category_lookup(topk)

    terms = load_terms(data_dir / "terms.csv")
    items = load_items(data_dir / "items.csv")
    item_meta_lookup = build_item_meta_lookup(data_dir / "items.csv")
    x_train = rows_to_frame(
        train_rows,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        f"fold {selected_fold} train features",
    )
    x_valid = rows_to_frame(
        valid_rows,
        terms,
        items,
        term_category_lookup,
        item_meta_lookup,
        f"fold {selected_fold} valid features",
    )
    semantic_features: list[str] = []
    if args.semantic_features:
        semantic_features = [
            feature
            for feature in OPTIONAL_SEMANTIC_FEATURES
            if feature in train_rows.columns and feature in valid_rows.columns
        ]
        for feature in semantic_features:
            x_train[feature] = pd.to_numeric(
                train_rows[feature], errors="raise"
            ).to_numpy(dtype=np.float32)
            x_valid[feature] = pd.to_numeric(
                valid_rows[feature], errors="raise"
            ).to_numpy(dtype=np.float32)
        if not semantic_features:
            print("semantic features requested, but the slate contains none")
    y_train = train_rows["label"].to_numpy(dtype=np.int8)
    y_valid = valid_rows["label"].to_numpy(dtype=np.int8)
    fit_index, inner_valid_index = inner_group_split(
        train_rows,
        args.inner_valid_size,
        args.seed + selected_fold * 100,
    )
    x_fit = x_train.iloc[fit_index]
    y_fit = y_train[fit_index]
    if len(inner_valid_index):
        x_inner_valid: pd.DataFrame | None = x_train.iloc[inner_valid_index]
        y_inner_valid: np.ndarray | None = y_train[inner_valid_index]
    else:
        x_inner_valid = None
        y_inner_valid = None
    print(
        f"fold={selected_fold} model_fit_rows={len(fit_index):,} "
        f"inner_early_stop_rows={len(inner_valid_index):,} outer_score_rows={len(valid_rows):,}"
    )

    score_output = valid_rows[SLATE_COLUMNS].copy().reset_index(drop=True)
    score_output["lexical_score"] = x_valid["lexical_score"].to_numpy(dtype=np.float32)
    for feature in semantic_features:
        score_output[feature] = x_valid[feature].to_numpy(dtype=np.float32)
    models = set(args.models)
    if "catboost" in models:
        _, catboost_probability = train_catboost(
            x_fit,
            y_fit,
            x_inner_valid,
            y_inner_valid,
            x_valid,
            output_dir,
            args,
        )
        score_output["catboost_prob"] = catboost_probability
    if "lgbm" in models:
        _, lgbm_probability = train_lgbm(
            x_fit,
            y_fit,
            x_inner_valid,
            y_inner_valid,
            x_valid,
            output_dir,
            args,
        )
        score_output["lgbm_prob"] = lgbm_probability

    score_path = output_dir / "oof_scores.csv"
    score_output.to_csv(score_path, index=False)
    metadata = {
        "fold": selected_fold,
        "models": sorted(models),
        "train_rows": len(train_rows),
        "valid_rows": len(valid_rows),
        "train_terms": len(train_term_ids),
        "valid_terms": valid_rows["term_id"].nunique(),
        "model_fit_rows": len(fit_index),
        "inner_early_stop_rows": len(inner_valid_index),
        "semantic_features": semantic_features,
        "train_positive_rate": float(y_train.mean()),
        "valid_positive_rate": float(y_valid.mean()),
        "elapsed_minutes": (time.time() - started) / 60.0,
    }
    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print(json.dumps(metadata, indent=2))
    print(f"wrote {score_path}")

    del x_train, x_valid, terms, items, item_meta_lookup
    gc.collect()
    return score_path


def combine_fold_scores(
    output_dir: Path, n_splits: int, output: Path | None = None
) -> Path:
    paths = [output_dir / f"fold_{fold}" / "oof_scores.csv" for fold in range(n_splits)]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing fold score files: {missing}")
    frames: list[pd.DataFrame] = []
    for fold, path in enumerate(paths):
        frame = pd.read_csv(
            path, dtype={"slate_id": str, "term_id": str, "item_id": str}
        )
        transformer_path = output_dir / f"fold_{fold}" / "transformer_oof_scores.csv"
        if transformer_path.exists():
            transformer = pd.read_csv(transformer_path, dtype={"slate_id": str})
            required = {"slate_id", "transformer_prob"}
            missing_columns = required - set(transformer.columns)
            if missing_columns:
                raise ValueError(
                    f"{transformer_path} is missing {sorted(missing_columns)}"
                )
            frame = frame.merge(
                transformer[["slate_id", "transformer_prob"]],
                on="slate_id",
                how="left",
                validate="one_to_one",
            )
            if frame["transformer_prob"].isna().any():
                raise ValueError(
                    f"{transformer_path} does not cover every row in fold {fold}"
                )
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    if combined["slate_id"].duplicated().any():
        examples = (
            combined.loc[combined["slate_id"].duplicated(), "slate_id"].head().tolist()
        )
        raise ValueError(f"Duplicate OOF slate ids, examples={examples}")
    combined = combined.sort_values("slate_id").reset_index(drop=True)
    output_path = output or output_dir / "oof_scores.csv"
    combined.to_csv(output_path, index=False)
    print(f"wrote {output_path} rows={len(combined):,}")
    return output_path


def run_all(args: argparse.Namespace) -> None:
    for fold in range(args.n_splits):
        run_fold(args, fold=fold)
    combine_fold_scores(Path(args.output_dir), args.n_splits)


def simplex_weights(n_components: int, step: float) -> Iterable[tuple[float, ...]]:
    if n_components <= 0:
        raise ValueError("At least one score component is required")
    units = int(round(1.0 / step))
    if units <= 0 or not np.isclose(units * step, 1.0, atol=1e-8):
        raise ValueError(
            "weight-step must divide 1.0 exactly, for example 0.25, 0.20, or 0.10"
        )
    for separators in itertools.combinations(
        range(units + n_components - 1), n_components - 1
    ):
        boundaries = (-1,) + separators + (units + n_components - 1,)
        counts = tuple(
            boundaries[i + 1] - boundaries[i] - 1 for i in range(n_components)
        )
        yield tuple(count / units for count in counts)


def macro_f1_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    positive_denominator = 2 * tp + fp + fn
    negative_denominator = 2 * tn + fp + fn
    positive_f1 = 2 * tp / positive_denominator if positive_denominator else 0.0
    negative_f1 = 2 * tn / negative_denominator if negative_denominator else 0.0
    return 0.5 * (positive_f1 + negative_f1)


def evaluate_rates(
    scores: np.ndarray,
    labels: np.ndarray,
    term_ids: np.ndarray,
    rates: list[float],
    constraint_base: int,
) -> list[tuple[float, float, int]]:
    n_rows = len(scores)
    total_positive = int(labels.sum())
    mandatory = np.zeros(n_rows, dtype=bool)
    if constraint_base > 0:
        frame = pd.DataFrame({"term_id": term_ids, "score": scores})
        group_size = frame.groupby("term_id", sort=False)["score"].transform("size")
        minimum = (group_size - constraint_base).clip(lower=0)
        rank = frame.groupby("term_id", sort=False)["score"].rank(
            method="first", ascending=False
        )
        mandatory = rank.le(minimum).to_numpy()

    mandatory_count = int(mandatory.sum())
    mandatory_positive = int(labels[mandatory].sum())
    eligible_positions = np.flatnonzero(~mandatory)
    order = np.argsort(-scores[eligible_positions], kind="stable")
    ordered_labels = labels[eligible_positions[order]].astype(np.int64)
    cumulative_positive = np.cumsum(ordered_labels)

    output: list[tuple[float, float, int]] = []
    for rate in rates:
        predicted_positive = int(round(n_rows * rate))
        if predicted_positive < mandatory_count or predicted_positive > n_rows:
            continue
        additional = predicted_positive - mandatory_count
        additional_tp = int(cumulative_positive[additional - 1]) if additional else 0
        tp = mandatory_positive + additional_tp
        fp = predicted_positive - tp
        fn = total_positive - tp
        tn = n_rows - tp - fp - fn
        output.append((rate, macro_f1_from_counts(tp, fp, fn, tn), mandatory_count))
    return output


def merge_extra_scores(frame: pd.DataFrame, paths: list[str]) -> pd.DataFrame:
    merged = frame
    for raw_path in paths:
        path = Path(raw_path)
        extra = pd.read_csv(path, dtype={"slate_id": str})
        if "slate_id" not in extra.columns:
            raise ValueError(f"{path} must contain slate_id")
        value_columns = [column for column in extra.columns if column != "slate_id"]
        duplicates = set(value_columns) & set(merged.columns)
        if duplicates:
            raise ValueError(
                f"{path} duplicates existing columns: {sorted(duplicates)}"
            )
        merged = merged.merge(extra, on="slate_id", how="left", validate="one_to_one")
        if merged[value_columns].isna().any().any():
            raise ValueError(f"{path} does not cover every OOF slate_id")
    return merged


def optimize(args: argparse.Namespace) -> None:
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(args.oof_scores, dtype={"slate_id": str, "term_id": str})
    frame = merge_extra_scores(frame, args.extra_scores)
    for column in ["label", "fold"]:
        frame[column] = pd.to_numeric(frame[column], errors="raise").astype(np.int16)

    if args.components:
        components = args.components
    else:
        preferred = [
            "catboost_prob",
            "lgbm_prob",
            "transformer_prob",
            "semantic_cosine",
            "semantic_rank_pct",
            "lexical_score",
        ]
        components = [column for column in preferred if column in frame.columns]
    if not components:
        raise ValueError("No score components were found; pass --components explicitly")
    missing = set(components) - set(frame.columns)
    if missing:
        raise ValueError(f"OOF score file is missing components: {sorted(missing)}")

    rank_columns: list[np.ndarray] = []
    for component in components:
        numeric = pd.to_numeric(frame[component], errors="coerce")
        if numeric.isna().any():
            raise ValueError(
                f"Component {component} contains missing/non-numeric values"
            )
        fold_rank = numeric.groupby(frame["fold"], sort=False).rank(
            method="average", pct=True
        )
        rank_columns.append(fold_rank.to_numpy(dtype=np.float32))

    labels = frame["label"].to_numpy(dtype=np.int8)
    term_ids = frame["term_id"].astype(str).to_numpy()
    rates = sorted(set(float(rate) for rate in args.rates))
    constraints = sorted(set(int(base) for base in args.constraint_bases))
    results: list[dict[str, float | int]] = []

    weight_grid = list(simplex_weights(len(components), args.weight_step))
    print(
        f"optimizing rows={len(frame):,} components={components} "
        f"weight_combinations={len(weight_grid)} rates={len(rates)} constraints={constraints}"
    )
    for weights in tqdm(weight_grid, desc="OOF blend grid"):
        blend = np.zeros(len(frame), dtype=np.float32)
        for values, weight in zip(rank_columns, weights):
            if weight:
                blend += values * np.float32(weight)
        for constraint_base in constraints:
            for rate, score, mandatory_count in evaluate_rates(
                blend,
                labels,
                term_ids,
                rates,
                constraint_base,
            ):
                row: dict[str, float | int] = {
                    "macro_f1": score,
                    "rate": rate,
                    "constraint_base": constraint_base,
                    "mandatory_count": mandatory_count,
                }
                for component, weight in zip(components, weights):
                    row[f"weight_{component}"] = weight
                results.append(row)

    result_frame = (
        pd.DataFrame(results)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )
    result_path = output_dir / "oof_optimization.csv"
    result_frame.to_csv(result_path, index=False)
    best = result_frame.iloc[0].to_dict()
    report = {
        "rows": len(frame),
        "positive_rate": float(labels.mean()),
        "components": components,
        "weight_step": args.weight_step,
        "best": best,
    }
    report_path = output_dir / "oof_best.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(result_frame.head(args.show_top).to_string(index=False))
    print(f"wrote {result_path}")
    print(f"wrote {report_path}")


def add_training_arguments(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--output-dir", default="outputs/grouped_oof")
    parser.add_argument("--slates")
    parser.add_argument(
        "--models",
        nargs="+",
        choices=["catboost", "lgbm"],
        default=["catboost", "lgbm"],
    )
    parser.add_argument(
        "--max-train-rows", type=int, default=0, help="Debug only; 0 uses all rows."
    )
    parser.add_argument(
        "--max-valid-rows", type=int, default=0, help="Debug only; 0 uses all rows."
    )
    parser.add_argument("--category-topk", type=int, default=5)
    parser.add_argument("--category-max-examples", type=int, default=0)
    parser.add_argument("--category-max-features", type=int, default=120_000)
    parser.add_argument("--category-max-iter", type=int, default=80)
    parser.add_argument("--category-min-class-count", type=int, default=2)
    parser.add_argument("--category-chunk-size", type=int, default=25_000)
    parser.add_argument("--task-type", choices=["CPU", "GPU"], default="CPU")
    parser.add_argument("--devices", default="0")
    parser.add_argument("--threads", type=int, default=-1)
    parser.add_argument("--verbose", type=int, default=100)
    parser.add_argument("--early-stopping-rounds", type=int, default=100)
    parser.add_argument(
        "--inner-valid-size",
        type=float,
        default=0.10,
        help=(
            "Fraction of outer-training terms reserved for early stopping; "
            "outer fold labels are never used."
        ),
    )
    parser.add_argument("--catboost-iterations", type=int, default=2_000)
    parser.add_argument("--catboost-learning-rate", type=float, default=0.035)
    parser.add_argument("--catboost-depth", type=int, default=7)
    parser.add_argument("--catboost-l2-leaf-reg", type=float, default=8.0)
    parser.add_argument("--catboost-random-strength", type=float, default=0.8)
    parser.add_argument("--catboost-subsample", type=float, default=0.85)
    parser.add_argument("--catboost-rsm", type=float, default=0.95)
    parser.add_argument("--lgbm-estimators", type=int, default=4_000)
    parser.add_argument("--lgbm-learning-rate", type=float, default=0.03)
    parser.add_argument("--lgbm-num-leaves", type=int, default=128)
    parser.add_argument("--lgbm-min-child-samples", type=int, default=80)
    parser.add_argument("--lgbm-subsample", type=float, default=0.9)
    parser.add_argument("--lgbm-colsample-bytree", type=float, default=0.9)
    parser.add_argument("--lgbm-reg-lambda", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--semantic-features",
        action=argparse.BooleanOptionalAction,
        default=True,
        help=(
            "Use semantic_cosine/semantic_rank_pct when those optional columns "
            "exist in the slate."
        ),
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Leakage-safe term-grouped candidate-slate OOF validation."
    )
    subparsers = parser.add_subparsers(dest="command", required=True)

    slate_parser = subparsers.add_parser("build-slates")
    slate_parser.add_argument("--data-dir", default="data")
    slate_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    slate_parser.add_argument("--n-splits", type=int, default=5)
    slate_parser.add_argument("--base-candidates", type=int, default=100)
    slate_parser.add_argument("--chunk-size", type=int, default=128)
    slate_parser.add_argument("--max-features", type=int, default=250_000)
    slate_parser.add_argument("--min-df", type=int, default=2)
    slate_parser.add_argument("--word-ngram-max", type=int, default=2)
    slate_parser.add_argument("--seed", type=int, default=42)
    slate_parser.add_argument("--limit-terms", type=int, default=0, help="Debug only.")
    slate_parser.add_argument("--limit-items", type=int, default=0, help="Debug only.")
    slate_parser.set_defaults(func=build_slates)

    fold_parser = subparsers.add_parser("run-fold")
    add_training_arguments(fold_parser)
    fold_parser.add_argument("--fold", type=int, required=True)
    fold_parser.set_defaults(func=run_fold)

    all_parser = subparsers.add_parser("run-all")
    add_training_arguments(all_parser)
    all_parser.add_argument("--n-splits", type=int, default=5)
    all_parser.set_defaults(func=run_all)

    combine_parser = subparsers.add_parser("combine")
    combine_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    combine_parser.add_argument("--n-splits", type=int, default=5)
    combine_parser.add_argument("--output")
    combine_parser.set_defaults(
        func=lambda args: combine_fold_scores(
            Path(args.output_dir),
            args.n_splits,
            Path(args.output) if args.output else None,
        )
    )

    optimize_parser = subparsers.add_parser("optimize")
    optimize_parser.add_argument(
        "--oof-scores", default="outputs/grouped_oof/oof_scores.csv"
    )
    optimize_parser.add_argument("--output-dir", default="outputs/grouped_oof")
    optimize_parser.add_argument("--extra-scores", nargs="*", default=[])
    optimize_parser.add_argument("--components", nargs="*")
    optimize_parser.add_argument("--weight-step", type=float, default=0.25)
    optimize_parser.add_argument(
        "--rates",
        nargs="+",
        type=float,
        default=[round(value, 3) for value in np.arange(0.15, 0.351, 0.005)],
    )
    optimize_parser.add_argument(
        "--constraint-bases", nargs="+", type=int, default=[0, 100]
    )
    optimize_parser.add_argument("--show-top", type=int, default=20)
    optimize_parser.set_defaults(func=optimize)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/hybrid_embedding_retrieval.py
"""Cache lexical and semantic retrieval, mine negatives, and build validation slates."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from tqdm import tqdm

DEFAULT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CACHE_SCHEMA_VERSION = 1
TEXT_FORMAT_VERSION = "title|brand|category|gender|age_group|attributes:v1"
ITEM_COLUMNS = [
    "item_id",
    "title",
    "category",
    "brand",
    "gender",
    "age_group",
    "attributes",
]
NEGATIVE_COLUMNS = [
    "term_id",
    "item_id",
    "negative_source",
    "source_rank",
    "source_score",
]
BASE_SLATE_COLUMNS = [
    "slate_id",
    "term_id",
    "item_id",
    "fold",
    "label",
    "candidate_count",
    "is_retrieved",
    "retrieval_rank",
    "retrieval_score",
]
SPACE_RE = re.compile(r"\s+")
NON_WORD_RE = re.compile(r"[^a-z0-9çğıöşü]+")


def clean_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return SPACE_RE.sub(" ", str(value).strip())


def normalize_text(value: object) -> str:
    text = clean_text(value).lower().replace("ı", "i")
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character for character in text if not unicodedata.combining(character)
    )
    return SPACE_RE.sub(" ", NON_WORD_RE.sub(" ", text)).strip()


def build_item_text(row: Any) -> str:
    parts = [clean_text(getattr(row, column, "")) for column in ITEM_COLUMNS[1:]]
    return " | ".join(part for part in parts if part)


def item_search_text(items: pd.DataFrame) -> pd.Series:
    columns = [
        column
        for column in ["title", "brand", "category", "attributes"]
        if column in items
    ]
    output = pd.Series("", index=items.index, dtype=object)
    for column in columns:
        values = items[column].map(normalize_text)
        output = output.str.cat(values, sep=" ")
    return output.str.strip()


def stable_seed(seed: int, *parts: object) -> int:
    payload = "\x1f".join([str(seed), *(str(part) for part in parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little", signed=False)


def file_fingerprint(path: Path, sample_bytes: int = 1 << 20) -> dict[str, object]:
    stat = path.stat()
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        digest.update(handle.read(sample_bytes))
        if stat.st_size > sample_bytes:
            handle.seek(max(0, stat.st_size - sample_bytes))
            digest.update(handle.read(sample_bytes))
    return {
        "name": path.name,
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
        "sample_sha256": digest.hexdigest(),
    }


def canonical_fingerprint(payload: dict[str, object]) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def cache_paths(cache_dir: Path) -> dict[str, Path]:
    return {
        "manifest": cache_dir / "manifest.json",
        "item_ids": cache_dir / "item_ids.csv",
        "term_ids": cache_dir / "term_ids.csv",
        "item_embeddings": cache_dir / "item_embeddings.npy",
        "term_embeddings": cache_dir / "term_embeddings.npy",
        "item_tfidf": cache_dir / "item_tfidf.npz",
        "term_tfidf": cache_dir / "term_tfidf.npz",
        "vectorizer": cache_dir / "tfidf_vectorizer.joblib",
        "retrieval": cache_dir / "retrieval_topk.npz",
    }


def load_manifest(cache_dir: Path, require_complete: bool = True) -> dict[str, Any]:
    path = cache_paths(cache_dir)["manifest"]
    if not path.exists():
        raise FileNotFoundError(f"Missing cache manifest: {path}")
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("schema_version") != CACHE_SCHEMA_VERSION:
        raise ValueError(
            f"Unsupported cache schema in {path}: {manifest.get('schema_version')}"
        )
    if require_complete and not manifest.get("complete"):
        raise ValueError(f"Cache is incomplete: {path}")
    return manifest


def cache_is_valid(cache_dir: Path, expected_key: str | None = None) -> bool:
    try:
        manifest = load_manifest(cache_dir)
    except (FileNotFoundError, ValueError, json.JSONDecodeError):
        return False
    if expected_key is not None and manifest.get("cache_key") != expected_key:
        return False
    paths = cache_paths(cache_dir)
    required = [
        paths["item_ids"],
        paths["term_ids"],
        paths["item_embeddings"],
        paths["term_embeddings"],
        paths["item_tfidf"],
        paths["term_tfidf"],
        paths["retrieval"],
    ]
    return all(path.exists() for path in required)


def choose_device(torch: Any, requested: str) -> Any:
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if requested.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(f"Requested {requested}, but CUDA is unavailable")
    return torch.device(requested)


def mean_pool(last_hidden_state: Any, attention_mask: Any, torch: Any) -> Any:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return torch.nn.functional.normalize(pooled, p=2, dim=1)


def encode_pretrained_texts(
    texts: list[str],
    model_name: str,
    max_length: int,
    batch_size: int,
    device_name: str,
) -> np.ndarray:
    try:
        import torch
        from transformers import AutoModel, AutoTokenizer
    except (ImportError, ModuleNotFoundError) as exc:
        raise RuntimeError("build-cache requires torch and transformers") from exc

    device = choose_device(torch, device_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    arrays: list[np.ndarray] = []
    amp_enabled = device.type == "cuda"
    with torch.inference_mode():
        for start in tqdm(
            range(0, len(texts), batch_size), desc="frozen MiniLM encoding"
        ):
            batch = tokenizer(
                texts[start : start + batch_size],
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.autocast(
                device_type=device.type, enabled=amp_enabled, dtype=torch.float16
            ):
                output = model(**batch)
                embedding = mean_pool(
                    output.last_hidden_state, batch["attention_mask"], torch
                )
            arrays.append(embedding.float().cpu().numpy().astype(np.float16))
    if not arrays:
        return np.empty((0, int(model.config.hidden_size)), dtype=np.float16)
    return np.concatenate(arrays, axis=0)


def _deterministic_topk(
    scores: np.ndarray, indices: np.ndarray, k: int
) -> tuple[np.ndarray, np.ndarray]:
    if scores.shape != indices.shape:
        raise ValueError("scores and indices must have identical shapes")
    take = min(k, scores.shape[1])
    out_scores = np.empty((scores.shape[0], take), dtype=np.float32)
    out_indices = np.empty((scores.shape[0], take), dtype=np.int64)
    for row in range(scores.shape[0]):
        if take < scores.shape[1]:
            # Argpartition alone may choose arbitrary members of a tie at the
            # cutoff. Keep all strictly better scores, then the smallest item
            # indices at the boundary so CPU/GPU block sizes cannot alter IDs.
            threshold = np.partition(scores[row], -take)[-take]
            strict = np.flatnonzero(scores[row] > threshold)
            boundary = np.flatnonzero(scores[row] == threshold)
            boundary_order = np.argsort(indices[row, boundary], kind="stable")
            needed = take - len(strict)
            selected = np.concatenate([strict, boundary[boundary_order[:needed]]])
        else:
            selected = np.arange(scores.shape[1])
        order = np.lexsort((indices[row, selected], -scores[row, selected]))
        selected = selected[order]
        out_scores[row] = scores[row, selected]
        out_indices[row] = indices[row, selected]
    return out_indices, out_scores


def exact_cosine_topk(
    query_embeddings: np.ndarray,
    item_embeddings: np.ndarray,
    k: int,
    query_block_size: int = 64,
    item_block_size: int = 16_384,
    device: str = "cpu",
) -> tuple[np.ndarray, np.ndarray]:
    """Deterministic exact cosine top-k over already L2-normalized arrays."""
    if query_embeddings.ndim != 2 or item_embeddings.ndim != 2:
        raise ValueError("Embeddings must be two-dimensional")
    if query_embeddings.shape[1] != item_embeddings.shape[1]:
        raise ValueError("Query/item embedding dimensions differ")
    if not 0 < k <= len(item_embeddings):
        raise ValueError(f"k must be in 1..{len(item_embeddings)}, got {k}")

    use_torch = device != "cpu"
    torch = None
    torch_device = None
    if use_torch:
        try:
            import torch as torch_module
        except ModuleNotFoundError as exc:
            raise RuntimeError("GPU retrieval requires torch") from exc
        torch = torch_module
        torch_device = choose_device(torch, device)

    all_indices = np.empty((len(query_embeddings), k), dtype=np.int32)
    all_scores = np.empty((len(query_embeddings), k), dtype=np.float32)
    for q_start in tqdm(
        range(0, len(query_embeddings), query_block_size), desc="exact cosine top-k"
    ):
        q_stop = min(q_start + query_block_size, len(query_embeddings))
        query = np.asarray(query_embeddings[q_start:q_stop], dtype=np.float32)
        kept_indices = np.empty((len(query), 0), dtype=np.int64)
        kept_scores = np.empty((len(query), 0), dtype=np.float32)
        if use_torch:
            query_tensor = torch.from_numpy(query).to(torch_device)
        for i_start in range(0, len(item_embeddings), item_block_size):
            i_stop = min(i_start + item_block_size, len(item_embeddings))
            item = np.asarray(item_embeddings[i_start:i_stop], dtype=np.float32)
            if use_torch:
                item_tensor = torch.from_numpy(item).to(torch_device)
                block_scores = (query_tensor @ item_tensor.T).float().cpu().numpy()
                del item_tensor
            else:
                block_scores = query @ item.T
            block_indices = np.broadcast_to(
                np.arange(i_start, i_stop, dtype=np.int64)[None, :], block_scores.shape
            )
            merged_scores = np.concatenate([kept_scores, block_scores], axis=1)
            merged_indices = np.concatenate([kept_indices, block_indices], axis=1)
            kept_indices, kept_scores = _deterministic_topk(
                merged_scores, merged_indices, k
            )
        all_indices[q_start:q_stop] = kept_indices.astype(np.int32)
        all_scores[q_start:q_stop] = kept_scores
    return all_indices, all_scores


def sparse_cosine_topk(
    term_matrix: sparse.csr_matrix,
    item_matrix: sparse.csr_matrix,
    k: int,
) -> tuple[np.ndarray, np.ndarray]:
    if not 0 < k <= item_matrix.shape[0]:
        raise ValueError(f"k must be in 1..{item_matrix.shape[0]}, got {k}")
    all_indices = np.empty((term_matrix.shape[0], k), dtype=np.int32)
    all_scores = np.empty((term_matrix.shape[0], k), dtype=np.float32)
    item_matrix_t = item_matrix.T.tocsr()
    for row in tqdm(range(term_matrix.shape[0]), desc="exact TF-IDF top-k"):
        product = (term_matrix.getrow(row) @ item_matrix_t).tocsr()
        dense_indices = product.indices.astype(np.int64)
        dense_scores = product.data.astype(np.float32)
        if len(dense_indices):
            order = np.lexsort((dense_indices, -dense_scores))
            dense_indices = dense_indices[order]
            dense_scores = dense_scores[order]
        take = min(k, len(dense_indices))
        selected_indices = dense_indices[:take].tolist()
        selected_scores = dense_scores[:take].tolist()
        if take < k:
            selected_set = set(selected_indices)
            for position in range(item_matrix.shape[0]):
                if position in selected_set:
                    continue
                selected_indices.append(position)
                selected_scores.append(0.0)
                if len(selected_indices) == k:
                    break
        all_indices[row] = np.asarray(selected_indices, dtype=np.int32)
        all_scores[row] = np.asarray(selected_scores, dtype=np.float32)
    return all_indices, all_scores


def save_retrieval_pool(
    path: Path,
    lexical_indices: np.ndarray,
    lexical_scores: np.ndarray,
    embedding_indices: np.ndarray,
    embedding_scores: np.ndarray,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        path,
        lexical_indices=np.asarray(lexical_indices, dtype=np.int32),
        lexical_scores=np.asarray(lexical_scores, dtype=np.float16),
        embedding_indices=np.asarray(embedding_indices, dtype=np.int32),
        embedding_scores=np.asarray(embedding_scores, dtype=np.float16),
    )


def load_retrieval_pool(cache_dir: Path) -> dict[str, np.ndarray]:
    manifest = load_manifest(cache_dir)
    pool = np.load(cache_paths(cache_dir)["retrieval"], mmap_mode="r")
    required = {
        "lexical_indices",
        "lexical_scores",
        "embedding_indices",
        "embedding_scores",
    }
    missing = required - set(pool.files)
    if missing:
        raise ValueError(f"Retrieval pool is missing arrays: {sorted(missing)}")
    topk = int(manifest["retrieval_topk"])
    if any(
        pool[name].shape[1] < topk for name in ["lexical_indices", "embedding_indices"]
    ):
        raise ValueError("Retrieval pool shape does not match its manifest")
    return {name: pool[name] for name in required}


def write_embedding_cache_fixture(
    cache_dir: Path,
    item_ids: list[str],
    term_ids: list[str],
    item_embeddings: np.ndarray,
    term_embeddings: np.ndarray,
    lexical_indices: np.ndarray,
    lexical_scores: np.ndarray,
    embedding_indices: np.ndarray,
    embedding_scores: np.ndarray,
) -> None:
    """Write a tiny complete cache for deterministic tests and smoke runs."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = cache_paths(cache_dir)
    pd.DataFrame({"item_id": item_ids}).to_csv(paths["item_ids"], index=False)
    pd.DataFrame({"term_id": term_ids}).to_csv(paths["term_ids"], index=False)
    np.save(paths["item_embeddings"], np.asarray(item_embeddings, dtype=np.float16))
    np.save(paths["term_embeddings"], np.asarray(term_embeddings, dtype=np.float16))
    identity_items = sparse.eye(len(item_ids), dtype=np.float32, format="csr")
    identity_terms = sparse.csr_matrix((len(term_ids), len(item_ids)), dtype=np.float32)
    sparse.save_npz(paths["item_tfidf"], identity_items)
    sparse.save_npz(paths["term_tfidf"], identity_terms)
    save_retrieval_pool(
        paths["retrieval"],
        lexical_indices,
        lexical_scores,
        embedding_indices,
        embedding_scores,
    )
    payload: dict[str, object] = {
        "schema_version": CACHE_SCHEMA_VERSION,
        "model_name": "synthetic-fixture",
        "text_format": TEXT_FORMAT_VERSION,
        "query_max_length": 48,
        "item_max_length": 128,
        "item_count": len(item_ids),
        "term_count": len(term_ids),
        "embedding_dim": int(item_embeddings.shape[1]),
        "embedding_dtype": "float16",
        "retrieval_topk": int(lexical_indices.shape[1]),
        "source_data": {},
        "complete": True,
    }
    payload["cache_key"] = canonical_fingerprint(payload)
    paths["manifest"].write_text(json.dumps(payload, indent=2), encoding="utf-8")


def build_cache(args: argparse.Namespace) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    cache_dir = Path(args.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = cache_paths(cache_dir)
    config: dict[str, object] = {
        "schema_version": CACHE_SCHEMA_VERSION,
        "model_name": args.model_name,
        "text_format": TEXT_FORMAT_VERSION,
        "query_max_length": args.query_max_length,
        "item_max_length": args.item_max_length,
        "retrieval_topk": args.retrieval_topk,
        "tfidf_max_features": args.tfidf_max_features,
        "tfidf_min_df": args.tfidf_min_df,
        "source_data": {
            filename: file_fingerprint(data_dir / filename)
            for filename in ["items.csv", "terms.csv"]
        },
        "limits": {"terms": args.limit_terms, "items": args.limit_items},
    }
    cache_key = canonical_fingerprint(config)
    if not args.force and cache_is_valid(cache_dir, cache_key):
        print(f"resume: valid cache already exists at {cache_dir} key={cache_key[:12]}")
        return
    if paths["manifest"].exists() and not args.force:
        previous = json.loads(paths["manifest"].read_text(encoding="utf-8"))
        if previous.get("cache_key") != cache_key:
            raise ValueError(
                "Cache configuration changed "
                f"({previous.get('cache_key', '')[:12]} -> {cache_key[:12]}). "
                "Use a new --cache-dir or pass --force."
            )
        print(f"resume incomplete cache at {cache_dir} key={cache_key[:12]}")

    incomplete = {
        **config,
        "cache_key": cache_key,
        "complete": False,
        "started_at": time.time(),
    }
    paths["manifest"].write_text(json.dumps(incomplete, indent=2), encoding="utf-8")
    terms = (
        pd.read_csv(
            data_dir / "terms.csv",
            usecols=["term_id", "query"],
            dtype=str,
            keep_default_na=False,
            nrows=args.limit_terms or None,
        )
        .sort_values("term_id")
        .reset_index(drop=True)
    )
    items = (
        pd.read_csv(
            data_dir / "items.csv",
            usecols=ITEM_COLUMNS,
            dtype=str,
            keep_default_na=False,
            nrows=args.limit_items or None,
        )
        .sort_values("item_id")
        .reset_index(drop=True)
    )
    if len(items) < args.retrieval_topk:
        raise ValueError("Catalog is smaller than retrieval-topk")
    term_ids = terms["term_id"].astype(str).tolist()
    item_ids = items["item_id"].astype(str).tolist()
    term_texts = terms["query"].map(clean_text).tolist()
    item_texts = [build_item_text(row) for row in items.itertuples(index=False)]

    embedding_stage_complete = all(
        paths[name].exists()
        for name in ["term_ids", "item_ids", "term_embeddings", "item_embeddings"]
    )
    if embedding_stage_complete and not args.force:
        cached_term_ids = pd.read_csv(paths["term_ids"], dtype=str)["term_id"].tolist()
        cached_item_ids = pd.read_csv(paths["item_ids"], dtype=str)["item_id"].tolist()
        if cached_term_ids == term_ids and cached_item_ids == item_ids:
            term_embeddings = np.load(paths["term_embeddings"], mmap_mode="r")
            item_embeddings = np.load(paths["item_embeddings"], mmap_mode="r")
            print("resume completed embedding stage")
        else:
            embedding_stage_complete = False
    if not embedding_stage_complete or args.force:
        term_embeddings = encode_pretrained_texts(
            term_texts,
            args.model_name,
            args.query_max_length,
            args.encode_batch_size,
            args.device,
        )
        item_embeddings = encode_pretrained_texts(
            item_texts,
            args.model_name,
            args.item_max_length,
            args.encode_batch_size,
            args.device,
        )
        pd.DataFrame({"term_id": term_ids}).to_csv(paths["term_ids"], index=False)
        pd.DataFrame({"item_id": item_ids}).to_csv(paths["item_ids"], index=False)
        np.save(paths["term_embeddings"], term_embeddings)
        np.save(paths["item_embeddings"], item_embeddings)

    term_norm = terms["query"].map(normalize_text)
    item_norm = item_search_text(items)
    tfidf_stage_complete = all(
        paths[name].exists() for name in ["vectorizer", "term_tfidf", "item_tfidf"]
    )
    if tfidf_stage_complete and not args.force:
        term_tfidf = sparse.load_npz(paths["term_tfidf"]).tocsr()
        item_tfidf = sparse.load_npz(paths["item_tfidf"]).tocsr()
        if term_tfidf.shape[0] == len(terms) and item_tfidf.shape[0] == len(items):
            print("resume completed TF-IDF stage")
        else:
            tfidf_stage_complete = False
    if not tfidf_stage_complete or args.force:
        vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=args.tfidf_min_df,
            max_features=args.tfidf_max_features,
            dtype=np.float32,
            norm="l2",
            sublinear_tf=True,
        )
        vectorizer.fit(pd.concat([term_norm, item_norm], ignore_index=True))
        term_tfidf = vectorizer.transform(term_norm).tocsr()
        item_tfidf = vectorizer.transform(item_norm).tocsr()
        joblib.dump(vectorizer, paths["vectorizer"])
        sparse.save_npz(paths["term_tfidf"], term_tfidf)
        sparse.save_npz(paths["item_tfidf"], item_tfidf)

    if paths["retrieval"].exists() and not args.force:
        existing_pool = np.load(paths["retrieval"])
        expected_shape = (len(terms), args.retrieval_topk)
        retrieval_stage_complete = all(
            existing_pool[name].shape == expected_shape
            for name in [
                "lexical_indices",
                "lexical_scores",
                "embedding_indices",
                "embedding_scores",
            ]
        )
    else:
        retrieval_stage_complete = False
    if retrieval_stage_complete:
        print("resume completed exact retrieval stage")
    else:
        embedding_indices, embedding_scores = exact_cosine_topk(
            term_embeddings,
            item_embeddings,
            args.retrieval_topk,
            args.query_block_size,
            args.item_block_size,
            args.device,
        )
        lexical_indices, lexical_scores = sparse_cosine_topk(
            term_tfidf, item_tfidf, args.retrieval_topk
        )
        save_retrieval_pool(
            paths["retrieval"],
            lexical_indices,
            lexical_scores,
            embedding_indices,
            embedding_scores,
        )
    manifest = {
        **config,
        "cache_key": cache_key,
        "complete": True,
        "item_count": len(items),
        "term_count": len(terms),
        "embedding_dim": int(item_embeddings.shape[1]),
        "embedding_dtype": "float16",
        "elapsed_seconds": time.time() - started,
    }
    paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"wrote frozen retrieval cache {cache_dir} key={cache_key[:12]}")


def load_positive_pairs(data_dir: Path) -> pd.DataFrame:
    frame = pd.read_csv(
        data_dir / "training_pairs.csv",
        usecols=lambda column: column in {"term_id", "item_id", "label"},
        dtype=str,
        keep_default_na=False,
    )
    if "label" in frame:
        frame = frame.loc[frame["label"].ne("0")]
    output = frame[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if output.empty:
        raise ValueError("No positive training pairs found")
    return output


class ConservativeNegativeFilter:
    """Reject known positives and near-duplicate items during negative mining."""

    def __init__(self, items: pd.DataFrame, positives: pd.DataFrame):
        self.item_ids = items["item_id"].astype(str).to_numpy()
        self.position_by_id = {
            item_id: position for position, item_id in enumerate(self.item_ids)
        }
        self.keys = np.asarray(
            [
                f"{normalize_text(row.title)}\t{normalize_text(row.brand)}"
                for row in items.itertuples(index=False)
            ],
            dtype=object,
        )
        self.tokens = [
            frozenset(normalize_text(value).split()) for value in items["title"]
        ]
        self.positive_ids: dict[str, set[str]] = {}
        self.positive_keys: dict[str, set[str]] = {}
        self.positive_tokens: dict[str, list[frozenset[str]]] = {}
        for term_id, group in positives.groupby("term_id", sort=False):
            ids = set(group["item_id"].astype(str))
            positions = [
                self.position_by_id[item_id]
                for item_id in ids
                if item_id in self.position_by_id
            ]
            term = str(term_id)
            self.positive_ids[term] = ids
            self.positive_keys[term] = {
                str(self.keys[position]) for position in positions
            }
            self.positive_tokens[term] = [
                self.tokens[position] for position in positions
            ]

    @staticmethod
    def jaccard(left: frozenset[str], right: frozenset[str]) -> float:
        if not left and not right:
            return 1.0
        union = left | right
        return len(left & right) / len(union) if union else 0.0

    def valid(self, term_id: str, position: int) -> bool:
        item_id = str(self.item_ids[position])
        if item_id in self.positive_ids.get(term_id, set()):
            return False
        if str(self.keys[position]) in self.positive_keys.get(term_id, set()):
            return False
        tokens = self.tokens[position]
        return all(
            self.jaccard(tokens, positive) < 0.90
            for positive in self.positive_tokens.get(term_id, [])
        )


def _rank_band_candidates(
    term_id: str,
    ranked_positions: np.ndarray,
    ranked_scores: np.ndarray,
    low_rank: int,
    high_rank: int,
    negative_filter: ConservativeNegativeFilter,
    selected: set[int],
) -> list[tuple[int, int, float]]:
    output: list[tuple[int, int, float]] = []
    stop = min(high_rank, len(ranked_positions))
    for zero_index in range(low_rank - 1, stop):
        position = int(ranked_positions[zero_index])
        if position in selected or not negative_filter.valid(term_id, position):
            continue
        output.append((position, zero_index + 1, float(ranked_scores[zero_index])))
    return output


def _sample_rank_band(
    candidates: list[tuple[int, int, float]],
    count: int,
    rng: np.random.Generator,
) -> list[tuple[int, int, float]]:
    if len(candidates) <= count:
        return candidates
    chosen = np.sort(rng.choice(len(candidates), size=count, replace=False))
    return [candidates[int(index)] for index in chosen]


def _sample_uniform_positions(
    term_id: str,
    count: int,
    negative_filter: ConservativeNegativeFilter,
    selected: set[int],
    rng: np.random.Generator,
) -> list[int]:
    output: list[int] = []
    attempts = 0
    maximum = max(1_000, count * 500)
    while len(output) < count and attempts < maximum:
        position = int(rng.integers(0, len(negative_filter.item_ids)))
        attempts += 1
        if position in selected or not negative_filter.valid(term_id, position):
            continue
        selected.add(position)
        output.append(position)
    if len(output) < count:
        offset = stable_seed(0, term_id, "fallback") % len(negative_filter.item_ids)
        for step in range(len(negative_filter.item_ids)):
            position = int((offset + step) % len(negative_filter.item_ids))
            if position in selected or not negative_filter.valid(term_id, position):
                continue
            selected.add(position)
            output.append(position)
            if len(output) == count:
                break
    if len(output) != count:
        raise ValueError(
            f"Could not sample {count} valid unique negatives for term {term_id}"
        )
    return output


def mine_negative_rows(
    term_ids: Iterable[str],
    term_position: dict[str, int],
    pool: dict[str, np.ndarray],
    negative_filter: ConservativeNegativeFilter,
    seed: int,
    recipe: str,
) -> list[dict[str, object]]:
    if recipe == "contrastive":
        source_specs = [
            ("tfidf_semi_hard", "lexical", 101, 500, 10),
            ("embedding_semi_hard", "embedding", 101, 500, 10),
        ]
        uniform_count = 0
    elif recipe == "reranker":
        source_specs = [
            ("tfidf_near_hard", "lexical", 21, 200, 15),
            ("embedding_near_hard", "embedding", 21, 200, 15),
        ]
        uniform_count = 20
    else:
        raise ValueError(f"Unknown negative recipe: {recipe}")

    rows: list[dict[str, object]] = []
    for term_id in tqdm(
        sorted(set(str(value) for value in term_ids)), desc=f"mine {recipe} negatives"
    ):
        if term_id not in term_position:
            raise KeyError(f"Term {term_id} is absent from the retrieval cache")
        row_position = term_position[term_id]
        selected: set[int] = set()
        term_rng = np.random.default_rng(stable_seed(seed, recipe, term_id))
        if uniform_count:
            for position in _sample_uniform_positions(
                term_id, uniform_count, negative_filter, selected, term_rng
            ):
                rows.append(
                    {
                        "term_id": term_id,
                        "item_id": negative_filter.item_ids[position],
                        "negative_source": "uniform_random",
                        "source_rank": 0,
                        "source_score": 0.0,
                    }
                )

        for source_name, array_prefix, low_rank, high_rank, count in source_specs:
            ranked_positions = pool[f"{array_prefix}_indices"][row_position]
            ranked_scores = pool[f"{array_prefix}_scores"][row_position]
            eligible = _rank_band_candidates(
                term_id,
                ranked_positions,
                ranked_scores,
                low_rank,
                high_rank,
                negative_filter,
                selected,
            )
            chosen = _sample_rank_band(eligible, count, term_rng)
            for position, rank, score in chosen:
                selected.add(position)
                rows.append(
                    {
                        "term_id": term_id,
                        "item_id": negative_filter.item_ids[position],
                        "negative_source": source_name,
                        "source_rank": rank,
                        "source_score": score,
                    }
                )
            shortage = count - len(chosen)
            if shortage:
                for position in _sample_uniform_positions(
                    term_id, shortage, negative_filter, selected, term_rng
                ):
                    rows.append(
                        {
                            "term_id": term_id,
                            "item_id": negative_filter.item_ids[position],
                            "negative_source": f"uniform_backfill_{array_prefix}",
                            "source_rank": 0,
                            "source_score": 0.0,
                        }
                    )
    return rows


def write_negative_output(
    rows: list[dict[str, object]],
    output: Path,
    positives: pd.DataFrame,
    recipe: str,
    seed: int,
) -> None:
    frame = pd.DataFrame(rows, columns=NEGATIVE_COLUMNS)
    if frame.duplicated(["term_id", "item_id"]).any():
        raise AssertionError("Duplicate mined negative pairs")
    positive_keys = set(
        positives["term_id"].astype(str) + "\t" + positives["item_id"].astype(str)
    )
    mined_keys = frame["term_id"].astype(str) + "\t" + frame["item_id"].astype(str)
    if mined_keys.isin(positive_keys).any():
        raise AssertionError("A mined negative overlaps a known positive")
    ranked = frame.loc[frame["source_rank"].astype(int).gt(0)]
    if (
        recipe == "contrastive"
        and not ranked["source_rank"].astype(int).between(101, 500).all()
    ):
        raise AssertionError("Contrastive negative rank outside 101..500")
    if (
        recipe == "reranker"
        and not ranked["source_rank"].astype(int).between(21, 200).all()
    ):
        raise AssertionError("Reranker negative rank outside 21..200")
    output.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output, index=False)
    source_counts = Counter(frame["negative_source"].astype(str))
    audit = {
        "recipe": recipe,
        "seed": seed,
        "rows": len(frame),
        "terms": int(frame["term_id"].nunique()),
        "duplicates": int(frame.duplicated(["term_id", "item_id"]).sum()),
        "positive_overlaps": int(mined_keys.isin(positive_keys).sum()),
        "source_counts": dict(sorted(source_counts.items())),
        "source_ratios": {
            key: value / max(1, len(frame))
            for key, value in sorted(source_counts.items())
        },
    }
    output.with_suffix(".audit.json").write_text(
        json.dumps(audit, indent=2), encoding="utf-8"
    )
    print(f"wrote {output} rows={len(frame):,}")


def load_cache_ids(cache_dir: Path) -> tuple[list[str], list[str]]:
    paths = cache_paths(cache_dir)
    item_ids = (
        pd.read_csv(paths["item_ids"], dtype=str, keep_default_na=False)["item_id"]
        .astype(str)
        .tolist()
    )
    term_ids = (
        pd.read_csv(paths["term_ids"], dtype=str, keep_default_na=False)["term_id"]
        .astype(str)
        .tolist()
    )
    return item_ids, term_ids


def run_miner(args: argparse.Namespace, recipe: str) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    cache_dir = Path(args.cache_dir)
    manifest = load_manifest(cache_dir)
    required_rank = 500 if recipe == "contrastive" else 200
    if int(manifest["retrieval_topk"]) < required_rank:
        raise ValueError(f"{recipe} mining requires retrieval_topk >= {required_rank}")
    item_ids, term_ids = load_cache_ids(cache_dir)
    term_position = {term_id: position for position, term_id in enumerate(term_ids)}
    positives = load_positive_pairs(data_dir)
    positives = positives.loc[positives["term_id"].isin(term_position)].reset_index(
        drop=True
    )
    if args.limit_terms:
        keep_terms = sorted(positives["term_id"].unique())[: args.limit_terms]
        positives = positives.loc[positives["term_id"].isin(keep_terms)].reset_index(
            drop=True
        )
    items = pd.read_csv(
        data_dir / "items.csv",
        usecols=["item_id", "title", "brand"],
        dtype=str,
        keep_default_na=False,
    )
    items = items.set_index("item_id").reindex(item_ids).reset_index()
    if items[["title", "brand"]].isna().any().any():
        raise KeyError("Some cached catalog IDs are missing from items.csv")
    negative_filter = ConservativeNegativeFilter(items, positives)
    pool = load_retrieval_pool(cache_dir)
    rows = mine_negative_rows(
        positives["term_id"].unique(),
        term_position,
        pool,
        negative_filter,
        args.seed,
        recipe,
    )
    output_path = Path(args.output)
    write_negative_output(rows, output_path, positives, recipe, args.seed)
    audit_path = output_path.with_suffix(".audit.json")
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    audit["cache_fingerprint"] = manifest["cache_key"]
    audit["elapsed_seconds"] = time.time() - started
    audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")


def make_term_split_manifest(
    positives: pd.DataFrame, mode: str, seed: int
) -> pd.DataFrame:
    counts = (
        positives.groupby("term_id", sort=True)
        .size()
        .rename("n_positives")
        .reset_index()
    )
    counts["fold"] = np.int16(-1)
    if mode == "pilot":
        if len(counts) < 2:
            raise ValueError("Pilot split requires at least two terms")
        n_bins = max(1, min(10, len(counts) // 5))
        bins = pd.qcut(
            counts["n_positives"].rank(method="first"),
            q=n_bins,
            labels=False,
            duplicates="drop",
        ).astype(int)
        n_holdout = min(len(counts) - 1, max(1, int(round(0.20 * len(counts)))))
        try:
            splitter = StratifiedShuffleSplit(
                n_splits=1, test_size=n_holdout, random_state=seed
            )
            train_index, holdout_index = next(
                splitter.split(np.zeros(len(counts)), bins)
            )
        except ValueError:
            rng = np.random.default_rng(seed)
            order = rng.permutation(len(counts))
            holdout_index, train_index = order[:n_holdout], order[n_holdout:]
        counts.loc[train_index, "fold"] = np.int16(1)
        counts.loc[holdout_index, "fold"] = np.int16(0)
        counts["split"] = np.where(counts["fold"].eq(0), "outer_holdout", "outer_train")
    elif mode == "confirm":
        if len(counts) < 5:
            raise ValueError("Confirm split requires at least five terms")
        n_bins = max(1, min(10, len(counts) // 5))
        bins = pd.qcut(
            counts["n_positives"].rank(method="first"),
            q=n_bins,
            labels=False,
            duplicates="drop",
        ).astype(int)
        splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold, (_, valid_index) in enumerate(
            splitter.split(np.zeros(len(counts)), bins)
        ):
            counts.loc[valid_index, "fold"] = np.int16(fold)
        counts["split"] = "oof"
    elif mode == "final":
        counts["fold"] = np.int16(-1)
        counts["split"] = "all_train"
    else:
        raise ValueError(f"Unknown mode: {mode}")
    if counts["fold"].lt(-1).any():
        raise AssertionError("Invalid term fold assignment")
    return counts[["term_id", "fold", "split", "n_positives"]]


def build_slates(args: argparse.Namespace) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    cache_dir = Path(args.cache_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(cache_dir)
    if int(manifest["retrieval_topk"]) < 100:
        raise ValueError("Hybrid slates require retrieval_topk >= 100 for Recall@100")
    item_ids, term_ids = load_cache_ids(cache_dir)
    item_position = {item_id: position for position, item_id in enumerate(item_ids)}
    term_position = {term_id: position for position, term_id in enumerate(term_ids)}
    positives = load_positive_pairs(data_dir)
    positives = positives.loc[
        positives["term_id"].isin(term_position)
        & positives["item_id"].isin(item_position)
    ].reset_index(drop=True)
    if args.limit_terms:
        kept = sorted(positives["term_id"].unique())[: args.limit_terms]
        positives = positives.loc[positives["term_id"].isin(kept)].reset_index(
            drop=True
        )
    split = make_term_split_manifest(positives, args.mode, args.seed)
    split_path = output_dir / "split_manifest.csv"
    split.to_csv(split_path, index=False)
    fold_by_term = split.set_index("term_id")["fold"].astype(int).to_dict()
    split_by_term = split.set_index("term_id")["split"].astype(str).to_dict()

    positive_positions = {
        str(term_id): [
            item_position[item_id] for item_id in group["item_id"].astype(str)
        ]
        for term_id, group in positives.groupby("term_id", sort=False)
    }
    pool = load_retrieval_pool(cache_dir)
    term_embeddings = np.load(cache_paths(cache_dir)["term_embeddings"], mmap_mode="r")
    item_embeddings = np.load(cache_paths(cache_dir)["item_embeddings"], mmap_mode="r")
    rows: list[dict[str, object]] = []
    retrieval_counts = Counter()
    total_positives = len(positives)

    for term_id in tqdm(sorted(positive_positions), desc="build hybrid slates"):
        tpos = term_position[term_id]
        lexical = [int(value) for value in pool["lexical_indices"][tpos, :40]]
        embedding = [int(value) for value in pool["embedding_indices"][tpos, :40]]
        lexical_meta = {
            position: (rank, float(pool["lexical_scores"][tpos, rank - 1]))
            for rank, position in enumerate(lexical, 1)
        }
        embedding_meta = {
            position: (rank, float(pool["embedding_scores"][tpos, rank - 1]))
            for rank, position in enumerate(embedding, 1)
        }
        positives_for_term = set(positive_positions[term_id])
        selected = set(lexical) | set(embedding)
        rng = np.random.default_rng(stable_seed(args.seed, args.mode, "slate", term_id))
        uniform: list[int] = []
        while len(uniform) < 20:
            position = int(rng.integers(0, len(item_ids)))
            if position in selected or position in positives_for_term:
                continue
            selected.add(position)
            uniform.append(position)
        base_positions = list(dict.fromkeys([*lexical, *embedding, *uniform]))
        appended = sorted(positives_for_term - set(base_positions))
        candidates = base_positions + appended
        cosine = np.asarray(item_embeddings[candidates], dtype=np.float32) @ np.asarray(
            term_embeddings[tpos], dtype=np.float32
        )
        semantic_pct = (
            pd.Series(cosine)
            .rank(method="average", pct=True)
            .to_numpy(dtype=np.float32)
        )
        lexical_100 = set(int(value) for value in pool["lexical_indices"][tpos, :100])
        embedding_100 = set(
            int(value) for value in pool["embedding_indices"][tpos, :100]
        )
        retrieval_counts["tfidf_hits"] += len(positives_for_term & lexical_100)
        retrieval_counts["embedding_hits"] += len(positives_for_term & embedding_100)
        retrieval_counts["hybrid_hits"] += len(
            positives_for_term & (lexical_100 | embedding_100)
        )

        for local_index, position in enumerate(candidates):
            sources: list[str] = []
            if position in lexical_meta:
                sources.append("tfidf")
            if position in embedding_meta:
                sources.append("embedding")
            if position in uniform:
                sources.append("uniform")
            if position in appended:
                sources.append("positive_union")
            lexical_rank, lexical_score = lexical_meta.get(position, (0, 0.0))
            embedding_rank, _ = embedding_meta.get(position, (0, 0.0))
            rows.append(
                {
                    "slate_id": f"HYBRID_{len(rows):010d}",
                    "term_id": term_id,
                    "item_id": item_ids[position],
                    "fold": fold_by_term[term_id],
                    "label": int(position in positives_for_term),
                    "candidate_count": len(candidates),
                    "is_retrieved": int(bool(lexical_rank or embedding_rank)),
                    "retrieval_rank": lexical_rank,
                    "retrieval_score": lexical_score,
                    "semantic_cosine": float(cosine[local_index]),
                    "semantic_rank_pct": float(semantic_pct[local_index]),
                    "candidate_sources": "+".join(sources),
                    "tfidf_rank": lexical_rank,
                    "embedding_rank": embedding_rank,
                    "split": split_by_term[term_id],
                }
            )
    slate = pd.DataFrame(rows)
    if int(slate["label"].sum()) != total_positives:
        raise AssertionError(
            "Every known positive must occur exactly once in its validation slate"
        )
    slate_path = output_dir / "hybrid_validation_slates.csv"
    slate.to_csv(slate_path, index=False)
    metrics = {
        "mode": args.mode,
        "seed": args.seed,
        "rows": len(slate),
        "terms": int(slate["term_id"].nunique()),
        "positives": total_positives,
        "tfidf_recall_at_100": retrieval_counts["tfidf_hits"] / max(1, total_positives),
        "frozen_embedding_recall_at_100": retrieval_counts["embedding_hits"]
        / max(1, total_positives),
        "hybrid_recall_at_100": retrieval_counts["hybrid_hits"]
        / max(1, total_positives),
        "cache_fingerprint": manifest["cache_key"],
        "elapsed_seconds": time.time() - started,
    }
    (output_dir / "retrieval_metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    print(json.dumps(metrics, indent=2))
    print(f"wrote {slate_path}")


def add_common_mining_arguments(
    parser: argparse.ArgumentParser, default_output: str
) -> None:
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--cache-dir", default="outputs/hybrid_embedding/frozen_cache")
    parser.add_argument("--output", default=default_output)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--limit-terms", type=int, default=0)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Cached exact hybrid retrieval and conservative negative mining."
    )
    sub = parser.add_subparsers(dest="operation", required=True)

    cache_cmd = sub.add_parser("build-cache")
    cache_cmd.add_argument("--data-dir", default="data")
    cache_cmd.add_argument(
        "--cache-dir", default="outputs/hybrid_embedding/frozen_cache"
    )
    cache_cmd.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    cache_cmd.add_argument("--query-max-length", type=int, default=48)
    cache_cmd.add_argument("--item-max-length", type=int, default=128)
    cache_cmd.add_argument("--encode-batch-size", type=int, default=512)
    cache_cmd.add_argument("--retrieval-topk", type=int, default=500)
    cache_cmd.add_argument("--query-block-size", type=int, default=64)
    cache_cmd.add_argument("--item-block-size", type=int, default=16_384)
    cache_cmd.add_argument("--tfidf-max-features", type=int, default=250_000)
    cache_cmd.add_argument("--tfidf-min-df", type=int, default=2)
    cache_cmd.add_argument("--device", default="auto")
    cache_cmd.add_argument("--limit-terms", type=int, default=0)
    cache_cmd.add_argument("--limit-items", type=int, default=0)
    cache_cmd.add_argument("--force", action="store_true")
    cache_cmd.set_defaults(func=build_cache)

    contrastive_cmd = sub.add_parser("mine-contrastive")
    add_common_mining_arguments(
        contrastive_cmd, "outputs/hybrid_embedding/contrastive_negative_pool.csv"
    )
    contrastive_cmd.set_defaults(func=lambda args: run_miner(args, "contrastive"))

    reranker_cmd = sub.add_parser("mine-reranker")
    add_common_mining_arguments(
        reranker_cmd, "outputs/hybrid_embedding/reranker_negatives.csv"
    )
    reranker_cmd.set_defaults(func=lambda args: run_miner(args, "reranker"))

    slate_cmd = sub.add_parser("build-slates")
    slate_cmd.add_argument("--data-dir", default="data")
    slate_cmd.add_argument(
        "--cache-dir", default="outputs/hybrid_embedding/frozen_cache"
    )
    slate_cmd.add_argument("--output-dir", default="outputs/hybrid_embedding/pilot")
    slate_cmd.add_argument(
        "--mode", choices=["pilot", "confirm", "final"], default="pilot"
    )
    slate_cmd.add_argument("--seed", type=int, default=42)
    slate_cmd.add_argument("--limit-terms", type=int, default=0)
    slate_cmd.set_defaults(func=build_slates)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/contrastive_biencoder.py
"""Train and score a contrastive shared-encoder two-tower retrieval model."""

from __future__ import annotations

import argparse
import csv
import gc
import hashlib
import json
import math
import random
import re
import shutil
import time
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm import tqdm

DEFAULT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
ITEM_COLUMNS = [
    "item_id",
    "title",
    "category",
    "brand",
    "gender",
    "age_group",
    "attributes",
]
SPACE_RE = re.compile(r"\s+")


def clean_text(value: object) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return SPACE_RE.sub(" ", str(value).strip())


def build_item_text(row: Any) -> str:
    parts = [clean_text(getattr(row, column, "")) for column in ITEM_COLUMNS[1:]]
    return " | ".join(part for part in parts if part)


def stable_seed(seed: int, *parts: object) -> int:
    payload = "\x1f".join([str(seed), *(str(part) for part in parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little", signed=False)


def file_fingerprint(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(1 << 20)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def require_transformer_stack() -> tuple[Any, ...]:
    try:
        import torch
        from torch import nn
        from torch.nn import functional as F
        from torch.utils.data import DataLoader, Dataset
        from transformers import (
            AutoModel,
            AutoTokenizer,
            get_linear_schedule_with_warmup,
        )
    except (ImportError, ModuleNotFoundError) as exc:
        raise RuntimeError(
            "Contrastive training and encoding require torch and transformers"
        ) from exc
    return (
        torch,
        nn,
        F,
        DataLoader,
        Dataset,
        AutoModel,
        AutoTokenizer,
        get_linear_schedule_with_warmup,
    )


def seed_everything(seed: int, torch: Any | None = None) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if hasattr(torch, "use_deterministic_algorithms"):
            torch.use_deterministic_algorithms(True, warn_only=True)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False


def choose_device(torch: Any, requested: str) -> Any:
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if requested.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(f"Requested {requested}, but CUDA is unavailable")
    return torch.device(requested)


def load_positive_pairs(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    if "label" in frame:
        frame = frame.loc[frame["label"].ne("0")]
    frame = frame[["term_id", "item_id"]].drop_duplicates().reset_index(drop=True)
    if frame.empty:
        raise ValueError(f"No positives found in {path}")
    return frame


def grouped_inner_split(
    positives: pd.DataFrame, valid_size: float, seed: int
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not 0.0 < valid_size < 1.0:
        raise ValueError(f"valid_size must be in (0, 1), got {valid_size}")
    terms = positives["term_id"].drop_duplicates().astype(str).to_numpy()
    if len(terms) < 2:
        raise ValueError(
            "At least two terms are required for grouped checkpoint selection"
        )
    rng = np.random.default_rng(seed)
    terms = terms.copy()
    rng.shuffle(terms)
    n_valid = min(len(terms) - 1, max(1, int(round(len(terms) * valid_size))))
    valid_terms = set(terms[:n_valid])
    inner_train = positives.loc[~positives["term_id"].isin(valid_terms)].reset_index(
        drop=True
    )
    inner_valid = positives.loc[positives["term_id"].isin(valid_terms)].reset_index(
        drop=True
    )
    if set(inner_train["term_id"]) & set(inner_valid["term_id"]):
        raise AssertionError("Term leakage in inner validation split")
    split = (
        pd.DataFrame(
            {
                "term_id": terms,
                "split": [
                    "inner_valid" if term in valid_terms else "inner_train"
                    for term in terms
                ],
            }
        )
        .sort_values("term_id")
        .reset_index(drop=True)
    )
    return inner_train, inner_valid, split


def apply_outer_term_split(
    positives: pd.DataFrame,
    split_manifest: Path | None,
    outer_fold: int,
) -> tuple[pd.DataFrame, set[str]]:
    if split_manifest is None or outer_fold < 0:
        return positives.reset_index(drop=True), set()
    split = pd.read_csv(split_manifest, dtype={"term_id": str}, keep_default_na=False)
    if not {"term_id", "fold"}.issubset(split.columns):
        raise ValueError(f"{split_manifest} must contain term_id and fold")
    split["fold"] = pd.to_numeric(split["fold"], errors="raise").astype(int)
    outer_terms = set(split.loc[split["fold"].eq(outer_fold), "term_id"].astype(str))
    if not outer_terms:
        raise ValueError(f"No terms assigned to outer fold {outer_fold}")
    outer_train = positives.loc[~positives["term_id"].isin(outer_terms)].reset_index(
        drop=True
    )
    if set(outer_train["term_id"]) & outer_terms:
        raise AssertionError("Outer term leakage")
    if outer_train.empty:
        raise ValueError("Outer split removed every positive pair")
    return outer_train, outer_terms


def known_positive_mask_numpy(
    batch_term_ids: list[str],
    candidate_item_ids: list[str],
    positives_by_term: dict[str, set[str]],
    target_columns: np.ndarray,
) -> np.ndarray:
    """True entries must be removed from logits; designated targets are always retained."""
    if len(batch_term_ids) != len(target_columns):
        raise ValueError("target_columns must contain one target for every query")
    mask = np.zeros((len(batch_term_ids), len(candidate_item_ids)), dtype=bool)
    for row, term_id in enumerate(batch_term_ids):
        known = positives_by_term.get(str(term_id), set())
        if known:
            mask[row] = np.fromiter(
                (str(item_id) in known for item_id in candidate_item_ids),
                dtype=bool,
                count=len(candidate_item_ids),
            )
        target = int(target_columns[row])
        if target < 0 or target >= len(candidate_item_ids):
            raise IndexError(f"Invalid target column {target}")
        mask[row, target] = False
    return mask


def masked_infonce_loss(
    query_embeddings: Any,
    candidate_embeddings: Any,
    mask: Any,
    target_columns: Any,
    temperature: float,
    F: Any,
) -> Any:
    if temperature <= 0.0:
        raise ValueError("temperature must be positive")
    logits = query_embeddings @ candidate_embeddings.T / temperature
    if logits.shape != mask.shape:
        raise ValueError(
            f"mask shape {mask.shape} does not match logits {logits.shape}"
        )
    if bool(mask.gather(1, target_columns.unsqueeze(1)).any().item()):
        raise AssertionError("A designated InfoNCE target was masked")
    logits = logits.masked_fill(mask, torch_finfo_min(logits))
    return F.cross_entropy(logits, target_columns)


def torch_finfo_min(tensor: Any) -> float:
    # -inf can create NaN gradients in some fp16 kernels; the finite minimum is safe here.
    import torch

    return torch.finfo(tensor.dtype).min


def source_family(value: str) -> str:
    value = value.lower()
    if "tfidf" in value or value.endswith("lexical"):
        return "lexical"
    if "embedding" in value:
        return "embedding"
    return "uniform"


def load_negative_pool(path: Path) -> dict[str, dict[str, list[str]]]:
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = {"term_id", "item_id", "negative_source"} - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    if frame.duplicated(["term_id", "item_id"]).any():
        raise ValueError(f"{path} contains duplicate pairs")
    pool: dict[str, dict[str, list[str]]] = {}
    for row in frame.itertuples(index=False):
        family = source_family(str(row.negative_source))
        if family == "uniform":
            # Conservative miner backfills retain the requested family in their suffix.
            if "lexical" in str(row.negative_source):
                family = "lexical"
            elif "embedding" in str(row.negative_source):
                family = "embedding"
        pool.setdefault(str(row.term_id), {}).setdefault(family, []).append(
            str(row.item_id)
        )
    return pool


def assemble_candidate_item_ids(
    batch_rows: pd.DataFrame,
    negative_pool: dict[str, dict[str, list[str]]],
    catalog_item_ids: np.ndarray,
    seed: int,
    epoch: int,
    uniform_multiplier: int = 3,
    use_semi_hard: bool = True,
) -> tuple[list[str], np.ndarray, list[str]]:
    """Build B targets + B alternating semi-hard + 3B shared uniform candidates."""
    batch_size = len(batch_rows)
    if batch_size == 0:
        raise ValueError("Cannot assemble an empty contrastive batch")
    if uniform_multiplier < 0:
        raise ValueError("uniform_multiplier must be non-negative")
    positives = batch_rows["item_id"].astype(str).tolist()
    candidates = list(positives)
    sources = ["positive"] * batch_size
    selected = set(candidates)
    row_indices = (
        batch_rows["row_index"].astype(int).to_numpy()
        if "row_index" in batch_rows
        else np.arange(batch_size)
    )
    rng = np.random.default_rng(
        stable_seed(seed, epoch, *sorted(int(value) for value in row_indices))
    )

    if use_semi_hard:
        for local_row, row in enumerate(batch_rows.itertuples(index=False)):
            term_id = str(row.term_id)
            row_index = int(getattr(row, "row_index", local_row))
            preferred = "lexical" if (row_index + epoch) % 2 == 0 else "embedding"
            alternatives = [
                preferred,
                "embedding" if preferred == "lexical" else "lexical",
                "uniform",
            ]
            chosen: str | None = None
            chosen_source = preferred
            for family in alternatives:
                values = negative_pool.get(term_id, {}).get(family, [])
                if not values:
                    continue
                offset = stable_seed(seed, epoch, row_index, family) % len(values)
                for step in range(len(values)):
                    item_id = str(values[(offset + step) % len(values)])
                    if item_id not in selected:
                        chosen = item_id
                        chosen_source = family
                        break
                if chosen is not None:
                    break
            if chosen is None:
                for _ in range(max(1_000, len(catalog_item_ids))):
                    item_id = str(
                        catalog_item_ids[int(rng.integers(0, len(catalog_item_ids)))]
                    )
                    if item_id not in selected:
                        chosen = item_id
                        chosen_source = "uniform_backfill"
                        break
            if chosen is None:
                raise ValueError(
                    "Catalog is too small to create unique semi-hard candidates"
                )
            candidates.append(chosen)
            sources.append(chosen_source)
            selected.add(chosen)

    uniform_needed = uniform_multiplier * batch_size
    attempts = 0
    while uniform_needed:
        item_id = str(catalog_item_ids[int(rng.integers(0, len(catalog_item_ids)))])
        attempts += 1
        if item_id in selected:
            if attempts > max(10_000, len(catalog_item_ids) * 10):
                for fallback in catalog_item_ids:
                    fallback_id = str(fallback)
                    if fallback_id not in selected:
                        item_id = fallback_id
                        break
                else:
                    raise ValueError(
                        "Catalog is too small for the requested shared uniform negatives"
                    )
            else:
                continue
        candidates.append(item_id)
        sources.append("uniform")
        selected.add(item_id)
        uniform_needed -= 1
    targets = np.arange(batch_size, dtype=np.int64)
    return candidates, targets, sources


def make_model_class(torch: Any, nn: Any, F: Any):
    class ContrastiveBiEncoder(nn.Module):
        def __init__(self, encoder: Any, hidden_size: int, projection_dim: int):
            super().__init__()
            self.encoder = encoder
            self.query_projection = nn.Linear(hidden_size, projection_dim)
            self.item_projection = nn.Linear(hidden_size, projection_dim)

        @staticmethod
        def pool(output: Any, attention_mask: Any) -> Any:
            token_embeddings = output.last_hidden_state
            mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
            return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(
                min=1e-9
            )

        def encode_query(self, batch: dict[str, Any]) -> Any:
            pooled = self.pool(self.encoder(**batch), batch["attention_mask"])
            return F.normalize(self.query_projection(pooled), p=2, dim=1)

        def encode_item(self, batch: dict[str, Any]) -> Any:
            pooled = self.pool(self.encoder(**batch), batch["attention_mask"])
            return F.normalize(self.item_projection(pooled), p=2, dim=1)

    return ContrastiveBiEncoder


def load_text_context(
    data_dir: Path,
) -> tuple[dict[str, str], dict[str, str], np.ndarray]:
    terms = pd.read_csv(
        data_dir / "terms.csv",
        usecols=["term_id", "query"],
        dtype=str,
        keep_default_na=False,
    )
    term_texts = dict(zip(terms["term_id"].astype(str), terms["query"].map(clean_text)))
    items = pd.read_csv(
        data_dir / "items.csv", usecols=ITEM_COLUMNS, dtype=str, keep_default_na=False
    )
    item_ids = items["item_id"].astype(str).to_numpy()
    item_texts = {
        str(row.item_id): build_item_text(row) for row in items.itertuples(index=False)
    }
    return term_texts, item_texts, item_ids


class HybridBatchBuilder:
    """Build deterministic batches with in-batch and mined negative items."""

    def __init__(
        self,
        tokenizer: Any,
        term_texts: dict[str, str],
        item_texts: dict[str, str],
        catalog_item_ids: np.ndarray,
        negative_pool: dict[str, dict[str, list[str]]],
        positives_by_term: dict[str, set[str]],
        query_max_length: int,
        item_max_length: int,
        seed: int,
        uniform_multiplier: int,
        use_semi_hard: bool,
    ):
        self.tokenizer = tokenizer
        self.term_texts = term_texts
        self.item_texts = item_texts
        self.catalog_item_ids = catalog_item_ids
        self.negative_pool = negative_pool
        self.positives_by_term = positives_by_term
        self.query_max_length = query_max_length
        self.item_max_length = item_max_length
        self.seed = seed
        self.uniform_multiplier = uniform_multiplier
        self.use_semi_hard = use_semi_hard
        self.epoch = 0
        self.source_counts: Counter[str] = Counter()

    def __call__(self, records: list[dict[str, object]]) -> tuple[Any, ...]:
        rows = pd.DataFrame(records)
        candidates, targets, sources = assemble_candidate_item_ids(
            rows,
            self.negative_pool,
            self.catalog_item_ids,
            self.seed,
            self.epoch,
            self.uniform_multiplier,
            self.use_semi_hard,
        )
        term_ids = rows["term_id"].astype(str).tolist()
        query_texts = [self.term_texts[term_id] for term_id in term_ids]
        try:
            item_batch_texts = [self.item_texts[item_id] for item_id in candidates]
        except KeyError as exc:
            raise KeyError(
                f"Negative pool contains unknown catalog item {exc.args[0]}"
            ) from exc
        query_batch = self.tokenizer(
            query_texts,
            padding=True,
            truncation=True,
            max_length=self.query_max_length,
            return_tensors="pt",
        )
        item_batch = self.tokenizer(
            item_batch_texts,
            padding=True,
            truncation=True,
            max_length=self.item_max_length,
            return_tensors="pt",
        )
        mask = known_positive_mask_numpy(
            term_ids, candidates, self.positives_by_term, targets
        )
        self.source_counts.update(sources)
        return query_batch, item_batch, mask, targets


def move_batch(batch: dict[str, Any], device: Any) -> dict[str, Any]:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def save_checkpoint(
    model_dir: Path,
    model: Any,
    tokenizer: Any,
    config: dict[str, object],
    torch: Any,
) -> None:
    model_dir.mkdir(parents=True, exist_ok=True)
    model.encoder.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    torch.save(
        {
            "query_projection": model.query_projection.state_dict(),
            "item_projection": model.item_projection.state_dict(),
        },
        model_dir / "projection_heads.pt",
    )
    (model_dir / "contrastive_config.json").write_text(
        json.dumps(config, indent=2), encoding="utf-8"
    )


def load_checkpoint(model_dir: Path, device: Any) -> tuple[Any, ...]:
    torch, nn, F, _, _, AutoModel, AutoTokenizer, _ = require_transformer_stack()
    config = json.loads(
        (model_dir / "contrastive_config.json").read_text(encoding="utf-8")
    )
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    encoder = AutoModel.from_pretrained(model_dir)
    Model = make_model_class(torch, nn, F)
    model = Model(encoder, int(config["hidden_size"]), int(config["projection_dim"]))
    state = torch.load(model_dir / "projection_heads.pt", map_location="cpu")
    model.query_projection.load_state_dict(state["query_projection"])
    model.item_projection.load_state_dict(state["item_projection"])
    model.to(device).eval()
    return torch, F, tokenizer, model, config


def train(args: argparse.Namespace) -> None:
    started = time.time()
    torch, nn, F, DataLoader, Dataset, AutoModel, AutoTokenizer, get_scheduler = (
        require_transformer_stack()
    )
    seed_everything(args.seed, torch)
    device = choose_device(torch, args.device)
    data_dir = Path(args.data_dir)
    model_dir = Path(args.model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)
    positives = load_positive_pairs(data_dir / "training_pairs.csv")
    positives, outer_terms = apply_outer_term_split(
        positives,
        Path(args.split_manifest) if args.split_manifest else None,
        args.outer_fold,
    )
    if args.limit_positives and len(positives) > args.limit_positives:
        kept_terms = positives["term_id"].drop_duplicates().astype(str)
        positives = (
            positives.loc[positives["term_id"].isin(set(kept_terms))]
            .head(args.limit_positives)
            .reset_index(drop=True)
        )
    inner_train, inner_valid, inner_split = grouped_inner_split(
        positives, args.inner_valid_size, args.seed
    )
    inner_split["outer_fold"] = args.outer_fold
    inner_split["used_in_full_refit"] = int(args.refit_full)
    inner_split.to_csv(model_dir / "training_term_split.csv", index=False)
    positives_by_term = {
        str(term_id): set(group["item_id"].astype(str))
        for term_id, group in positives.groupby("term_id", sort=False)
    }
    term_texts, item_texts, catalog_item_ids = load_text_context(data_dir)
    missing_terms = set(positives["term_id"].astype(str)) - set(term_texts)
    missing_items = set(positives["item_id"].astype(str)) - set(item_texts)
    if missing_terms or missing_items:
        raise KeyError(
            f"missing_terms={len(missing_terms)} missing_items={len(missing_items)}"
        )
    negative_pool = (
        load_negative_pool(Path(args.negative_pool)) if args.use_semi_hard else {}
    )

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    encoder = AutoModel.from_pretrained(args.model_name)
    hidden_size = int(encoder.config.hidden_size)
    Model = make_model_class(torch, nn, F)
    model = Model(encoder, hidden_size, args.projection_dim).to(device)

    class PositiveDataset(Dataset):
        def __init__(self, rows: pd.DataFrame):
            self.rows = rows.reset_index(drop=True).copy()
            self.rows["row_index"] = np.arange(len(self.rows), dtype=np.int64)

        def __len__(self) -> int:
            return len(self.rows)

        def __getitem__(self, index: int) -> dict[str, object]:
            row = self.rows.iloc[index]
            return {
                "term_id": str(row.term_id),
                "item_id": str(row.item_id),
                "row_index": int(row.row_index),
            }

    train_builder = HybridBatchBuilder(
        tokenizer,
        term_texts,
        item_texts,
        catalog_item_ids,
        negative_pool,
        positives_by_term,
        args.query_max_length,
        args.item_max_length,
        args.seed,
        args.uniform_multiplier,
        args.use_semi_hard,
    )
    valid_builder = HybridBatchBuilder(
        tokenizer,
        term_texts,
        item_texts,
        catalog_item_ids,
        negative_pool,
        positives_by_term,
        args.query_max_length,
        args.item_max_length,
        args.seed + 1,
        args.uniform_multiplier,
        args.use_semi_hard,
    )
    generator = torch.Generator()
    generator.manual_seed(args.seed)
    train_loader = DataLoader(
        PositiveDataset(inner_train),
        batch_size=args.batch_size,
        shuffle=True,
        generator=generator,
        num_workers=args.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=train_builder,
    )
    valid_loader = DataLoader(
        PositiveDataset(inner_valid),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=valid_builder,
    )
    parameter_groups = [
        {
            "params": model.encoder.parameters(),
            "lr": args.learning_rate,
            "weight_decay": args.weight_decay,
        },
        {
            "params": model.query_projection.parameters(),
            "lr": args.head_learning_rate,
            "weight_decay": args.weight_decay,
        },
        {
            "params": model.item_projection.parameters(),
            "lr": args.head_learning_rate,
            "weight_decay": args.weight_decay,
        },
    ]
    optimizer = torch.optim.AdamW(parameter_groups)
    steps_per_epoch = math.ceil(len(train_loader) / args.grad_accum_steps)
    total_steps = (
        min(args.max_steps, steps_per_epoch * args.epochs)
        if args.max_steps
        else steps_per_epoch * args.epochs
    )
    scheduler = get_scheduler(
        optimizer,
        num_warmup_steps=int(total_steps * args.warmup_ratio),
        num_training_steps=total_steps,
    )
    amp_enabled = args.fp16 and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    cache_fingerprint = None
    if args.cache_manifest:
        cache_fingerprint = json.loads(
            Path(args.cache_manifest).read_text(encoding="utf-8")
        ).get("cache_key")
    base_config: dict[str, object] = {
        "base_model": args.model_name,
        "hidden_size": hidden_size,
        "projection_dim": args.projection_dim,
        "temperature": args.temperature,
        "query_max_length": args.query_max_length,
        "item_max_length": args.item_max_length,
        "batch_size": args.batch_size,
        "epochs": args.epochs,
        "uniform_multiplier": args.uniform_multiplier,
        "use_semi_hard": args.use_semi_hard,
        "fp16": args.fp16,
        "split_seed": args.seed,
        "outer_fold": args.outer_fold,
        "outer_holdout_terms": len(outer_terms),
        "inner_train_terms": int(inner_train["term_id"].nunique()),
        "inner_valid_terms": int(inner_valid["term_id"].nunique()),
        "cache_fingerprint": cache_fingerprint,
        "negative_pool_fingerprint": (
            file_fingerprint(Path(args.negative_pool)) if args.use_semi_hard else None
        ),
    }

    def batch_loss(batch: tuple[Any, ...]) -> Any:
        query_batch, item_batch, mask_array, target_array = batch
        query_batch = move_batch(query_batch, device)
        item_batch = move_batch(item_batch, device)
        mask = torch.from_numpy(mask_array).to(device=device, dtype=torch.bool)
        targets = torch.from_numpy(target_array).to(device=device, dtype=torch.long)
        with torch.autocast(
            device_type=device.type, enabled=amp_enabled, dtype=torch.float16
        ):
            query_embedding = model.encode_query(query_batch)
            item_embedding = model.encode_item(item_batch)
            return masked_infonce_loss(
                query_embedding, item_embedding, mask, targets, args.temperature, F
            )

    def evaluate() -> float:
        model.eval()
        total_loss = 0.0
        total_rows = 0
        valid_builder.epoch = 0
        with torch.inference_mode():
            for batch in tqdm(valid_loader, desc="inner validation", leave=False):
                loss = batch_loss(batch)
                rows = len(batch[3])
                total_loss += float(loss.detach().cpu()) * rows
                total_rows += rows
        return total_loss / max(1, total_rows)

    best_loss = float("inf")
    best_epoch = 0
    global_step = 0
    epoch_metrics: list[dict[str, float | int]] = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        train_builder.epoch = epoch - 1
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        total_rows = 0
        for step, batch in enumerate(
            tqdm(train_loader, desc=f"contrastive epoch {epoch}"), start=1
        ):
            loss = batch_loss(batch) / args.grad_accum_steps
            scaler.scale(loss).backward()
            rows = len(batch[3])
            total_loss += float(loss.detach().cpu()) * args.grad_accum_steps * rows
            total_rows += rows
            if step % args.grad_accum_steps == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
                if args.max_steps and global_step >= args.max_steps:
                    break
        valid_loss = evaluate()
        train_loss = total_loss / max(1, total_rows)
        epoch_metrics.append(
            {"epoch": epoch, "train_loss": train_loss, "inner_valid_loss": valid_loss}
        )
        print(
            f"epoch={epoch} train_loss={train_loss:.6f} inner_valid_loss={valid_loss:.6f}"
        )
        if valid_loss < best_loss:
            best_loss = valid_loss
            best_epoch = epoch
            save_checkpoint(
                model_dir, model, tokenizer, {**base_config, "best_epoch": epoch}, torch
            )
        if args.max_steps and global_step >= args.max_steps:
            break

    refit_source_counts: dict[str, int] | None = None
    if args.refit_full:
        # Epoch selection above is leakage-safe but only sees the inner-training
        # terms. Start again from the pretrained encoder and fit the selected
        # number of epochs on every outer-training positive term.
        del model, optimizer, scheduler, scaler, parameter_groups
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        seed_everything(args.seed + 10_000, torch)
        encoder = AutoModel.from_pretrained(args.model_name)
        model = Model(encoder, hidden_size, args.projection_dim).to(device)
        full_builder = HybridBatchBuilder(
            tokenizer,
            term_texts,
            item_texts,
            catalog_item_ids,
            negative_pool,
            positives_by_term,
            args.query_max_length,
            args.item_max_length,
            args.seed + 10_000,
            args.uniform_multiplier,
            args.use_semi_hard,
        )
        full_generator = torch.Generator()
        full_generator.manual_seed(args.seed + 10_000)
        full_loader = DataLoader(
            PositiveDataset(positives),
            batch_size=args.batch_size,
            shuffle=True,
            generator=full_generator,
            num_workers=args.num_workers,
            pin_memory=device.type == "cuda",
            collate_fn=full_builder,
        )
        refit_parameter_groups = [
            {
                "params": model.encoder.parameters(),
                "lr": args.learning_rate,
                "weight_decay": args.weight_decay,
            },
            {
                "params": model.query_projection.parameters(),
                "lr": args.head_learning_rate,
                "weight_decay": args.weight_decay,
            },
            {
                "params": model.item_projection.parameters(),
                "lr": args.head_learning_rate,
                "weight_decay": args.weight_decay,
            },
        ]
        optimizer = torch.optim.AdamW(refit_parameter_groups)
        refit_epochs = max(1, best_epoch)
        refit_steps = math.ceil(len(full_loader) / args.grad_accum_steps) * refit_epochs
        if args.max_steps:
            refit_steps = min(refit_steps, args.max_steps)
        scheduler = get_scheduler(
            optimizer,
            num_warmup_steps=int(refit_steps * args.warmup_ratio),
            num_training_steps=refit_steps,
        )
        scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
        refit_global_step = 0
        for epoch in range(1, refit_epochs + 1):
            model.train()
            full_builder.epoch = epoch - 1
            optimizer.zero_grad(set_to_none=True)
            for step, batch in enumerate(
                tqdm(full_loader, desc=f"full outer refit {epoch}"), start=1
            ):
                loss = batch_loss(batch) / args.grad_accum_steps
                scaler.scale(loss).backward()
                if step % args.grad_accum_steps == 0 or step == len(full_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(), args.max_grad_norm
                    )
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    refit_global_step += 1
                    if args.max_steps and refit_global_step >= args.max_steps:
                        break
            if args.max_steps and refit_global_step >= args.max_steps:
                break
        save_checkpoint(
            model_dir,
            model,
            tokenizer,
            {
                **base_config,
                "best_epoch": best_epoch,
                "refit_on_all_outer_train": True,
                "refit_epochs": refit_epochs,
                "refit_positive_pairs": len(positives),
                "refit_terms": int(positives["term_id"].nunique()),
            },
            torch,
        )
        refit_source_counts = dict(sorted(full_builder.source_counts.items()))

    metrics = {
        "best_epoch": best_epoch,
        "best_inner_valid_loss": best_loss,
        "epochs": epoch_metrics,
        "training_positive_pairs": len(inner_train),
        "inner_validation_positive_pairs": len(inner_valid),
        "negative_source_counts": dict(sorted(train_builder.source_counts.items())),
        "refit_on_all_outer_train": args.refit_full,
        "refit_positive_pairs": len(positives) if args.refit_full else 0,
        "refit_terms": int(positives["term_id"].nunique()) if args.refit_full else 0,
        "refit_negative_source_counts": refit_source_counts,
        "elapsed_seconds": time.time() - started,
    }
    (model_dir / "training_metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    print(json.dumps(metrics, indent=2))


def encode_batches(
    ids: list[str],
    text_by_id: dict[str, str],
    tokenizer: Any,
    model: Any,
    tower: str,
    max_length: int,
    batch_size: int,
    device: Any,
    torch: Any,
    fp16: bool,
) -> np.ndarray:
    parts: list[np.ndarray] = []
    amp_enabled = fp16 and device.type == "cuda"
    model.eval()
    with torch.inference_mode():
        for start in tqdm(range(0, len(ids), batch_size), desc=f"encode {tower}"):
            texts = [text_by_id[value] for value in ids[start : start + batch_size]]
            batch = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            batch = move_batch(batch, device)
            with torch.autocast(
                device_type=device.type, enabled=amp_enabled, dtype=torch.float16
            ):
                embeddings = (
                    model.encode_query(batch)
                    if tower == "query"
                    else model.encode_item(batch)
                )
            parts.append(embeddings.float().cpu().numpy().astype(np.float16))
    return (
        np.concatenate(parts)
        if parts
        else np.empty((0, int(model.query_projection.out_features)), dtype=np.float16)
    )


def load_entity_texts(
    data_dir: Path, entity: str, ids: set[str] | None = None
) -> tuple[list[str], dict[str, str]]:
    if entity == "terms":
        frame = pd.read_csv(
            data_dir / "terms.csv",
            usecols=["term_id", "query"],
            dtype=str,
            keep_default_na=False,
        )
        if ids is not None:
            frame = frame.loc[frame["term_id"].isin(ids)]
        frame = frame.sort_values("term_id")
        ordered = frame["term_id"].astype(str).tolist()
        return ordered, dict(zip(ordered, frame["query"].map(clean_text)))
    if entity == "items":
        frame = pd.read_csv(
            data_dir / "items.csv",
            usecols=ITEM_COLUMNS,
            dtype=str,
            keep_default_na=False,
        )
        if ids is not None:
            frame = frame.loc[frame["item_id"].isin(ids)]
        frame = frame.sort_values("item_id")
        ordered = frame["item_id"].astype(str).tolist()
        return ordered, {
            str(row.item_id): build_item_text(row)
            for row in frame.itertuples(index=False)
        }
    raise ValueError(f"Unknown entity: {entity}")


def encode(args: argparse.Namespace) -> None:
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch, _, tokenizer, model, config = load_checkpoint(
        Path(args.model_dir), choose_device(require_transformer_stack()[0], args.device)
    )
    device = next(model.parameters()).device
    ids, text_by_id = load_entity_texts(Path(args.data_dir), args.entity)
    tower = "query" if args.entity == "terms" else "item"
    max_length = int(
        config["query_max_length"] if tower == "query" else config["item_max_length"]
    )
    embeddings = encode_batches(
        ids,
        text_by_id,
        tokenizer,
        model,
        tower,
        max_length,
        args.batch_size,
        device,
        torch,
        args.fp16,
    )
    id_column = "term_id" if args.entity == "terms" else "item_id"
    pd.DataFrame({id_column: ids}).to_csv(
        output_dir / f"{args.entity}_ids.csv", index=False
    )
    np.save(output_dir / f"{args.entity}_embeddings.npy", embeddings)
    manifest = {
        "entity": args.entity,
        "rows": len(ids),
        "dimension": int(embeddings.shape[1]),
        "dtype": str(embeddings.dtype),
        "checkpoint": str(Path(args.model_dir).resolve()),
        "checkpoint_config": config,
    }
    (output_dir / f"{args.entity}_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )


def score_embedding_pairs(
    pairs: pd.DataFrame,
    term_ids: list[str],
    term_embeddings: np.ndarray,
    item_ids: list[str],
    item_embeddings: np.ndarray,
    chunk_size: int = 200_000,
) -> np.ndarray:
    term_position = {value: position for position, value in enumerate(term_ids)}
    item_position = {value: position for position, value in enumerate(item_ids)}
    scores = np.empty(len(pairs), dtype=np.float32)
    for start in range(0, len(pairs), chunk_size):
        stop = min(start + chunk_size, len(pairs))
        chunk = pairs.iloc[start:stop]
        query_index = np.fromiter(
            (term_position[str(value)] for value in chunk["term_id"]),
            dtype=np.int64,
            count=len(chunk),
        )
        item_index = np.fromiter(
            (item_position[str(value)] for value in chunk["item_id"]),
            dtype=np.int64,
            count=len(chunk),
        )
        query = np.asarray(term_embeddings[query_index], dtype=np.float32)
        item = np.asarray(item_embeddings[item_index], dtype=np.float32)
        scores[start:stop] = np.einsum("ij,ij->i", query, item)
    return scores


def score_pairs(args: argparse.Namespace) -> None:
    started = time.time()
    data_dir = Path(args.data_dir)
    pair_path = Path(args.pairs) if args.pairs else data_dir / "submission_pairs.csv"
    pairs = pd.read_csv(
        pair_path, dtype=str, keep_default_na=False, nrows=args.limit_pairs or None
    )
    required = {"term_id", "item_id"}
    if not required.issubset(pairs.columns):
        raise ValueError(
            f"{pair_path} is missing {sorted(required - set(pairs.columns))}"
        )
    id_column = args.id_column or (
        "id" if "id" in pairs else "slate_id" if "slate_id" in pairs else ""
    )
    if not id_column:
        pairs["id"] = [f"PAIR_{index:010d}" for index in range(len(pairs))]
        id_column = "id"
    if id_column not in pairs:
        raise ValueError(f"Missing id column {id_column}")
    unique_terms = set(pairs["term_id"].astype(str))
    unique_items = set(pairs["item_id"].astype(str))
    term_ids, term_texts = load_entity_texts(data_dir, "terms", unique_terms)
    item_ids, item_texts = load_entity_texts(data_dir, "items", unique_items)
    if set(term_ids) != unique_terms or set(item_ids) != unique_items:
        raise KeyError("Some pair term/item IDs are missing from the source tables")

    torch_module = require_transformer_stack()[0]
    device = choose_device(torch_module, args.device)
    torch, _, tokenizer, model, config = load_checkpoint(Path(args.model_dir), device)
    term_embeddings = encode_batches(
        term_ids,
        term_texts,
        tokenizer,
        model,
        "query",
        int(config["query_max_length"]),
        args.encode_batch_size,
        device,
        torch,
        args.fp16,
    )
    item_embeddings = encode_batches(
        item_ids,
        item_texts,
        tokenizer,
        model,
        "item",
        int(config["item_max_length"]),
        args.encode_batch_size,
        device,
        torch,
        args.fp16,
    )
    scores = score_embedding_pairs(
        pairs,
        term_ids,
        term_embeddings,
        item_ids,
        item_embeddings,
        args.pair_chunk_size,
    )
    output = pairs[[id_column, "term_id", "item_id"]].copy()
    output["semantic_cosine"] = scores
    output["semantic_rank_pct"] = (
        output.groupby("term_id", sort=False)["semantic_cosine"]
        .rank(method="average", pct=True)
        .astype(np.float32)
    )
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(output_path, index=False)
    metrics = {
        "rows": len(output),
        "unique_ids": int(output[id_column].nunique()),
        "unique_terms": len(term_ids),
        "unique_items": len(item_ids),
        "elapsed_seconds": time.time() - started,
    }
    output_path.with_suffix(".metrics.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )
    print(json.dumps(metrics, indent=2))


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Contrastive MiniLM two-tower training and pair scoring."
    )
    sub = parser.add_subparsers(dest="operation", required=True)

    train_cmd = sub.add_parser("train")
    train_cmd.add_argument("--data-dir", default="data")
    train_cmd.add_argument(
        "--negative-pool",
        default="outputs/hybrid_embedding/contrastive_negative_pool.csv",
    )
    train_cmd.add_argument(
        "--model-dir", default="outputs/hybrid_embedding/contrastive_model"
    )
    train_cmd.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    train_cmd.add_argument("--split-manifest")
    train_cmd.add_argument("--outer-fold", type=int, default=-1)
    train_cmd.add_argument("--cache-manifest")
    train_cmd.add_argument("--projection-dim", type=int, default=256)
    train_cmd.add_argument("--temperature", type=float, default=0.05)
    train_cmd.add_argument("--query-max-length", type=int, default=48)
    train_cmd.add_argument("--item-max-length", type=int, default=128)
    train_cmd.add_argument("--batch-size", type=int, default=64)
    train_cmd.add_argument("--epochs", type=int, default=2)
    train_cmd.add_argument("--inner-valid-size", type=float, default=0.10)
    train_cmd.add_argument("--uniform-multiplier", type=int, default=3)
    train_cmd.add_argument(
        "--use-semi-hard", action=argparse.BooleanOptionalAction, default=True
    )
    train_cmd.add_argument(
        "--fp16", action=argparse.BooleanOptionalAction, default=True
    )
    train_cmd.add_argument("--learning-rate", type=float, default=2e-5)
    train_cmd.add_argument("--head-learning-rate", type=float, default=1e-4)
    train_cmd.add_argument("--weight-decay", type=float, default=0.01)
    train_cmd.add_argument("--warmup-ratio", type=float, default=0.06)
    train_cmd.add_argument("--max-grad-norm", type=float, default=1.0)
    train_cmd.add_argument("--grad-accum-steps", type=int, default=1)
    train_cmd.add_argument("--num-workers", type=int, default=0)
    train_cmd.add_argument("--device", default="auto")
    train_cmd.add_argument("--seed", type=int, default=42)
    train_cmd.add_argument("--limit-positives", type=int, default=0)
    train_cmd.add_argument("--max-steps", type=int, default=0)
    train_cmd.add_argument(
        "--refit-full",
        action=argparse.BooleanOptionalAction,
        default=True,
        help=(
            "After inner-term epoch selection, restart and refit that many epochs "
            "on all outer-training terms."
        ),
    )
    train_cmd.set_defaults(func=train)

    encode_cmd = sub.add_parser("encode")
    encode_cmd.add_argument("--data-dir", default="data")
    encode_cmd.add_argument(
        "--model-dir", default="outputs/hybrid_embedding/contrastive_model"
    )
    encode_cmd.add_argument(
        "--output-dir", default="outputs/hybrid_embedding/contrastive_embeddings"
    )
    encode_cmd.add_argument("--entity", choices=["terms", "items"], required=True)
    encode_cmd.add_argument("--batch-size", type=int, default=512)
    encode_cmd.add_argument("--device", default="auto")
    encode_cmd.add_argument(
        "--fp16", action=argparse.BooleanOptionalAction, default=True
    )
    encode_cmd.set_defaults(func=encode)

    score_cmd = sub.add_parser("score-pairs")
    score_cmd.add_argument("--data-dir", default="data")
    score_cmd.add_argument(
        "--model-dir", default="outputs/hybrid_embedding/contrastive_model"
    )
    score_cmd.add_argument("--pairs")
    score_cmd.add_argument("--id-column")
    score_cmd.add_argument(
        "--output", default="outputs/hybrid_embedding/contrastive_submission_scores.csv"
    )
    score_cmd.add_argument("--encode-batch-size", type=int, default=512)
    score_cmd.add_argument("--pair-chunk-size", type=int, default=200_000)
    score_cmd.add_argument("--device", default="auto")
    score_cmd.add_argument(
        "--fp16", action=argparse.BooleanOptionalAction, default=True
    )
    score_cmd.add_argument("--limit-pairs", type=int, default=0)
    score_cmd.set_defaults(func=score_pairs)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/embedding_experiment.py
"""Compare embedding recipes and turn the selected score blend into a submission."""

from __future__ import annotations

import argparse
import itertools
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

BASELINE_COMPONENTS = ["catboost_prob", "lgbm_prob", "lexical_score"]


def macro_f1(labels: np.ndarray, prediction: np.ndarray) -> float:
    labels = labels.astype(np.int8)
    prediction = prediction.astype(np.int8)
    tp = int(((labels == 1) & (prediction == 1)).sum())
    fp = int(((labels == 0) & (prediction == 1)).sum())
    fn = int(((labels == 1) & (prediction == 0)).sum())
    tn = int(((labels == 0) & (prediction == 0)).sum())
    pos_denominator = 2 * tp + fp + fn
    neg_denominator = 2 * tn + fp + fn
    pos_f1 = 2 * tp / pos_denominator if pos_denominator else 0.0
    neg_f1 = 2 * tn / neg_denominator if neg_denominator else 0.0
    return 0.5 * (pos_f1 + neg_f1)


def simplex_weights(n_components: int, step: float) -> list[tuple[float, ...]]:
    units = int(round(1.0 / step))
    if n_components <= 0 or units <= 0 or not np.isclose(units * step, 1.0):
        raise ValueError("weight-step must divide one exactly")
    output: list[tuple[float, ...]] = []
    for separators in itertools.combinations(
        range(units + n_components - 1), n_components - 1
    ):
        boundaries = (-1,) + separators + (units + n_components - 1,)
        counts = tuple(
            boundaries[index + 1] - boundaries[index] - 1
            for index in range(n_components)
        )
        output.append(tuple(count / units for count in counts))
    return output


def rank_by_fold(values: pd.Series, folds: pd.Series) -> np.ndarray:
    return (
        values.groupby(folds, sort=False)
        .rank(method="average", pct=True)
        .to_numpy(dtype=np.float32)
    )


def best_rate(
    scores: np.ndarray, labels: np.ndarray, rates: list[float]
) -> tuple[float, float]:
    order = np.argsort(-scores, kind="stable")
    best = (-1.0, 0.0)
    for rate in rates:
        n_positive = min(len(scores), max(0, int(round(len(scores) * rate))))
        prediction = np.zeros(len(scores), dtype=np.int8)
        prediction[order[:n_positive]] = 1
        value = macro_f1(labels, prediction)
        if value > best[0]:
            best = (value, rate)
    return best


def load_score_file(path: Path, value_name: str) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype={"slate_id": str, "id": str}, keep_default_na=False)
    key = "slate_id" if "slate_id" in frame else "id" if "id" in frame else ""
    if not key:
        raise ValueError(f"{path} must contain slate_id or id")
    if "semantic_cosine" not in frame:
        raise ValueError(f"{path} must contain semantic_cosine")
    output = frame[[key, "semantic_cosine"]].rename(
        columns={key: "slate_id", "semantic_cosine": value_name}
    )
    if output["slate_id"].duplicated().any():
        raise ValueError(f"{path} contains duplicate IDs")
    output[value_name] = pd.to_numeric(output[value_name], errors="raise").astype(
        np.float32
    )
    return output


def optimize_recipe(
    frame: pd.DataFrame,
    components: list[str],
    rates: list[float],
    weight_step: float,
    required_positive: set[str] | None = None,
) -> dict[str, object]:
    labels = frame["label"].to_numpy(dtype=np.int8)
    folds = frame["fold"]
    ranks = [
        rank_by_fold(pd.to_numeric(frame[column], errors="raise"), folds)
        for column in components
    ]
    best: dict[str, object] | None = None
    for weights in simplex_weights(len(components), weight_step):
        if required_positive and any(
            weights[components.index(component)] <= 0.0
            for component in required_positive
        ):
            continue
        blend = np.zeros(len(frame), dtype=np.float32)
        for values, weight in zip(ranks, weights):
            if weight:
                blend += values * np.float32(weight)
        score, rate = best_rate(blend, labels, rates)
        row: dict[str, object] = {
            "macro_f1": score,
            "positive_rate": rate,
            "weights": {
                component: weight for component, weight in zip(components, weights)
            },
        }
        if best is None or float(row["macro_f1"]) > float(best["macro_f1"]):
            best = row
    if best is None:
        raise RuntimeError("Empty optimization grid")
    return best


def optimize_ablations(args: argparse.Namespace) -> None:
    started = time.time()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(
        args.baseline_scores,
        dtype={"slate_id": str, "term_id": str},
        keep_default_na=False,
    )
    required = {"slate_id", "term_id", "label", "fold"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{args.baseline_scores} is missing {sorted(missing)}")
    if frame["slate_id"].duplicated().any():
        raise ValueError("Baseline OOF scores contain duplicate slate IDs")
    frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(np.int8)
    frame["fold"] = pd.to_numeric(frame["fold"], errors="raise").astype(np.int16)
    if not set(frame["label"].unique()).issubset({0, 1}):
        raise ValueError("OOF labels must be binary")
    if frame.groupby("term_id", sort=False)["fold"].nunique().gt(1).any():
        raise AssertionError("A validation term appears in more than one outer fold")
    baseline_components = [column for column in BASELINE_COMPONENTS if column in frame]
    if not baseline_components:
        raise ValueError("No baseline score components were found")

    recipes: list[tuple[str, list[str]]] = [
        ("current_gbdt_baseline", baseline_components)
    ]
    if args.frozen_scores:
        frozen = load_score_file(Path(args.frozen_scores), "frozen_semantic_cosine")
        frame = frame.merge(frozen, on="slate_id", how="left", validate="one_to_one")
        recipes.append(
            ("frozen_cosine_feature", [*baseline_components, "frozen_semantic_cosine"])
        )
    elif "semantic_cosine" in frame:
        frame["frozen_semantic_cosine"] = pd.to_numeric(
            frame["semantic_cosine"], errors="raise"
        )
        recipes.append(
            ("frozen_cosine_feature", [*baseline_components, "frozen_semantic_cosine"])
        )
    if args.contrastive_scores:
        contrastive = load_score_file(
            Path(args.contrastive_scores), "contrastive_semantic_cosine"
        )
        frame = frame.merge(
            contrastive, on="slate_id", how="left", validate="one_to_one"
        )
        recipes.append(
            (
                "contrastive_cosine_feature",
                [*baseline_components, "contrastive_semantic_cosine"],
            )
        )
    if args.hybrid_scores:
        hybrid = load_score_file(Path(args.hybrid_scores), "hybrid_semantic_cosine")
        frame = frame.merge(hybrid, on="slate_id", how="left", validate="one_to_one")
        recipes.append(
            (
                "contrastive_plus_hybrid_negatives",
                [*baseline_components, "hybrid_semantic_cosine"],
            )
        )
    if frame.isna().any().any():
        null_columns = frame.columns[frame.isna().any()].tolist()
        raise ValueError(
            f"A score file does not cover the identical OOF split: {null_columns}"
        )

    retrieval_metrics: dict[str, object] = {}
    if args.retrieval_metrics:
        retrieval_metrics = json.loads(
            Path(args.retrieval_metrics).read_text(encoding="utf-8")
        )
    default_recall = float(retrieval_metrics.get("hybrid_recall_at_100", 0.0))
    recall_by_recipe = {
        "current_gbdt_baseline": float(
            retrieval_metrics.get("tfidf_recall_at_100", 0.0)
        ),
        "frozen_cosine_feature": float(
            retrieval_metrics.get("frozen_embedding_recall_at_100", 0.0)
        ),
        "contrastive_cosine_feature": args.contrastive_recall_at_100,
        "contrastive_plus_hybrid_negatives": args.hybrid_contrastive_recall_at_100
        or default_recall,
    }
    rates = sorted(set(float(value) for value in args.rates))
    detailed: list[dict[str, object]] = []
    for name, components in recipes:
        added = set(components) - set(baseline_components)
        best = optimize_recipe(
            frame, components, rates, args.weight_step, required_positive=added
        )
        detailed.append(
            {
                "recipe": name,
                "macro_f1": best["macro_f1"],
                "retrieval_recall_at_100": recall_by_recipe.get(name, 0.0),
                "positive_rate": best["positive_rate"],
                "components": components,
                "weights": best["weights"],
            }
        )
    detailed.sort(
        key=lambda row: (
            -float(row["macro_f1"]),
            -float(row["retrieval_recall_at_100"]),
            str(row["recipe"]),
        )
    )
    selected = {
        **detailed[0],
        "selection_rule": "highest Macro-F1; Recall@100 tie-breaker",
        "split_rows": len(frame),
        "split_terms": int(frame["term_id"].nunique()),
        "elapsed_seconds": time.time() - started,
    }
    serializable = pd.DataFrame(
        [
            {
                **row,
                "components": json.dumps(row["components"]),
                "weights": json.dumps(row["weights"], sort_keys=True),
            }
            for row in detailed
        ]
    )
    serializable.to_csv(output_dir / "ablation_results.csv", index=False)
    (output_dir / "selected_blend.json").write_text(
        json.dumps(selected, indent=2), encoding="utf-8"
    )
    print(serializable.to_string(index=False))
    print(json.dumps(selected, indent=2))


def parse_component(specification: str) -> tuple[str, Path, str]:
    try:
        name, raw_path, column = specification.split("=", 1)[0], *specification.split(
            "=", 1
        )[1].rsplit(":", 1)
    except ValueError as exc:
        raise ValueError("Components use NAME=PATH:COLUMN syntax") from exc
    return name, Path(raw_path), column


def load_submission_component(name: str, path: Path, column: str) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype={"id": str, "term_id": str}, keep_default_na=False)
    missing = {"id", "term_id", column} - set(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing {sorted(missing)}")
    output = frame[["id", "term_id", column]].rename(columns={column: name})
    if output["id"].duplicated().any():
        raise ValueError(f"{path} contains duplicate ids")
    output[name] = pd.to_numeric(output[name], errors="raise").astype(np.float32)
    return output


def make_submission(args: argparse.Namespace) -> None:
    selected = json.loads(Path(args.selected_blend).read_text(encoding="utf-8"))
    selected_weights = {
        str(key): float(value)
        for key, value in selected["weights"].items()
        if float(value) > 0
    }
    component_frames: dict[str, pd.DataFrame] = {}
    for specification in args.component:
        name, path, column = parse_component(specification)
        component_frames[name] = load_submission_component(name, path, column)
    missing = set(selected_weights) - set(component_frames)
    if missing:
        raise ValueError(
            f"Selected blend needs missing final components: {sorted(missing)}"
        )
    ordered_names = list(selected_weights)
    merged = component_frames[ordered_names[0]]
    for name in ordered_names[1:]:
        merged = merged.merge(
            component_frames[name],
            on=["id", "term_id"],
            how="inner",
            validate="one_to_one",
        )
    expected = args.expected_rows or len(component_frames[ordered_names[0]])
    if len(merged) != expected:
        raise ValueError(
            f"Final score alignment has {len(merged):,} rows, expected {expected:,}"
        )
    blend = np.zeros(len(merged), dtype=np.float32)
    weight_sum = sum(selected_weights.values())
    for name, weight in selected_weights.items():
        ranks = (
            merged.groupby("term_id", sort=False)[name]
            .rank(method="average", pct=True)
            .to_numpy(dtype=np.float32)
        )
        blend += ranks * np.float32(weight / weight_sum)
    positive_rate = (
        args.positive_rate
        if args.positive_rate is not None
        else float(selected["positive_rate"])
    )
    n_positive = int(round(len(merged) * positive_rate))
    order = np.argsort(-blend, kind="stable")
    prediction = np.zeros(len(merged), dtype=np.int8)
    prediction[order[:n_positive]] = 1
    if not set(np.unique(prediction)).issubset({0, 1}):
        raise AssertionError("Predictions are not binary")
    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"id": merged["id"], "prediction": prediction}).to_csv(
        output_path, index=False
    )
    score_path = output_path.with_name(output_path.stem + "_scores.csv")
    pd.DataFrame(
        {"id": merged["id"], "term_id": merged["term_id"], "prob": blend}
    ).to_csv(score_path, index=False)
    report = {
        "rows": len(merged),
        "unique_ids": int(merged["id"].nunique()),
        "binary_predictions": True,
        "positives": int(prediction.sum()),
        "positive_rate": float(prediction.mean()),
        "selected_recipe": selected["recipe"],
        "weights": selected_weights,
    }
    output_path.with_suffix(".report.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print(json.dumps(report, indent=2))


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Embedding ablations, blend selection, and final submissions."
    )
    sub = parser.add_subparsers(dest="operation", required=True)

    optimize_cmd = sub.add_parser("optimize-ablations")
    optimize_cmd.add_argument("--baseline-scores", required=True)
    optimize_cmd.add_argument("--frozen-scores")
    optimize_cmd.add_argument("--contrastive-scores")
    optimize_cmd.add_argument("--hybrid-scores")
    optimize_cmd.add_argument("--retrieval-metrics")
    optimize_cmd.add_argument("--output-dir", required=True)
    optimize_cmd.add_argument("--weight-step", type=float, default=0.10)
    optimize_cmd.add_argument(
        "--rates",
        type=float,
        nargs="+",
        default=[round(value, 3) for value in np.arange(0.15, 0.351, 0.005)],
    )
    optimize_cmd.add_argument("--contrastive-recall-at-100", type=float, default=0.0)
    optimize_cmd.add_argument(
        "--hybrid-contrastive-recall-at-100", type=float, default=0.0
    )
    optimize_cmd.set_defaults(func=optimize_ablations)

    submit_cmd = sub.add_parser("make-submission")
    submit_cmd.add_argument("--selected-blend", required=True)
    submit_cmd.add_argument(
        "--component",
        action="append",
        required=True,
        help="NAME=PATH:COLUMN; repeat for every component",
    )
    submit_cmd.add_argument("--output", required=True)
    submit_cmd.add_argument("--positive-rate", type=float)
    submit_cmd.add_argument("--expected-rows", type=int, default=3_359_679)
    submit_cmd.set_defaults(func=make_submission)

    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


In [ ]:
sys.path.insert(0, str(SRC_DIR))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(SRC_DIR)
ENV.setdefault("TOKENIZERS_PARALLELISM", "false")
ENV.setdefault("PYTHONHASHSEED", str(SEED))
ENV.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")


def run_command(arguments):
    command = [sys.executable, *map(str, arguments)]
    print("\n$", " ".join(command), flush=True)
    subprocess.run(command, check=True, env=ENV)


run_command([SRC_DIR / "hybrid_embedding_retrieval.py", "--help"])
run_command([SRC_DIR / "contrastive_biencoder.py", "--help"])

## 1. Frozen MiniLM cache and exact hybrid retrieval

In [ ]:
cache_manifest = CACHE_DIR / "manifest.json"
run_command(
    [
        SRC_DIR / "hybrid_embedding_retrieval.py",
        "build-cache",
        "--data-dir",
        DATA_DIR,
        "--cache-dir",
        CACHE_DIR,
        "--model-name",
        BASE_MODEL,
        "--query-max-length",
        48,
        "--item-max-length",
        128,
        "--encode-batch-size",
        ENCODE_BATCH_SIZE,
        "--retrieval-topk",
        500,
        "--query-block-size",
        64,
        "--item-block-size",
        16384,
        "--device",
        DEVICE,
    ]
)

display(json.loads(cache_manifest.read_text()))

## 2. Conservative hybrid negative files

In [ ]:
contrastive_negative_pool = ROOT_OUT / "contrastive_negative_pool.csv"
reranker_negatives = ROOT_OUT / "reranker_negatives.csv"
cache_key = json.loads(cache_manifest.read_text())["cache_key"]


def negative_cache_matches(path):
    audit = path.with_suffix(".audit.json")
    return (
        path.exists()
        and audit.exists()
        and json.loads(audit.read_text()).get("cache_fingerprint") == cache_key
    )


if not SKIP_EXISTING or not negative_cache_matches(contrastive_negative_pool):
    run_command(
        [
            SRC_DIR / "hybrid_embedding_retrieval.py",
            "mine-contrastive",
            "--data-dir",
            DATA_DIR,
            "--cache-dir",
            CACHE_DIR,
            "--output",
            contrastive_negative_pool,
            "--seed",
            SEED,
        ]
    )
if not SKIP_EXISTING or not negative_cache_matches(reranker_negatives):
    run_command(
        [
            SRC_DIR / "hybrid_embedding_retrieval.py",
            "mine-reranker",
            "--data-dir",
            DATA_DIR,
            "--cache-dir",
            CACHE_DIR,
            "--output",
            reranker_negatives,
            "--seed",
            SEED,
        ]
    )

display(json.loads(contrastive_negative_pool.with_suffix(".audit.json").read_text()))
display(json.loads(reranker_negatives.with_suffix(".audit.json").read_text()))

## 3. Fixed pilot or five-fold hybrid validation slates

In [ ]:
slate_path = RUN_DIR / "hybrid_validation_slates.csv"
split_manifest = RUN_DIR / "split_manifest.csv"
retrieval_metrics = RUN_DIR / "retrieval_metrics.json"
if MODE in {"pilot", "confirm"}:
    slate_cache_matches = (
        slate_path.exists()
        and retrieval_metrics.exists()
        and json.loads(retrieval_metrics.read_text()).get("cache_fingerprint")
        == cache_key
    )
    if not SKIP_EXISTING or not slate_cache_matches:
        run_command(
            [
                SRC_DIR / "hybrid_embedding_retrieval.py",
                "build-slates",
                "--data-dir",
                DATA_DIR,
                "--cache-dir",
                CACHE_DIR,
                "--output-dir",
                RUN_DIR,
                "--mode",
                MODE,
                "--seed",
                SEED,
            ]
        )
    display(json.loads(retrieval_metrics.read_text()))
else:
    print("final mode uses submission_pairs.csv; no outer validation slate is built")

## 4. Current GBDT baseline on the identical split

In [ ]:
baseline_scores = RUN_DIR / "baseline_oof_scores.csv"
if MODE in {"pilot", "confirm"} and RUN_GBDT:
    folds = [0] if MODE == "pilot" else list(range(5))
    gbdt_root = RUN_DIR / "gbdt"
    for fold in folds:
        fold_output = gbdt_root / f"fold_{fold}" / "oof_scores.csv"
        if SKIP_EXISTING and fold_output.exists():
            print(f"resume {fold_output}")
            continue
        arguments = [
            SRC_DIR / "grouped_oof_validation.py",
            "run-fold",
            "--data-dir",
            DATA_DIR,
            "--output-dir",
            gbdt_root,
            "--slates",
            slate_path,
            "--fold",
            fold,
            "--models",
            *GBDT_MODELS,
            "--inner-valid-size",
            0.10,
            "--catboost-iterations",
            CATBOOST_ITERATIONS,
            "--lgbm-estimators",
            LGBM_ESTIMATORS,
            "--task-type",
            "GPU" if "catboost" in GBDT_MODELS else "CPU",
            "--devices",
            CATBOOST_DEVICES,
            "--seed",
            SEED,
            "--no-semantic-features",
        ]
        run_command(arguments)
    if MODE == "pilot":
        shutil.copy2(gbdt_root / "fold_0" / "oof_scores.csv", baseline_scores)
    else:
        run_command(
            [
                SRC_DIR / "grouped_oof_validation.py",
                "combine",
                "--output-dir",
                gbdt_root,
                "--n-splits",
                5,
                "--output",
                baseline_scores,
            ]
        )
elif MODE in {"pilot", "confirm"}:
    raise ValueError("Set RUN_GBDT=True or provide baseline_oof_scores.csv in RUN_DIR")

## 5. Contrastive checkpoints and held-out scores

In [ ]:
import pandas as pd

slates = (
    pd.read_csv(
        slate_path,
        dtype={"slate_id": str, "term_id": str, "item_id": str},
    )
    if MODE != "final"
    else None
)


def train_and_score(label, use_semi_hard, fold):
    model_dir = RUN_DIR / "models" / label / f"fold_{fold}"
    score_path = RUN_DIR / "scores" / f"{label}_fold_{fold}.csv"
    pair_path = RUN_DIR / "scores" / f"pairs_fold_{fold}.csv"
    model_dir.parent.mkdir(parents=True, exist_ok=True)
    score_path.parent.mkdir(parents=True, exist_ok=True)
    heldout = slates.loc[slates["fold"].eq(fold), ["slate_id", "term_id", "item_id"]]
    heldout.to_csv(pair_path, index=False)
    if not SKIP_EXISTING or not (model_dir / "training_metrics.json").exists():
        arguments = [
            SRC_DIR / "contrastive_biencoder.py",
            "train",
            "--data-dir",
            DATA_DIR,
            "--negative-pool",
            contrastive_negative_pool,
            "--model-dir",
            model_dir,
            "--model-name",
            BASE_MODEL,
            "--split-manifest",
            split_manifest,
            "--outer-fold",
            fold,
            "--cache-manifest",
            cache_manifest,
            "--projection-dim",
            256,
            "--temperature",
            0.05,
            "--query-max-length",
            48,
            "--item-max-length",
            128,
            "--batch-size",
            TRAIN_BATCH_SIZE,
            "--epochs",
            TRAIN_EPOCHS,
            "--inner-valid-size",
            0.10,
            "--uniform-multiplier",
            3,
            "--device",
            DEVICE,
            "--fp16",
            "--seed",
            SEED,
            "--use-semi-hard" if use_semi_hard else "--no-use-semi-hard",
        ]
        run_command(arguments)
    if not SKIP_EXISTING or not score_path.exists():
        run_command(
            [
                SRC_DIR / "contrastive_biencoder.py",
                "score-pairs",
                "--data-dir",
                DATA_DIR,
                "--model-dir",
                model_dir,
                "--pairs",
                pair_path,
                "--id-column",
                "slate_id",
                "--output",
                score_path,
                "--encode-batch-size",
                ENCODE_BATCH_SIZE,
                "--device",
                DEVICE,
                "--fp16",
            ]
        )
    return model_dir, score_path


contrastive_scores = None
hybrid_scores = None
models_for_recall = {}
if MODE == "pilot":
    plain_model, contrastive_scores = train_and_score("contrastive_random", False, 0)
    hybrid_model, hybrid_scores = train_and_score("contrastive_hybrid", True, 0)
    models_for_recall = {"contrastive": plain_model, "hybrid": hybrid_model}
elif MODE == "confirm":
    if PILOT_SELECTED_BLEND is None:
        raise ValueError(
            "MODE='confirm' requires PILOT_SELECTED_BLEND from the pilot artifacts"
        )
    pilot_selected = json.loads(Path(PILOT_SELECTED_BLEND).read_text())
    winning_recipe = pilot_selected["recipe"]
    print("confirming pilot winner", winning_recipe)
    fold_scores = []
    for fold in range(5):
        if winning_recipe == "contrastive_cosine_feature":
            model, score = train_and_score("contrastive_random", False, fold)
            contrastive_scores = score
        elif winning_recipe == "contrastive_plus_hybrid_negatives":
            model, score = train_and_score("contrastive_hybrid", True, fold)
            hybrid_scores = score
        else:
            break
        fold_scores.append(pd.read_csv(score, dtype={"slate_id": str}))
        models_for_recall[f"fold_{fold}"] = model
    if fold_scores:
        combined_path = RUN_DIR / "scores" / f"{winning_recipe}_oof.csv"
        pd.concat(fold_scores, ignore_index=True).sort_values("slate_id").to_csv(
            combined_path, index=False
        )
        if winning_recipe == "contrastive_cosine_feature":
            contrastive_scores = combined_path
        else:
            hybrid_scores = combined_path

## 6. Retrieval Recall@100 for checkpoint tie-breaking

In [ ]:
from hybrid_embedding_retrieval import cache_paths, exact_cosine_topk


def checkpoint_recall_at_100(label, model_dir, heldout_fold=0):
    embedding_dir = RUN_DIR / "encoded" / label
    term_file = embedding_dir / "terms_embeddings.npy"
    item_file = embedding_dir / "items_embeddings.npy"
    if not SKIP_EXISTING or not term_file.exists():
        run_command(
            [
                SRC_DIR / "contrastive_biencoder.py",
                "encode",
                "--data-dir",
                DATA_DIR,
                "--model-dir",
                model_dir,
                "--output-dir",
                embedding_dir,
                "--entity",
                "terms",
                "--batch-size",
                ENCODE_BATCH_SIZE,
                "--device",
                DEVICE,
                "--fp16",
            ]
        )
    if not SKIP_EXISTING or not item_file.exists():
        run_command(
            [
                SRC_DIR / "contrastive_biencoder.py",
                "encode",
                "--data-dir",
                DATA_DIR,
                "--model-dir",
                model_dir,
                "--output-dir",
                embedding_dir,
                "--entity",
                "items",
                "--batch-size",
                ENCODE_BATCH_SIZE,
                "--device",
                DEVICE,
                "--fp16",
            ]
        )
    term_ids = pd.read_csv(embedding_dir / "terms_ids.csv", dtype=str)[
        "term_id"
    ].tolist()
    item_ids = pd.read_csv(embedding_dir / "items_ids.csv", dtype=str)[
        "item_id"
    ].tolist()
    term_pos = {value: index for index, value in enumerate(term_ids)}
    item_pos = {value: index for index, value in enumerate(item_ids)}
    split = pd.read_csv(split_manifest, dtype={"term_id": str})
    selected_terms = set(split.loc[split["fold"].eq(heldout_fold), "term_id"])
    positives = pd.read_csv(DATA_DIR / "training_pairs.csv", dtype=str)
    positives = positives.loc[positives["term_id"].isin(selected_terms)]
    q_rows = [term_pos[value] for value in sorted(selected_terms)]
    q_ids = sorted(selected_terms)
    term_embedding = __import__("numpy").load(term_file, mmap_mode="r")[q_rows]
    item_embedding = __import__("numpy").load(item_file, mmap_mode="r")
    indices, _ = exact_cosine_topk(term_embedding, item_embedding, 100, device=DEVICE)
    retrieved = {
        term: {item_ids[position] for position in indices[row]}
        for row, term in enumerate(q_ids)
    }
    hits = sum(
        str(row.item_id) in retrieved[str(row.term_id)]
        for row in positives.itertuples(index=False)
    )
    value = hits / max(1, len(positives))
    metrics = {"recall_at_100": value, "positives": len(positives)}
    (embedding_dir / "recall_at_100.json").write_text(json.dumps(metrics, indent=2))
    return value


contrastive_recall = 0.0
hybrid_contrastive_recall = 0.0
if MODE == "pilot" and COMPUTE_CONTRASTIVE_RECALL:
    contrastive_recall = checkpoint_recall_at_100(
        "contrastive_random", models_for_recall["contrastive"]
    )
    hybrid_contrastive_recall = checkpoint_recall_at_100(
        "contrastive_hybrid", models_for_recall["hybrid"]
    )
print(
    {
        "contrastive_recall_at_100": contrastive_recall,
        "hybrid_contrastive_recall_at_100": hybrid_contrastive_recall,
    }
)

## 7. Identical-split ablations and recipe selection

In [ ]:
if MODE in {"pilot", "confirm"}:
    heldout_folds = {0} if MODE == "pilot" else set(range(5))
    frozen_scores = RUN_DIR / "scores" / "frozen_oof.csv"
    frozen_frame = slates.loc[
        slates["fold"].isin(heldout_folds),
        ["slate_id", "term_id", "item_id", "semantic_cosine", "semantic_rank_pct"],
    ]
    frozen_frame.to_csv(frozen_scores, index=False)
    arguments = [
        SRC_DIR / "embedding_experiment.py",
        "optimize-ablations",
        "--baseline-scores",
        baseline_scores,
        "--frozen-scores",
        frozen_scores,
        "--retrieval-metrics",
        retrieval_metrics,
        "--output-dir",
        RUN_DIR,
        "--weight-step",
        WEIGHT_STEP,
        "--contrastive-recall-at-100",
        contrastive_recall,
        "--hybrid-contrastive-recall-at-100",
        hybrid_contrastive_recall,
    ]
    if contrastive_scores is not None:
        arguments.extend(["--contrastive-scores", contrastive_scores])
    if hybrid_scores is not None:
        arguments.extend(["--hybrid-scores", hybrid_scores])
    run_command(arguments)
    display(pd.read_csv(RUN_DIR / "ablation_results.csv"))
    display(json.loads((RUN_DIR / "selected_blend.json").read_text()))

## 8. Final full-data training, unique-pair scoring, and submission

In [ ]:
if MODE == "final":
    if CONFIRMED_SELECTED_BLEND is None:
        raise ValueError("MODE='final' requires CONFIRMED_SELECTED_BLEND")
    selected = json.loads(Path(CONFIRMED_SELECTED_BLEND).read_text())
    recipe = selected["recipe"]
    final_specs = dict(FINAL_COMPONENTS)
    semantic_scores = RUN_DIR / "final_semantic_scores.csv"
    required_components = {
        name for name, weight in selected["weights"].items() if float(weight) > 0
    }

    # With the fast default (LightGBM + lexical), final mode is self-contained.
    # User-supplied FINAL_COMPONENTS still take precedence when confirmed full
    # score files are attached.
    if "lexical_score" in required_components and "lexical_score" not in final_specs:
        lexical_scores = RUN_DIR / "final_lexical_scores.csv"
        if not SKIP_EXISTING or not lexical_scores.exists():
            run_command(
                [
                    SRC_DIR / "lexical_baseline.py",
                    "score",
                    "--data-dir",
                    DATA_DIR,
                    "--pairs",
                    "submission_pairs.csv",
                    "--output",
                    lexical_scores,
                ]
            )
        final_specs["lexical_score"] = (lexical_scores, "score")
    if "lgbm_prob" in required_components and "lgbm_prob" not in final_specs:
        lgbm_model = RUN_DIR / "final_pu_lgbm.joblib"
        lgbm_scores = RUN_DIR / "final_pu_lgbm_scores.csv"
        if not SKIP_EXISTING or not lgbm_model.exists():
            run_command(
                [
                    SRC_DIR / "pu_lgbm.py",
                    "train",
                    "--data-dir",
                    DATA_DIR,
                    "--negatives",
                    reranker_negatives,
                    "--model",
                    lgbm_model,
                    "--n-estimators",
                    LGBM_ESTIMATORS,
                    "--seed",
                    SEED,
                ]
            )
        if not SKIP_EXISTING or not lgbm_scores.exists():
            run_command(
                [
                    SRC_DIR / "pu_lgbm.py",
                    "predict",
                    "--data-dir",
                    DATA_DIR,
                    "--model",
                    lgbm_model,
                    "--output",
                    lgbm_scores,
                ]
            )
        final_specs["lgbm_prob"] = (lgbm_scores, "prob")

    if recipe in {"contrastive_cosine_feature", "contrastive_plus_hybrid_negatives"}:
        final_model = RUN_DIR / "models" / "final_contrastive"
        use_hybrid = recipe == "contrastive_plus_hybrid_negatives"
        if not SKIP_EXISTING or not (final_model / "training_metrics.json").exists():
            run_command(
                [
                    SRC_DIR / "contrastive_biencoder.py",
                    "train",
                    "--data-dir",
                    DATA_DIR,
                    "--negative-pool",
                    contrastive_negative_pool,
                    "--model-dir",
                    final_model,
                    "--model-name",
                    BASE_MODEL,
                    "--cache-manifest",
                    cache_manifest,
                    "--projection-dim",
                    256,
                    "--temperature",
                    0.05,
                    "--query-max-length",
                    48,
                    "--item-max-length",
                    128,
                    "--batch-size",
                    TRAIN_BATCH_SIZE,
                    "--epochs",
                    TRAIN_EPOCHS,
                    "--inner-valid-size",
                    0.10,
                    "--uniform-multiplier",
                    3,
                    "--device",
                    DEVICE,
                    "--fp16",
                    "--seed",
                    SEED,
                    "--use-semi-hard" if use_hybrid else "--no-use-semi-hard",
                ]
            )
        if not SKIP_EXISTING or not semantic_scores.exists():
            run_command(
                [
                    SRC_DIR / "contrastive_biencoder.py",
                    "score-pairs",
                    "--data-dir",
                    DATA_DIR,
                    "--model-dir",
                    final_model,
                    "--pairs",
                    DATA_DIR / "submission_pairs.csv",
                    "--output",
                    semantic_scores,
                    "--encode-batch-size",
                    ENCODE_BATCH_SIZE,
                    "--device",
                    DEVICE,
                    "--fp16",
                ]
            )
        component_name = (
            "hybrid_semantic_cosine" if use_hybrid else "contrastive_semantic_cosine"
        )
        final_specs[component_name] = (semantic_scores, "semantic_cosine")
    elif recipe == "frozen_cosine_feature":
        if not SKIP_EXISTING or not semantic_scores.exists():
            import numpy as np
            from contrastive_biencoder import score_embedding_pairs

            pairs = pd.read_csv(DATA_DIR / "submission_pairs.csv", dtype=str)
            item_ids = pd.read_csv(CACHE_DIR / "item_ids.csv", dtype=str)[
                "item_id"
            ].tolist()
            term_ids = pd.read_csv(CACHE_DIR / "term_ids.csv", dtype=str)[
                "term_id"
            ].tolist()
            scores = score_embedding_pairs(
                pairs,
                term_ids,
                np.load(CACHE_DIR / "term_embeddings.npy", mmap_mode="r"),
                item_ids,
                np.load(CACHE_DIR / "item_embeddings.npy", mmap_mode="r"),
            )
            frozen = pairs[["id", "term_id", "item_id"]].copy()
            frozen["semantic_cosine"] = scores
            frozen["semantic_rank_pct"] = frozen.groupby("term_id")[
                "semantic_cosine"
            ].rank(method="average", pct=True)
            frozen.to_csv(semantic_scores, index=False)
        final_specs["frozen_semantic_cosine"] = (semantic_scores, "semantic_cosine")

    missing = required_components - set(final_specs)
    if missing:
        raise ValueError(
            f"Add full-data score paths to FINAL_COMPONENTS for: {sorted(missing)}"
        )
    arguments = [
        SRC_DIR / "embedding_experiment.py",
        "make-submission",
        "--selected-blend",
        CONFIRMED_SELECTED_BLEND,
        "--output",
        RUN_DIR / "submission_fast_hybrid_embedding.csv",
        "--expected-rows",
        EXPECTED_SUBMISSION_ROWS,
    ]
    for name, (path, column) in final_specs.items():
        if name in required_components:
            arguments.extend(["--component", f"{name}={path}:{column}"])
    run_command(arguments)
    display(
        json.loads(
            (RUN_DIR / "submission_fast_hybrid_embedding.report.json").read_text()
        )
    )

## 9. Package resumable artifacts

In [ ]:
if PACKAGE_ARTIFACTS:
    archive = shutil.make_archive(
        str(WORK_DIR / f"fast_hybrid_embedding_{MODE}_artifacts"),
        "gztar",
        ROOT_OUT,
    )
    print(f"packaged {archive}")

print("run complete", {"mode": MODE, "run_dir": str(RUN_DIR), "cache": str(CACHE_DIR)})